# Racialised discourse in German swimming pool reviews: prevalence across WHEN, WHERE, and WHO

## Notebook Summary


**Data provenance** (see `gmaps_data/README.md` and `socio-economic_data/README_data_processing.md` for full detail — not reprocessed here):
- `master_corpus_final.parquet`: 1,265,806 Google Maps reviews of 5,529 verified swimming-pool POIs in Germany. Every row has non-null `org_review`/`trans_review` — **this corpus contains only reviews that have text**, not every review Google recorded for these places (many Google reviews are a star rating with no text, and are simply absent from this corpus). We call this count `total_text_reviews` throughout to keep that scope explicit. `social` (yes/no) and `racialised` (yes/no/NaN) are review-level labels; `racialised` is non-null for every `social == 'yes'` row (504,072 rows), with **7,768 `yes`**. `approach`/`theme`/`connotation`/`context` are multi-label lists populated only for those 7,768 rows ("dimension scope").
- `federal_baseline_eng_selected_political.csv`: 299 constituency-level rows with socioeconomic indicators (population, age structure, income, employment, SGB II) and Bundestag 2025 **second-vote shares** (not raw counts, `_pct`  ).

## Research questions

- **RQ1.** How prevalent is racialised discourse?
--- **When**   What is the time series trend for  racialised rate across seasons?
               What is the time series trend  for the  dimensions of racial reviews?
--- **Where**  In which constituencies is racialised rate  prevalent
--- **Who**    Who  is racialising?


This notebook (Part 1) works toward RQ1 using the gmaps review data alone, organized into
WHEN (temporal trend), WHERE (geographic distribution), and WHO (reviewer ethnicity/
language) sections.

## Setup

Load the two canonical inputs and do a first-touch shape/label sanity check against the counts documented in the corpus README (7,768 racialised, 504,072 social=yes, 1,265,806 total) — if these don't match, something upstream has drifted and everything after this point is suspect.


In [ ]:
#packages
import warnings
from pathlib import Path
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats


warnings.filterwarnings("ignore", category=FutureWarning)
sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 60)

from statsmodels.stats.proportion import proportion_confint

def slugify(title):
    """Turn a plot title into a filesystem-safe filename stem (used when saving figures)."""
    return re.sub(r"[^a-z0-9]+", "_", title.lower()).strip("_")[:80]

In [ ]:
ANALYSIS_DIR = Path("/home/ethel/Desktop/swimming_pool_project/spp_15052026/analysis_10072026")
GMAPS_DIR = ANALYSIS_DIR / "gmaps_data"
SOCIO_DIR = ANALYSIS_DIR / "socio-economic_data"
PROJECT_ROOT = Path("/home/ethel/Desktop/swimming_pool_project")
SHAPEFILE_PATH = PROJECT_ROOT / "btw25_geometrie_wahlkreise_shp" / "btw25_geometrie_wahlkreise_shp.shp"
FIGURES_DIR = ANALYSIS_DIR / "descriptive_analysis_figures"
FIGURES_DIR.mkdir(exist_ok=True)

corpus = pd.read_parquet(GMAPS_DIR / "master_corpus_final.parquet")
political = pd.read_csv(SOCIO_DIR / "federal_baseline_eng_selected_political.csv")

print(f"corpus: {corpus.shape[0]:,} rows x {corpus.shape[1]} cols")
print(f"political/socioeconomic: {political.shape[0]} constituencies x {political.shape[1]} cols")

assert corpus.shape[0] == 1_265_806, "row count drifted from documented corpus size"
assert (corpus["racialised"] == "yes").sum() == 7_768, "racialised=yes count drifted"
assert (corpus["social"] == "yes").sum() == 504_072, "social=yes count drifted"
assert political.shape[0] == 299, "expected exactly 299 Bundestag constituencies"
print("Sanity checks passed: corpus and political data match documented provenance.")


In [ ]:
corpus.info()
#.info()
#corpus[corpus['social']== 'yes'].groupby('racialised').size().reset_index(name='count') 

## Rebuilding the place → constituency spatial join


 *method* — dissolve the 299 constituency polygons into one Germany polygon, keep only points strictly inside it, then `sjoin_nearest` the kept points against the individual polygons so every retained point is guaranteed exactly one constituency, even ones that sit awkwardly on a shared border. 
We use that method  once at the **place level** (5,529 unique places, keyed by `unique_identifier`) and then broadcast the result back onto all 1,265,806 review rows — same logic, ~230x fewer spatial operations, and verified end-to-end against the current canonical corpus.

**CRS choice:** the shapefile's native CRS is `EPSG:25832` (ETRS89 / UTM zone 32N), a metric projection appropriate for Germany. We reproject points into that CRS rather than reprojecting to Web Mercator (`EPSG:3857`, used in the openday script) — reprojecting to the shapefile's own native CRS avoids an extra, unnecessary reprojection step and the distortion Web Mercator introduces at this latitude.

In [ ]:
import geopandas as gpd

# --- place-level table (one row per unique_identifier) ---
places = (
    corpus[["unique_identifier", "place_name", "latitude", "longitude"]]
    .drop_duplicates(subset="unique_identifier")
    .reset_index(drop=True)
)
n_missing_coords = places[["latitude", "longitude"]].isna().any(axis=1).sum()
print(f"places (unique unique_identifier): {len(places):,}")
print(f"places missing lat/lon: {n_missing_coords}")

places_geo = gpd.GeoDataFrame(
    places,
    geometry=gpd.points_from_xy(places["longitude"], places["latitude"]),
    crs="EPSG:4326",
).to_crs("EPSG:25832")

# --- constituency shapefile ---
constituencies = gpd.read_file(SHAPEFILE_PATH)
print(f"constituencies: {len(constituencies)} polygons, CRS = {constituencies.crs}")

n_invalid = (~constituencies.geometry.is_valid).sum()
print(f"invalid polygon geometries before repair: {n_invalid}")
if n_invalid:
    constituencies["geometry"] = constituencies.geometry.buffer(0)

assert constituencies.crs.to_string() == "EPSG:25832", "shapefile CRS is not what we expect"
assert constituencies["WKR_NR"].nunique() == 299, "expected 299 unique constituency numbers"


In [ ]:
# --- step 1: keep only places strictly inside Germany (dissolved boundary) ---
germany_boundary = constituencies.dissolve().geometry.iloc[0]
germany_gdf = gpd.GeoDataFrame(geometry=[germany_boundary], crs=constituencies.crs)

inside_mask = gpd.sjoin(places_geo, germany_gdf, predicate="within", how="inner").index
places_inside = places_geo.loc[places_geo.index.isin(inside_mask)].copy()
n_dropped = len(places_geo) - len(places_inside)
print(f"places strictly inside Germany polygon: {len(places_inside):,} (dropped as outside: {n_dropped})")

# --- step 2: assign each inside place to its nearest constituency polygon ---
# (guarantees a match even for a point sitting exactly on / just outside a shared border)
joined = gpd.sjoin_nearest(
    places_inside,
    constituencies[["WKR_NR", "WKR_NAME", "LAND_NR", "LAND_NAME", "geometry"]],
    how="left",
)

n_dupe_matches = joined.duplicated(subset="unique_identifier").sum()
print(f"places matched more than once (nearest-neighbour ties): {n_dupe_matches}")
joined = joined.drop_duplicates(subset="unique_identifier", keep="first")

place_constituency = joined.rename(
    columns={"WKR_NR": "constituency_code", "WKR_NAME": "constituency", "LAND_NAME": "state"}
)[["unique_identifier", "constituency_code", "constituency", "state"]].reset_index(drop=True)

print(f"final place -> constituency mapping: {len(place_constituency):,} rows")
assert place_constituency["unique_identifier"].is_unique, "join must be 1:1 on unique_identifier"
place_constituency.head()


In [ ]:
# --- broadcast the place-level mapping back onto every review row ---
corpus = corpus.merge(place_constituency, on="unique_identifier", how="left")

n_reviews_unmatched = corpus["constituency_code"].isna().sum()
n_places_unmatched = places["unique_identifier"].nunique() - place_constituency["unique_identifier"].nunique()
print(f"review rows without a constituency match: {n_reviews_unmatched:,} ({n_reviews_unmatched / len(corpus):.4%})")
print(f"places without a constituency match: {n_places_unmatched:,}")

if n_reviews_unmatched:
    print("\nplace_name of unmatched places:")
    print(
        corpus.loc[corpus["constituency_code"].isna(), "place_name"].drop_duplicates().head(10).to_list()
    )

east_states = {
    "Brandenburg", "Mecklenburg-Vorpommern", "Sachsen", "Sachsen-Anhalt", "Thüringen",
}
corpus["region"] = np.where(corpus["state"].isin(east_states), "East", "West")
corpus.loc[corpus["state"] == "Berlin", "region"] = "Berlin"

print("\nreviews by region (East / West / Berlin split out):")
print(corpus["region"].value_counts(dropna=False))


**Takeaway.** The fresh join is a clean, complete match: all 5,528 places with reviews in `master_corpus_final` (5,529 verified POIs minus the one documented in the `gmaps_dat/README` as having no reviews) fall inside Germany and get exactly one constituency, with zero drops and zero nearest-neighbour ties — so **100% of the 1,265,806 review rows now carry a `constituency_code`/`constituency`/`state`**. T We also derive a simple `region` (East/West/Berlin) split for later comparisons, reusing the standard German reunification state grouping.

In [ ]:
# racialised=yes subset, reused throughout Part 1 (WHEN's dimension-over-time facet,
# WHERE's pool-concentration checks, WHO's ethnicity/language facet, and the
# dimension-content-overview section) -- built once here rather than re-derived per section.
racialised_df = corpus.loc[corpus["racialised"] == "yes"].copy()
print(f"racialised=yes reviews: {len(racialised_df):,}")

# Wilson score interval helper, used throughout Part 1 (WHEN's yearly/monthly rates, WHERE's
# constituency/state prevalence) -- defined once here, before either section needs it.
from statsmodels.stats.proportion import proportion_confint

def wilson(successes, n):
    if n == 0:
        return np.nan, np.nan, np.nan
    rate = successes / n
    lo, hi = proportion_confint(successes, n, method="wilson")
    return rate, lo, hi


# Part 1 — Prevalence of racialised discourse: WHEN, WHERE, and WHO

This section establishes the scale of racialised discourse in the gmaps corpus and how it distributes across constituencies. Social reviews make up 504,072 / 1,265,806 (~40%) of the corpus; racialised reviews are rarer still (7,768 / 1,265,806 ≈ 0.6% of all reviews, ~1.5% of social reviews). Spread over 299 constituencies, many will have few racialised reviews — not because there's no racialised discourse there, but because pool coverage itself is thin in the first place. We establish this coverage picture before computing rates and drawing maps.

Part 1 investigates the prevalence of racialised reviews across **WHEN** (temporal trend), **WHERE**
(geographic distribution), and **WHO** (reviewer ethnicity/language, including the content -- theme,
approach, connotation, context -- of what's said).

## Dimension classification coverage

Before the WHEN/WHERE/WHO breakdown, a quick coverage check: `approach`, `theme`, `connotation`, and `context` are populated for every one of the 7,768 racialised=yes reviews (multi-label -- a review can carry more than one value per dimension). The table below confirms 100% coverage per dimension and shows how the labels break down as a share of all racialised reviews.


In [ ]:
DIMENSIONS = ["approach", "theme", "connotation", "context"]

def is_classified(cell):
    if isinstance(cell, (list, tuple, set, np.ndarray)):
        return len(cell) > 0
    if isinstance(cell, str):
        return len(cell) > 0
    return False  # NaN or unexpected type

coverage_rows = []
label_rows = []
for dim in DIMENSIONS:
    n_classified = racialised_df[dim].apply(is_classified).sum()
    coverage_rows.append({
        "dimension": dim,
        "n_classified": n_classified,
        "pct_classified": n_classified / len(racialised_df),
    })
    counts = racialised_df[dim].explode().value_counts()
    for label, n in counts.items():
        label_rows.append({
            "dimension": dim,
            "label": label,
            "n_reviews": n,
            "pct_of_racialised_reviews": n / len(racialised_df),
        })

coverage_table = pd.DataFrame(coverage_rows).set_index("dimension")
print(f"Classification coverage (of {len(racialised_df):,} racialised reviews):")
print(coverage_table.assign(pct_classified=lambda d: (d["pct_classified"] * 100).round(1)))
assert (coverage_table["pct_classified"] == 1.0).all(), "not every racialised review is classified on every dimension"

label_breakdown = pd.DataFrame(label_rows).set_index(["dimension", "label"])
label_breakdown["pct_of_racialised_reviews"] = (label_breakdown["pct_of_racialised_reviews"] * 100).round(1)
display(label_breakdown)


In [ ]:
7768/504072 

# WHEN


How the racialised-review rate moves over time: the headline year-over-year trend, the
mechanics of recovering timestamps for the small residual share of reviews that lacked one, how far back the
window can honestly be extended, month/season granularity, and how the four content
dimensions (`theme`/`context`/`connotation`/`approach`) move over the same years -- closing
with formal significance tests for the trend and season effects (both overall and per
dimensional label) and a check on whether the post-2021 rise is a nationwide pattern or
concentrated in a handful of constituencies.


### Headline trend

Timestamps were only merged into the corpus recently — the new `review_timestamp` column
enables this trend for the first time — and, as of the most recent `gmaps_poi` DB refresh, **1,260,044 / 1,265,806 (99.5%)** reviews now have a parsed absolute timestamp (up from 69.5% before the refresh; the remaining 0.5% have unparseable relative time text at source). Before trusting any time trend, we first check whether that residual missingness is roughly random or structured — e.g. concentrated in specific states or disproportionately hitting racialised reviews, which would bias a trend line built only from the reviews that do have a timestamp.

In [ ]:
has_ts = corpus["review_timestamp"].notna()
print(f"reviews with a parsed timestamp: {has_ts.sum():,} / {len(corpus):,} ({has_ts.mean():.1%})")

# missingness by racialised status (restricted to social=yes, where racialised is defined)
social_mask = corpus["social"] == "yes"
miss_by_racialised = (
    corpus.loc[social_mask]
    .assign(has_ts=has_ts[social_mask])
    .groupby("racialised")["has_ts"]
    .mean()
)
print("\ntimestamp coverage by racialised status (within social=yes):")
print(miss_by_racialised)

# missingness by state
miss_by_state = corpus.assign(has_ts=has_ts).groupby("state")["has_ts"].mean().sort_values()
print("\ntimestamp coverage by state (min/max):")
print(miss_by_state.head(3))
print("...")
print(miss_by_state.tail(3))


Missingness is not badly structured: 99.44% vs. 98.22% timestamp coverage for racialised=no vs. yes (a small gap, not the kind that would badly bias a trend), and state-level coverage is now tight (99.2-99.95%) with no extreme outlier. We proceed with the trend using only rows with a parsed timestamp, treating this as a **very close approximation of the true time trend** given the residual 0.5% gap, and note per-state coverage as a caveat rather than a blocker.

## Recovering missing timestamps from the relative time text

Rather than stopping at "99.5% coverage," we can recover almost all of the rest. `review_time` is the original Google Maps **relative** time text (e.g. `"2 years ago"`, `"Edited 3 months ago"`) that the `gmaps_poi` DB parsed into the absolute `review_timestamp` we've been using — and the review-level `retrieved_time` (the scrape timestamp needed as the reference point for "N years ago") is null for **5,722 of the 5,762 rows missing `review_timestamp` (99.3%)**, which is almost certainly *why* the DB couldn't parse them, not a separate problem.

The place-level `place_retrieved_time` column, however, is **never null** and only takes **6 distinct values** across the whole corpus (a handful of scrape batches) — the review-level `retrieved_time` is presumably a more precise per-review scrape stamp where available, but the place-level batch date is a reasonable stand-in reference point when it isn't: reviews for the same place were almost certainly scraped in the same session. Using `place_retrieved_time` as the reference date, a regex against `review_time` (`(?:Edited )?<a|N> <unit(s)> ago`, treating "Edited N ago" the same as the rest of the corpus's already-parsed timestamps — it's still the most recent activity signal available for that review, consistent with how the DB's own parsing already treats it) matches **100% of the non-null `review_time` text** among the missing rows. The estimate is coarse (year/month/week granularity, not exact day), which is fine for a yearly trend but would not be fine for day-level analysis.

In [ ]:
REL_TIME_PATTERN = re.compile(
    r"^(?:Edited\s+)?(a|an|\d+)\s+(year|month|week|day|hour|minute)s?\s+ago$", re.IGNORECASE
)
UNIT_TO_DAYS = {"year": 365.25, "month": 30.44, "week": 7, "day": 1, "hour": 1 / 24, "minute": 1 / (24 * 60)}

def parse_relative_time(text):
    m = REL_TIME_PATTERN.match(text) if isinstance(text, str) else None
    if not m:
        return None
    qty_raw, unit = m.groups()
    qty = 1 if qty_raw.lower() in ("a", "an") else int(qty_raw)
    return pd.Timedelta(days=qty * UNIT_TO_DAYS[unit.lower()])

missing_ts_mask = corpus["review_timestamp"].isna()
deltas = corpus.loc[missing_ts_mask, "review_time"].apply(parse_relative_time)
n_parsed = deltas.notna().sum()
print(f"missing timestamps: {missing_ts_mask.sum():,}; recoverable via relative-time text: {n_parsed:,} "
      f"({n_parsed / missing_ts_mask.sum():.1%})")

# review_timestamp is tz-aware (parsed from the gmaps_poi DB); place_retrieved_time is tz-naive and the
# estimate ends up nanosecond-precision. We only need year/month granularity for the trend, so build
# both sides as plain tz-naive 'us'-precision series and combine, rather than assigning in place.
base = corpus["review_timestamp"].dt.tz_localize(None).astype("datetime64[us]")
estimated = pd.Series(pd.NaT, index=corpus.index, dtype="datetime64[us]")
raw_est = (corpus.loc[missing_ts_mask, "retrieved_time"] - deltas).astype("datetime64[us]")
#if there are missing use the place_retrieved_time - parsed delta to estimate the review_timestamp
raw_est = (corpus.loc[missing_ts_mask, "place_retrieved_time"] - deltas).astype("datetime64[us]")
estimated.loc[missing_ts_mask] = raw_est

corpus["review_timestamp_est"] = base.fillna(estimated)
corpus["timestamp_is_estimated"] = missing_ts_mask & corpus["review_timestamp_est"].notna()

new_coverage = corpus["review_timestamp_est"].notna().mean()
print(f"timestamp coverage after recovery: {new_coverage:.2%} (was {has_ts.mean():.1%})")
print(f"still unrecoverable (no review_time text at all): {corpus['review_timestamp_est'].isna().sum():,}")

### Trend over time for all reviews

In [ ]:
# --- trend over time: all reviews vs social=yes reviews (actual + estimated timestamps) ---
corpus["review_year"] = corpus["review_timestamp_est"].dt.year
social_mask = corpus["social"] == "yes"

yearly_all = (
    corpus.groupby("review_year")
    .agg(
        n_reviews=("review_id", "size"),
        pct_estimated=("timestamp_is_estimated", "mean"),
    )
)
yearly_all = yearly_all[yearly_all["n_reviews"] >= 1]  # drop sparse partial years at the edges

yearly_social = (
    corpus.loc[social_mask]
    .groupby("review_year")
    .agg(
        n_reviews=("review_id", "size"),
        pct_estimated=("timestamp_is_estimated", "mean"),
    )
)
yearly_social = yearly_social[yearly_social["n_reviews"] >= 20]

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(9, 6), sharex=True, gridspec_kw={"height_ratios": [2, 1]})

ax1.plot(yearly_all.index, yearly_all["n_reviews"], marker="o", color="#4C72B0", label="all reviews")
ax1.plot(yearly_social.index, yearly_social["n_reviews"], marker="o", color="#C44E52", label="social = yes")
ax1.set_ylabel("reviews")
ax1.set_title("Review volume over time")
ax1.legend(fontsize=9)

width = 0.4
ax2.bar(yearly_all.index - width / 2, yearly_all["pct_estimated"], width=width, color="#4C72B0", alpha=0.5, label="all reviews")
ax2.bar(yearly_social.index + width / 2, yearly_social["pct_estimated"], width=width, color="#C44E52", alpha=0.5, label="social = yes")
ax2.set_ylabel("% estimated")
ax2.set_xlabel("review year")
ax2.legend(fontsize=8)

plt.tight_layout()
plt.savefig(FIGURES_DIR / "review_volume_and_pct_estimated_over_time.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
# --- pool pre-2016 years into a single bucket (too sparse to read year-by-year), then plot
# racialised rate (line, top) against review volume -- all reviews vs social=yes reviews (bars, bottom) ---

CUTOFF_YEAR = 2016

def make_bucket(year):
    return f"pre-{CUTOFF_YEAR}" if year < CUTOFF_YEAR else str(int(year))

def bucketed_rate_table(df, count_col, success_fn):
    d = df[df["review_year"] < 2026].copy()  # drop partial current year
    d["bucket"] = d["review_year"].apply(make_bucket)
    table = (
        d.groupby("bucket")
        .agg(n_reviews=(count_col, "size"), n_success=(count_col, success_fn))
    )
    order = [f"pre-{CUTOFF_YEAR}"] + [str(y) for y in range(CUTOFF_YEAR, 2026)]
    table = table.reindex(order).dropna()
    rates = table.apply(
        lambda r: pd.Series(wilson(r["n_success"], r["n_reviews"]), index=["rate", "ci_lo", "ci_hi"]),
        axis=1,
    )
    return table.join(rates)


# n_reviews here = all reviews per bucket, rate = social rate (not plotted, kept for volume bars)
corpus["review_year"] = corpus["review_timestamp_est"].dt.year
yearly_totals = bucketed_rate_table(corpus, "social", lambda s: (s == "yes").sum())

# n_reviews here = social reviews per bucket, rate = racialised rate
ts_social = corpus.loc[social_mask & corpus["review_timestamp_est"].notna()].copy()
ts_social["review_year"] = ts_social["review_timestamp_est"].dt.year
yearly_social = bucketed_rate_table(ts_social, "racialised", lambda s: (s == "yes").sum())

buckets = yearly_totals.index
x = list(range(len(buckets)))
width = 0.4
social_aligned = yearly_social.reindex(buckets)


fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 8), sharex=True, gridspec_kw={"height_ratios": [1, 1.2]})

# top: racialised rate, line + Wilson CI band
pct = social_aligned["rate"] * 100
ax1.plot(x, pct, marker="o", color="#C44E52", linewidth=2)
ax1.fill_between(x, social_aligned["ci_lo"] * 100, social_aligned["ci_hi"] * 100, color="#C44E52", alpha=0.15)
ax1.set_ylabel("racialised reviews,\nas % of social reviews")
print(f"N={int(yearly_social['n_reviews'].sum()):}")
# ax1.set_title(
#     f"Racialised rate and review volume by year (N={int(yearly_social['n_reviews'].sum()):,} social reviews",
    #f"pre-{CUTOFF_YEAR} pooled and 2026 partial year dropped)",
#     fontsize=16 , fontweight="bold"
# )


for spine in ["top", "right"]:
    ax1.spines[spine].set_visible(False)

# bottom: review volume, all reviews vs social = yes reviews
ax2.bar([i + width / 2 for i in x], social_aligned["n_reviews"], width=width, color="#C44E52", label="social = yes")
ax2.bar([i - width / 2 for i in x], yearly_totals["n_reviews"], width=width, color="#4C72B0", label="all reviews")
ax2.set_ylabel("reviews")
ax2.set_xlabel("review year")
ax2.set_xticks(x)
ax2.set_xticklabels(buckets, rotation=45, ha="right")
ax2.legend(fontsize=9)
for spine in ["top", "right"]:
    ax2.spines[spine].set_visible(False)

plt.tight_layout()
plt.savefig(FIGURES_DIR / "racialised_rate_pre2016_bucketed.png", dpi=150, bbox_inches="tight")
plt.show()



In [ ]:
#corpus[['review_timestamp', 'review_time', 'place_retrieved_time', 'retrieved_time', 'review_timestamp_est', 'timestamp_is_estimated']].head(10)

In [ ]:
corpus["review_year"] = corpus  ["review_timestamp_est"].dt.year
ts_social = corpus.loc[social_mask & corpus["review_timestamp_est"].notna()].copy()


year_range = ts_social["review_year"].describe()[["min", "max"]]
print(f"timestamped (actual + estimated) social reviews span years {int(year_range['min'])}-{int(year_range['max'])}")

yearly = (
    ts_social.groupby("review_year")
    .agg(
        social_reviews=("racialised", "size"),
        racialised_reviews=("racialised", lambda s: (s == "yes").sum()),
        pct_estimated=("timestamp_is_estimated", "mean"),
    )
)
# put all sparse  years together

yearly = yearly[yearly["racialised_reviews"]>=1]  # drop sparse partial years at the edges
yearly_rates = yearly.apply(
    lambda r: pd.Series(wilson(r["racialised_reviews"], r["social_reviews"]), index=["rate", "ci_lo", "ci_hi"]),
    axis=1,
)
yearly = yearly.join(yearly_rates)
print(display(yearly))

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(9, 6), sharex=True, gridspec_kw={"height_ratios": [2, 1]})
ax1.plot(yearly.index, yearly["rate"], marker="o", color="#C44E52")
ax1.fill_between(yearly.index, yearly["ci_lo"], yearly["ci_hi"], alpha=0.2, color="#C44E52")
ax1.set_ylabel("racialised rate")
ax1.set_title("Racialised rate over time (years with >=50 timestamped social reviews, actual + estimated)")
ax2.bar(yearly.index, yearly["social_reviews"], color="#4C72B0", label="total")
ax2.bar(yearly.index, yearly["social_reviews"] * yearly["pct_estimated"], color="#B4C7E7", label="estimated time share")
ax2.set_ylabel("social reviews")
ax2.set_xlabel("review year")
ax2.legend(fontsize=8)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "racialised_rate_yearly_timestamped_social_reviews.png", dpi=150, bbox_inches="tight")
plt.show()


**Takeaway.** Recovering timestamps from the relative-time text against `place_retrieved_time` lifts coverage from 99.5% to **100.00%** (only 40 reviews with no `review_time` text at all remain unrecoverable) — a genuinely complete temporal picture. The same qualitative pattern holds and is now backed by real, not recovered-estimate, data for nearly the whole corpus: the racialised rate falls to a low of **0.77% in 2021**, then climbs steadily to **2.30% in 2025** — roughly a 3x rise over four years. 2012-2015 remain too thin to read as a baseline (109-1,254 reviews/year). **2026 is still a partial year** (14,367 reviews vs. 66-80k in 2023-2025) so its lower rate (1.46%) should still be read as a truncation artifact rather than a reversal of the trend. One new observation from the estimation itself: with real-timestamp coverage now at 99.5% before recovery, the estimated share of any given year is tiny (roughly 0.1-2.9%, highest in the most recent, still-being-scraped years) — the recovery step is now a genuine last-mile correction, not a load-bearing part of the trend the way it was when coverage was 69.5%.

## Month and season granularity

Beyond the yearly trend, do racialised reviews cluster in particular months or seasons? Two caveats before diving in:

1. **Month-level analysis below uses only reviews with a genuinely parsed `review_timestamp`, not the recovered-estimate values built above.** The recovery step subtracts a relative-time delta from `place_retrieved_time`, which takes only 6 distinct values and is overwhelmingly clustered in late April 2026 (>99% of scrapes). Checking the estimated-share by month confirms why this matters: month 4 (April) is 6.4% estimated vs. 0.0-0.3% for every other month — an artifact of the reference date, not a real April spike in review activity. Year-level analysis (the yearly-trend subsections above) isn't affected by this because it pools 12 months together, but month-level granularity is, so we deliberately drop back to the pre-recovery, actually-parsed-only subset (99.4% of social reviews, n=501,162) for this section.
2. `pool_season` (outdoor = Apr-Sep, indoor = Oct-Mar) is a **simplifying assumption**: we only know when a review was *posted*, not when the visit happened, and there's no indoor/outdoor venue flag in the data — a review posted in October could describe a summer visit, or an indoor pool visited in October. Treat this as a coarse proxy, not a validated visit-season label.


In [ ]:
# select only the actually-parsed (non-estimated) timestamps for the month/season analysis, since the
# estimated timestamps are only accurate to the day and we want to avoid any bias from the
ts_social_actual = corpus.loc[
    social_mask & corpus["review_timestamp_est"].notna() & ~corpus["timestamp_is_estimated"]
].copy()

ts_social_actual["review_month"] = ts_social_actual["review_timestamp_est"].dt.month
ts_social_actual["review_year"] = ts_social_actual["review_timestamp_est"].dt.year

SEASON_MAP = {12: "Winter", 1: "Winter", 2: "Winter", 3: "Spring", 4: "Spring", 5: "Spring",
              6: "Summer", 7: "Summer", 8: "Summer", 9: "Autumn", 10: "Autumn", 11: "Autumn"}
ts_social_actual["meteorological_season"] = ts_social_actual["review_month"].map(SEASON_MAP)

# --- outdoor vs indoor pool season (May-September vs October-April) ---
POOL_SEASON_MAP = {m: ("outdoor season" if 5 <= m <= 9 else "indoor season") for m in range(1, 13)}
ts_social_actual["pool_season"] = ts_social_actual["review_month"].map(POOL_SEASON_MAP)

print(f"actually-parsed (non-estimated) social reviews with a timestamp: {len(ts_social_actual):,} "
      f"({len(ts_social_actual) / social_mask.sum():.1%} of all social reviews)")
print(f"actually-parsed (non-estimated) racialised reviews with a timestamp, racialised=yes: "
      f"{ts_social_actual['racialised'].eq('yes').sum()}"
      f"({ts_social_actual['racialised'].eq('yes').sum() / corpus['racialised'].eq('yes').sum():.1%} of all social reviews)")


In [ ]:
# ============================================================
# SECTION 0 — SETUP: parse timestamps, define season mappings
# ============================================================
ts_social_actual["review_month"] = ts_social_actual["review_timestamp_est"].dt.month
ts_social_actual["review_year"] = ts_social_actual["review_timestamp_est"].dt.year

SEASON_MAP = {12: "Winter", 1: "Winter", 2: "Winter", 3: "Spring", 4: "Spring", 5: "Spring",
              6: "Summer", 7: "Summer", 8: "Summer", 9: "Autumn", 10: "Autumn", 11: "Autumn"}
ts_social_actual["meteorological_season"] = ts_social_actual["review_month"].map(SEASON_MAP)

def get_pool_season_year(row):
    """Indoor season spans Oct-Apr across a year boundary, so Jan-Apr belongs to
    the indoor block that started the PREVIOUS October, not a fresh one."""
    month, year = row["review_month"], row["review_year"]
    if 5 <= month <= 9:
        return year, "outdoor"
    elif month in (10, 11, 12):
        return year, "indoor"
    else:  # Jan-Apr
        return year - 1, "indoor"

ts_social_actual[["pool_year", "pool_season"]] = ts_social_actual.apply(
    lambda r: pd.Series(get_pool_season_year(r)), axis=1
)

CUTOFF_YEAR = 2016
SEASON_ORDER = ["Winter", "Spring", "Summer", "Autumn"]
THIN_THRESHOLD = 30

# ============================================================
# SECTION 0b — REUSABLE HELPERS
# ============================================================
def add_wilson_ci(df, k_col, n_col):
    rates = df.apply(
        lambda r: pd.Series(wilson(r[k_col], r[n_col]), index=["rate", "ci_lo", "ci_hi"]), axis=1
    )
    return df.join(rates)

def draw_ticks_and_gridlines(ax_top, ax_bottom, tick_positions, tick_labels):
    """Forces tick visibility + manual tick marks + vertical guide lines through both panels."""
    ax_bottom.set_xticks(tick_positions)
    ax_bottom.set_xticklabels(tick_labels, rotation=45, ha="right")
    ax_bottom.tick_params(axis="x", which="major", bottom=True, top=False,
                           direction="out", length=8, width=1.2, color="black")
    ymin, ymax = ax_bottom.get_ylim()
    tick_len = (ymax - ymin) * 0.04
    for pos in tick_positions:
        ax_bottom.plot([pos, pos], [ymin - tick_len, ymin], color="black", linewidth=1.2,
                        clip_on=False, zorder=10)
        ax_top.axvline(pos, color="gray", linestyle="--", linewidth=0.6, alpha=0.4, zorder=0)
        ax_bottom.axvline(pos, color="gray", linestyle="--", linewidth=0.6, alpha=0.4, zorder=0)

def trim_trailing_thin(df, n_col, threshold=THIN_THRESHOLD):
    """Drops trailing bucket(s) below threshold (usually an incomplete in-progress period)."""
    n_trim = 0
    while df[n_col].iloc[-(n_trim + 1)] < threshold:
        n_trim += 1
    if n_trim > 0:
        print(f"dropping {n_trim} trailing thin bucket(s):")
        print(df.iloc[-n_trim:])
        df = df.iloc[:-n_trim].copy()
    return df

def two_panel_trend_plot(df, rate_col, ci_lo_col, ci_hi_col, n_col, x_col, label_col,
                          title, ylabel_top="racialised rate", ylabel_bottom="n reviews",
                          xlabel_bottom="", color="#C44E52", n_color="#4C72B0",
                          ylim_top=None, figsize=(15, 7.5), max_ticks=20):
    fig, (ax_rate, ax_n) = plt.subplots(2, 1, figsize=figsize, gridspec_kw={"height_ratios": [2.5, 1]})

    yerr_lo = (df[rate_col] - df[ci_lo_col]).clip(lower=0)
    yerr_hi = (df[ci_hi_col] - df[rate_col]).clip(lower=0)
    ax_rate.errorbar(df[x_col], df[rate_col], yerr=[yerr_lo, yerr_hi],
                      fmt="o-", color=color, ecolor=color, capsize=3, markersize=4, linewidth=1.2)
    ax_rate.set_ylabel(ylabel_top)
    ax_rate.set_title(title)
    if ylim_top:
        ax_rate.set_ylim(*ylim_top)
    ax_rate.grid(True, alpha=0.3)

    ax_n.bar(df[x_col], df[n_col], color=n_color, width=0.8)
    ax_n.set_ylabel(ylabel_bottom)
    ax_n.set_xlabel(xlabel_bottom)

    step = max(1, len(df) // max_ticks)
    tick_pos = df[x_col].iloc[::step].tolist()
    tick_labels = df[label_col].iloc[::step].tolist()
    draw_ticks_and_gridlines(ax_rate, ax_n, tick_pos, tick_labels)

    fig.tight_layout()
    fig.subplots_adjust(bottom=0.2, hspace=0.15)
    plt.savefig(FIGURES_DIR / f"trend_{slugify(title)}.png", dpi=150, bbox_inches="tight")
    plt.show()

In [ ]:
# ============================================================
# SECTION 1 — ALL-YEARS POOLED: by calendar month & by season
# ============================================================
by_month_pooled = ts_social_actual.groupby("review_month").agg(
    social_reviews=("racialised", "size"), racialised_reviews=("racialised", lambda s: (s == "yes").sum())
)
by_month_pooled = add_wilson_ci(by_month_pooled, "racialised_reviews", "social_reviews")

by_season_pooled = ts_social_actual.groupby("meteorological_season").agg(
    social_reviews=("racialised", "size"), racialised_reviews=("racialised", lambda s: (s == "yes").sum())
).reindex(SEASON_ORDER)
by_season_pooled = add_wilson_ci(by_season_pooled, "racialised_reviews", "social_reviews")

print("By calendar month (all years pooled):")
print(by_month_pooled)
print("\nBy meteorological season (all years pooled):")
print(by_season_pooled)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].errorbar(
    by_month_pooled.index, by_month_pooled["rate"],
    yerr=[by_month_pooled["rate"] - by_month_pooled["ci_lo"], by_month_pooled["ci_hi"] - by_month_pooled["rate"]],
    fmt="o-", color="#C44E52", ecolor="#C44E52", capsize=3,
)
axes[0].set_xlabel("calendar month")
axes[0].set_ylabel("racialised rate")
axes[0].set_title(f"By month, all years pooled (n={len(ts_social_actual):,})")
axes[0].set_xticks(range(1, 13))

axes[1].bar(
    SEASON_ORDER, by_season_pooled.loc[SEASON_ORDER, "rate"], color="#4C72B0",
    yerr=[by_season_pooled.loc[SEASON_ORDER, "rate"] - by_season_pooled.loc[SEASON_ORDER, "ci_lo"],
          by_season_pooled.loc[SEASON_ORDER, "ci_hi"] - by_season_pooled.loc[SEASON_ORDER, "rate"]],
    capsize=4,
)
axes[1].set_ylabel("racialised rate")
axes[1].set_title("By meteorological season, all years pooled")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "racialised_rate_by_month_and_season_pooled.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ============================================================
# SECTION 2 — YEAR-MONTH TREND (pre-2016 bucketed, monthly grain)
# ============================================================
def make_year_month_bucket(row):
    if row["review_year"] < CUTOFF_YEAR:
        return ("pre-2016", 0)
    return (str(row["review_year"]), row["review_month"])

ts_social_actual["year_month_bucket_key"] = ts_social_actual.apply(make_year_month_bucket, axis=1)

by_year_month = ts_social_actual.groupby("year_month_bucket_key").agg(
    social_reviews=("racialised", "size"), racialised_reviews=("racialised", lambda s: (s == "yes").sum())
).reset_index()
by_year_month["sort_year"] = by_year_month["year_month_bucket_key"].apply(lambda k: -1 if k[0] == "pre-2016" else int(k[0]))
by_year_month["sort_month"] = by_year_month["year_month_bucket_key"].apply(lambda k: k[1])
by_year_month = by_year_month.sort_values(["sort_year", "sort_month"]).drop(columns=["sort_year", "sort_month"])
by_year_month = by_year_month.set_index("year_month_bucket_key")
by_year_month = add_wilson_ci(by_year_month, "racialised_reviews", "social_reviews")

by_year_month["label"] = by_year_month.index.map(lambda k: "pre-2016" if k[0] == "pre-2016" else f"{k[0]}-{int(k[1]):02d}")
by_year_month = trim_trailing_thin(by_year_month, "social_reviews")
by_year_month["x_pos"] = range(len(by_year_month))

print(f"\nyear-month buckets after trimming: {len(by_year_month)}")

two_panel_trend_plot(
    by_year_month, rate_col="rate", ci_lo_col="ci_lo", ci_hi_col="ci_hi",
    n_col="social_reviews", x_col="x_pos", label_col="label",
    title=f"By year-month — racialised review rate (n={len(ts_social_actual):,} social reviews)",
    xlabel_bottom="year-month (pre-2016 bucketed)", ylim_top=(0, 0.1),
)


In [ ]:
# ============================================================
# SECTION 3 — SEASON TREND, CHRONOLOGICAL (post-2016, one season per point)
# ============================================================
ts_post2016 = ts_social_actual[ts_social_actual["review_year"] >= CUTOFF_YEAR].copy()

SEASON_MONTH_ORDER = {"Winter": 0, "Spring": 1, "Summer": 2, "Autumn": 3}

season_chrono = ts_post2016.groupby(["review_year", "meteorological_season"]).agg(
    social_reviews=("racialised", "size"), racialised_reviews=("racialised", lambda s: (s == "yes").sum())
)
season_chrono = add_wilson_ci(season_chrono, "racialised_reviews", "social_reviews").reset_index()
season_chrono["season_order"] = season_chrono["meteorological_season"].map(SEASON_MONTH_ORDER)
season_chrono = season_chrono.sort_values(["review_year", "season_order"]).reset_index(drop=True)
season_chrono["label"] = season_chrono["review_year"].astype(str) + "-" + season_chrono["meteorological_season"]
season_chrono["x_pos"] = range(len(season_chrono))

two_panel_trend_plot(
    season_chrono, rate_col="rate", ci_lo_col="ci_lo", ci_hi_col="ci_hi",
    n_col="social_reviews", x_col="x_pos", label_col="label",
    title=f"Racialised rate by season, chronological (post-{CUTOFF_YEAR})",
    xlabel_bottom="year-season", figsize=(16, 6.5),
)



In [ ]:
# ============================================================
# SECTION 4 — INDOOR/OUTDOOR TREND, CHRONOLOGICAL (post-2016, corrected boundary)
# ============================================================
def make_pool_bucket(row):
    if row["pool_year"] < CUTOFF_YEAR:
        return ("pre-2016", "pool")
    return (str(row["pool_year"]), row["pool_season"])

ts_social_actual["pool_bucket_key"] = ts_social_actual.apply(make_pool_bucket, axis=1)

pool_chrono = ts_social_actual.groupby("pool_bucket_key").agg(
    social_reviews=("racialised", "size"), racialised_reviews=("racialised", lambda s: (s == "yes").sum())
).reset_index()
pool_chrono = add_wilson_ci(pool_chrono, "racialised_reviews", "social_reviews")

pool_chrono["year_part"] = pool_chrono["pool_bucket_key"].apply(lambda k: k[0])
pool_chrono["season_part"] = pool_chrono["pool_bucket_key"].apply(lambda k: k[1])
pool_chrono["sort_year"] = pool_chrono["year_part"].apply(lambda y: -1 if y == "pre-2016" else int(y))
pool_chrono["sort_season"] = pool_chrono["season_part"].apply(lambda s: 0 if s in ("outdoor", "pool") else 1)
pool_chrono = pool_chrono.sort_values(["sort_year", "sort_season"]).reset_index(drop=True)

pool_chrono["label"] = pool_chrono.apply(
    lambda r: "pre-2016" if r["year_part"] == "pre-2016"
    else (f"{r['year_part']}-outdoor" if r["season_part"] == "outdoor"
          else f"{r['year_part']}/{int(r['year_part'])+1}-indoor"),
    axis=1
)
pool_chrono = trim_trailing_thin(pool_chrono, "social_reviews")
pool_chrono["x_pos"] = range(len(pool_chrono))

two_panel_trend_plot(
    pool_chrono, rate_col="rate", ci_lo_col="ci_lo", ci_hi_col="ci_hi",
    n_col="social_reviews", x_col="x_pos", label_col="label",
    title=f"Racialised rate by indoor/outdoor season, chronological (post-{CUTOFF_YEAR}, corrected)",
    xlabel_bottom="season block (chronological)", color="#4C72B0", n_color="#55A868",
    figsize=(16, 6.5),
)

In [ ]:
# ============================================================
# SECTION 3b — SUMMER vs OTHER MONTHS, CHRONOLOGICAL (post-2016)
# ============================================================
ts_post2016["summer_vs_other"] = np.where(
    ts_post2016["meteorological_season"] == "Summer", "Summer", "Other months"
)

summer_chrono = ts_post2016.groupby(["review_year", "summer_vs_other"]).agg(
    social_reviews=("racialised", "size"), racialised_reviews=("racialised", lambda s: (s == "yes").sum())
)
summer_chrono = add_wilson_ci(summer_chrono, "racialised_reviews", "social_reviews").reset_index()

# order: Summer first, then Other months, within each year — pick whichever reads better
GROUP_ORDER = {"Summer": 0, "Other months": 1}
summer_chrono["group_order"] = summer_chrono["summer_vs_other"].map(GROUP_ORDER)
summer_chrono = summer_chrono.sort_values(["review_year", "group_order"]).reset_index(drop=True)
summer_chrono["label"] = summer_chrono["review_year"].astype(str) + "-" + summer_chrono["summer_vs_other"]
summer_chrono["x_pos"] = range(len(summer_chrono))

print(summer_chrono[["label", "social_reviews", "rate", "ci_lo", "ci_hi"]].to_string())

two_panel_trend_plot(
    summer_chrono, rate_col="rate", ci_lo_col="ci_lo", ci_hi_col="ci_hi",
    n_col="social_reviews", x_col="x_pos", label_col="label",
    title=f"Racialised rate: Summer vs Other months, chronological (post-{CUTOFF_YEAR})",
    xlabel_bottom="year-group", color="#C44E52", figsize=(16, 6.5),
)

what is the trend  over seasons  ( summer vs other seasons) and the indor and outdoor seasons?

**Takeaway.** There is a visible, not-flat pattern: summer months (Jun/Jul/Aug: 1.8-2.0%) run clearly above winter/spring/autumn (1.2-1.3%), and the meteorological-season cut confirms it — Summer sits at **1.92%** vs. 1.21-1.29% for the other three seasons. The simplified `pool_season` split points the same way (outdoor season ~1.80% vs. indoor season ~1.19%). This is plausibly a real seasonal effect — outdoor public pools draw much larger and more diverse crowds in summer, which is exactly the kind of situation the `Crowd`/`Etiquette` themes (see the dimension-content-overview section later in the notebook) describe — though it's still a review-*posting*-date pattern, not a confirmed visit-date one (see the `pool_season` caveat above).

## Dimension prevalence over time (racialised subset only, N=7,768)

The same year-level lens, but now applied to `theme`, `context`, `connotation`, and `approach` — computed only on the racialised=yes subset, so these are **within-racialised-reviews prevalence shares**, not the overall racialised rate. Per-year N is small before 2018 (2-245 racialised reviews/year, vs. 404-1,566/year from 2018 on) — too thin to trust individual pre-2018 years at this finer grain (splitting an already-small subset by year *and* category multiplies the sparsity problem), even though the extended-window subsection above showed individual years work fine back to 2012 for the *overall rate* (which uses the much larger `social_reviews` denominator). So here we bucket everything before 2018 into a single `pre-2018` row and show 2018-2026 individually, keeping 2026 marked as partial as usual. Month/season breakdowns are skipped for these dimensions: splitting 7,768 already-multi-label reviews by 12 months would leave many month x category cells with single-digit counts, too sparse to plot meaningfully.

In [ ]:
# ============================================================================
# SECTION 0 — SETUP: load racialised subset, multi-label matching, shared helpers
# ============================================================================
import numpy as np

racialised_ts = ts_social[(ts_social["racialised"] == "yes") & ~ts_social["review_timestamp"].isnull()].copy()
racialised_ts["review_year"] = racialised_ts["review_timestamp_est"].dt.year
racialised_ts["review_month"] = racialised_ts["review_timestamp_est"].dt.month
racialised_ts["meteorological_season"] = racialised_ts["review_month"].map(SEASON_MAP)

def get_pool_season_year(row):
    """Indoor season spans Oct-Apr across a year boundary, so Jan-Apr belongs to
    the indoor block that started the PREVIOUS October, not a fresh one."""
    month, year = row["review_month"], row["review_year"]
    if 5 <= month <= 9:
        return year, "outdoor"
    elif month in (10, 11, 12):
        return year, "indoor"
    else:
        return year - 1, "indoor"

racialised_ts[["pool_year", "pool_season"]] = racialised_ts.apply(
    lambda r: pd.Series(get_pool_season_year(r)), axis=1
)

CUTOFF_YEAR = 2018  # racialised subset (N~5,693-7,768 depending on filters) is much smaller than the overall social set
THIN_THRESHOLD = 30
SEASON_ORDER = ["Winter", "Spring", "Summer", "Autumn"]

CONNOTATION_LABELS = ["Negative", "Neutral", "Positive"]
CONTEXT_LABELS = ["generalised", "incident"]
APPROACH_LABELS = ["Explicit", "Implicit", "Coded"]

def label_match(cell, label):
    """Multi-label-safe matching — a review can carry >1 label per dimension, stored as ndarray."""
    if isinstance(cell, (list, tuple, set, np.ndarray)):
        return label in cell
    if isinstance(cell, str):
        return cell == label
    return False

def prevalence_with_ci_by_group(df, group_col, label_col, labels):
    """Generic: for each value of group_col, compute share (+ Wilson CI) of rows carrying each label."""
    rows = []
    for group_val, g in df.groupby(group_col):
        n = len(g)
        row = {group_col: group_val, "n_reviews": n}
        for lab in labels:
            k = g[label_col].apply(lambda cell: label_match(cell, lab)).sum()
            rate, ci_lo, ci_hi = wilson(k, n)
            row[f"{lab}_rate"] = rate
            row[f"{lab}_ci_lo"] = ci_lo
            row[f"{lab}_ci_hi"] = ci_hi
        rows.append(row)
    return pd.DataFrame(rows).set_index(group_col)

def trim_trailing_thin(df, n_col, threshold=THIN_THRESHOLD):
    n_trim = 0
    while df[n_col].iloc[-(n_trim + 1)] < threshold:
        n_trim += 1
    if n_trim > 0:
        print(f"dropping {n_trim} trailing thin bucket(s):")
        print(df.iloc[-n_trim:])
        df = df.iloc[:-n_trim].copy()
    return df

def draw_ticks_and_gridlines(ax_top, ax_bottom, tick_positions, tick_labels):
    ax_bottom.set_xticks(tick_positions)
    ax_bottom.set_xticklabels(tick_labels, rotation=45, ha="right")
    ax_bottom.tick_params(axis="x", which="major", bottom=True, top=False,
                           direction="out", length=8, width=1.2, color="black")
    ymin, ymax = ax_bottom.get_ylim()
    tick_len = (ymax - ymin) * 0.04
    for pos in tick_positions:
        ax_bottom.plot([pos, pos], [ymin - tick_len, ymin], color="black", linewidth=1.2,
                        clip_on=False, zorder=10)
        ax_top.axvline(pos, color="gray", linestyle="--", linewidth=0.6, alpha=0.4, zorder=0)
        ax_bottom.axvline(pos, color="gray", linestyle="--", linewidth=0.6, alpha=0.4, zorder=0)

def plot_prevalence_bands(df, labels, title, colors=None, xlabel="", n_threshold=None,
                           ylim=None, figsize=(15, 7.5), max_ticks=20, n_bar_color="#4C72B0"):
    """One reusable function for every prevalence-over-time plot: line + CI band on top,
    n_reviews bars below. If n_threshold is set, thin buckets are dropped and x_pos renumbered."""
    if n_threshold is not None:
        dropped = df[df["n_reviews"] < n_threshold]
        if len(dropped) > 0:
            print(f"excluding {len(dropped)} bucket(s) with n < {n_threshold}:")
            print(dropped[["label", "n_reviews"]].to_string())
        df = df[df["n_reviews"] >= n_threshold].copy()
        df["x_pos"] = range(len(df))
        title = f"{title} (excluding n < {n_threshold})"

    if colors is None:
        palette = plt.cm.tab10.colors
        colors = {lab: palette[i % len(palette)] for i, lab in enumerate(labels)}

    fig, (ax_rate, ax_n) = plt.subplots(2, 1, figsize=figsize, gridspec_kw={"height_ratios": [2.5, 1]})

    for lab in labels:
        rate, ci_lo, ci_hi = df[f"{lab}_rate"], df[f"{lab}_ci_lo"], df[f"{lab}_ci_hi"]
        ax_rate.plot(df["x_pos"], rate, marker="o", markersize=3, linewidth=1, label=lab, color=colors[lab])
        ax_rate.fill_between(df["x_pos"], ci_lo, ci_hi, color=colors[lab], alpha=0.15, linewidth=0)

    if ylim is None:
        ci_lo_cols = [f"{s}_ci_lo" for s in labels]
        ci_hi_cols = [f"{s}_ci_hi" for s in labels]
        y_floor = max(0, df[ci_lo_cols].min().min() - 0.05)
        y_ceil = min(1.02, df[ci_hi_cols].max().max() + 0.03)
        ylim = (y_floor, y_ceil)

    ax_rate.set_ylabel("share of racialised reviews")
    ax_rate.set_title(title)
    ax_rate.set_ylim(*ylim)
    ax_rate.legend(fontsize=8)
    ax_rate.grid(True, alpha=0.3)

    ax_n.bar(df["x_pos"], df["n_reviews"], color=n_bar_color, width=0.8)
    ax_n.set_ylabel("n racialised reviews")
    ax_n.set_xlabel(xlabel)

    step = max(1, len(df) // max_ticks)
    tick_pos = df["x_pos"].iloc[::step].tolist()
    tick_labels = df["label"].iloc[::step].tolist()
    draw_ticks_and_gridlines(ax_rate, ax_n, tick_pos, tick_labels)

    fig.tight_layout()
    fig.subplots_adjust(bottom=0.2, hspace=0.15)
    plt.savefig(FIGURES_DIR / f"bands_{slugify(title)}.png", dpi=150, bbox_inches="tight")
    plt.show()

# quick sanity check on label storage (run once to confirm assumptions above)
for col, expected_labels in [("connotation", CONNOTATION_LABELS), ("context", CONTEXT_LABELS), ("approach", APPROACH_LABELS)]:
    print(f"\n=== {col} ===")
    print("cell types:", racialised_ts[col].apply(type).value_counts().to_dict())
    print("sample values:", racialised_ts[col].head(3).tolist())

In [ ]:


# ============================================================================
# SECTION 1 — MONTHLY BUCKETS (pre-2018 pooled), per-dimension prevalence
# ============================================================================
def make_bucket(row):
    if row["review_year"] < CUTOFF_YEAR:
        return (f"pre-{CUTOFF_YEAR}", 0)
    return (str(row["review_year"]), row["review_month"])

racialised_ts["month_bucket_key"] = racialised_ts.apply(make_bucket, axis=1)

def build_bucket_base(df, group_col, cutoff_year, thin_threshold=THIN_THRESHOLD):
    n_by_bucket = df.groupby(group_col).size()
    idx = n_by_bucket.reset_index(name="n_reviews")
    idx["sort_year"] = idx[group_col].apply(lambda k: -1 if k[0] == f"pre-{cutoff_year}" else int(k[0]))
    idx["sort_month"] = idx[group_col].apply(lambda k: k[1])
    idx = idx.sort_values(["sort_year", "sort_month"]).drop(columns=["sort_year", "sort_month"])
    idx["label"] = idx[group_col].apply(lambda k: f"pre-{cutoff_year}" if k[0] == f"pre-{cutoff_year}" else f"{k[0]}-{int(k[1]):02d}")
    idx = idx.set_index(group_col)
    idx = trim_trailing_thin(idx, "n_reviews", thin_threshold)
    idx["x_pos"] = range(len(idx))
    return idx

month_bucket_base = build_bucket_base(racialised_ts, "month_bucket_key", CUTOFF_YEAR)
print(f"final monthly buckets: {len(month_bucket_base)}")

def join_prevalence(df, base, group_col, label_col, labels):
    prev = prevalence_with_ci_by_group(df, group_col, label_col, labels).drop(columns=["n_reviews"])
    return base.join(prev)

connotation_monthly = join_prevalence(racialised_ts, month_bucket_base, "month_bucket_key", "connotation", CONNOTATION_LABELS)
context_monthly = join_prevalence(racialised_ts, month_bucket_base, "month_bucket_key", "context", CONTEXT_LABELS)
approach_monthly = join_prevalence(racialised_ts, month_bucket_base, "month_bucket_key", "approach", APPROACH_LABELS)

plot_prevalence_bands(connotation_monthly, CONNOTATION_LABELS,
                       f"Connotation prevalence by month (racialised subset, N={len(racialised_ts):,})",
                       colors={"Negative": "#C44E52", "Neutral": "#DD8452", "Positive": "#55A868"},
                       xlabel=f"year-month (pre-{CUTOFF_YEAR} bucketed)", ylim=(0, 1))
plot_prevalence_bands(context_monthly, CONTEXT_LABELS,
                       f"Context prevalence by month (racialised subset, N={len(racialised_ts):,})",
                       colors={"generalised": "#4C72B0", "incident": "#DD8452"},
                       xlabel=f"year-month (pre-{CUTOFF_YEAR} bucketed)", ylim=(0, 1))
plot_prevalence_bands(approach_monthly, APPROACH_LABELS,
                       f"Approach prevalence by month (racialised subset, N={len(racialised_ts):,})",
                       colors={"Explicit": "#55A868", "Implicit": "#DD8452", "Coded": "#8172B2"},
                       xlabel=f"year-month (pre-{CUTOFF_YEAR} bucketed)", ylim=(0, 1))

In [ ]:
# ============================================================================
# SECTION 2 — COMBINED DOMINANT-LABEL VIEW (monthly grain)
# ============================================================================
combined4_monthly = month_bucket_base.copy()
for lab, src in [("Negative", connotation_monthly), ("generalised", context_monthly),
                 ("incident", context_monthly), ("Explicit", approach_monthly)]:
    combined4_monthly[f"{lab}_rate"] = src[f"{lab}_rate"]
    combined4_monthly[f"{lab}_ci_lo"] = src[f"{lab}_ci_lo"]
    combined4_monthly[f"{lab}_ci_hi"] = src[f"{lab}_ci_hi"]

series4 = ["Negative", "generalised", "incident", "Explicit"]
colors4 = {"Negative": "#C44E52", "generalised": "#4C72B0", "incident": "#DD8452", "Explicit": "#55A868"}
plot_prevalence_bands(combined4_monthly, series4,
                       f"Negative, Generalised, Incident, Explicit — prevalence by month (racialised subset, N={len(racialised_ts):,})",
                       colors=colors4, xlabel=f"year-month (pre-{CUTOFF_YEAR} bucketed)", ylim=(0, 1))

# zoomed 3-series version (drop incident) + n<30 filtered version
series3 = ["Negative", "generalised", "Explicit"]
colors3 = {"Negative": "#C44E52", "generalised": "#4C72B0", "Explicit": "#55A868"}
plot_prevalence_bands(combined4_monthly, series3,
                       f"Negative, Generalised, Explicit — prevalence by month (racialised subset, N={len(racialised_ts):,})",
                       colors=colors3, xlabel=f"year-month (pre-{CUTOFF_YEAR} bucketed)")
plot_prevalence_bands(combined4_monthly, series3,
                       f"Negative, Generalised, Explicit — prevalence by month (racialised subset, N={len(racialised_ts):,})",
                       colors=colors3, xlabel=f"year-month (pre-{CUTOFF_YEAR} bucketed, thin months dropped)",
                       n_threshold=N_THRESHOLD if 'N_THRESHOLD' in dir() else 30)


In [ ]:
# ============================================================================
# SECTION 3 — YEAR-LEVEL VIEW (pre-2018 pooled), top themes + all dimensions
# ============================================================================
racialised_ts["year_bucket"] = np.where(
    racialised_ts["review_year"] < CUTOFF_YEAR, f"pre-{CUTOFF_YEAR}", racialised_ts["review_year"].astype(int).astype(str)
)
YEAR_ORDER = [f"pre-{CUTOFF_YEAR}"] + [str(y) for y in range(CUTOFF_YEAR, 2027)]
YEAR_ORDER = [y for y in YEAR_ORDER if y in racialised_ts["year_bucket"].unique()]

print("racialised reviews per year_bucket:")
print(racialised_ts["year_bucket"].value_counts().reindex(YEAR_ORDER))

def prevalence_by_year(df, dim, categories):
    n_per_year = df.groupby("year_bucket").size()
    exploded = df[[dim, "year_bucket"]].explode(dim)
    out = {}
    for cat in categories:
        cnt = exploded.loc[exploded[dim] == cat].groupby("year_bucket").size()
        out[cat] = (cnt / n_per_year).reindex(YEAR_ORDER)
    return pd.DataFrame(out)

TOP_THEMES = racialised_df["theme"].explode().value_counts().head(5).index.tolist()
theme_prev = prevalence_by_year(racialised_ts, "theme", TOP_THEMES)
context_prev = prevalence_by_year(racialised_ts, "context", CONTEXT_LABELS)
connotation_prev = prevalence_by_year(racialised_ts, "connotation", CONNOTATION_LABELS)
approach_prev = prevalence_by_year(racialised_ts, "approach", APPROACH_LABELS)

def plot_year_level(prev_df, categories, title, colors=None, ylabel="share of racialised reviews"):
    fig, ax = plt.subplots(figsize=(10, 5))
    for cat in categories:
        kwargs = {"color": colors[cat]} if colors else {}
        ax.plot(range(len(YEAR_ORDER)), prev_df[cat], marker="o", label=cat, **kwargs)
    ax.set_xticks(range(len(YEAR_ORDER)))
    ax.set_xticklabels(YEAR_ORDER, rotation=45, ha="right")
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / f"yearlevel_{slugify(title)}.png", dpi=150, bbox_inches="tight")
    plt.show()

plot_year_level(theme_prev, TOP_THEMES, f"Top-5 theme prevalence over time (racialised subset, N={len(racialised_ts):,})")
plot_year_level(context_prev, CONTEXT_LABELS, f"Context prevalence over time (racialised subset, N={len(racialised_ts):,})",
                 colors={"generalised": "#4C72B0", "incident": "#DD8452"})
plot_year_level(connotation_prev, CONNOTATION_LABELS, f"Connotation prevalence over time (racialised subset, N={len(racialised_ts):,})",
                 colors={"Negative": "#C44E52", "Neutral": "#DD8452", "Positive": "#55A868"})
plot_year_level(approach_prev, APPROACH_LABELS, f"Approach prevalence over time (racialised subset, N={len(racialised_ts):,})",
                 colors={"Explicit": "#55A868", "Implicit": "#DD8452", "Coded": "#8172B2"})

dominant_prev_year = pd.DataFrame({
    "Negative (connotation)": connotation_prev["Negative"],
    "Generalised (context)": context_prev["generalised"],
    "Explicit (approach)": approach_prev["Explicit"],
}, index=YEAR_ORDER)
plot_year_level(dominant_prev_year, dominant_prev_year.columns,
                 f"Dominant label prevalence over time — Negative, Generalised, Explicit (N={len(racialised_ts):,})",
                 colors={"Negative (connotation)": "#C44E52", "Generalised (context)": "#4C72B0", "Explicit (approach)": "#55A868"})

In [ ]:
# ============================================================================
# SECTION 4 — POOL SEASON (indoor/outdoor), chronological, corrected year boundary
# ============================================================================
def make_pool_bucket(row):
    if row["pool_year"] < CUTOFF_YEAR:
        return (f"pre-{CUTOFF_YEAR}", "pool")
    return (str(row["pool_year"]), row["pool_season"])

racialised_ts["pool_bucket_key"] = racialised_ts.apply(make_pool_bucket, axis=1)

n_by_pool = racialised_ts.groupby("pool_bucket_key").size().reset_index(name="n_reviews")
n_by_pool["year_part"] = n_by_pool["pool_bucket_key"].apply(lambda k: k[0])
n_by_pool["season_part"] = n_by_pool["pool_bucket_key"].apply(lambda k: k[1])
n_by_pool["sort_year"] = n_by_pool["year_part"].apply(lambda y: -1 if y == f"pre-{CUTOFF_YEAR}" else int(y))
n_by_pool["sort_season"] = n_by_pool["season_part"].apply(lambda s: 0 if s in ("outdoor", "pool") else 1)
n_by_pool = n_by_pool.sort_values(["sort_year", "sort_season"]).set_index("pool_bucket_key")
n_by_pool["label"] = n_by_pool.apply(
    lambda r: f"pre-{CUTOFF_YEAR}" if r["year_part"] == f"pre-{CUTOFF_YEAR}"
    else (f"{r['year_part']}-outdoor" if r["season_part"] == "outdoor" else f"{r['year_part']}/{int(r['year_part'])+1}-indoor"),
    axis=1
)
n_by_pool = trim_trailing_thin(n_by_pool, "n_reviews")
n_by_pool["x_pos"] = range(len(n_by_pool))

print(f"\npool-season buckets: {len(n_by_pool)} — NOTE small racialised subset means these are likely THIN;")
print(n_by_pool[["label", "n_reviews"]].to_string())

pool_connotation = n_by_pool.join(prevalence_with_ci_by_group(racialised_ts, "pool_bucket_key", "connotation", CONNOTATION_LABELS).drop(columns=["n_reviews"]))
pool_context = n_by_pool.join(prevalence_with_ci_by_group(racialised_ts, "pool_bucket_key", "context", CONTEXT_LABELS).drop(columns=["n_reviews"]))
pool_approach = n_by_pool.join(prevalence_with_ci_by_group(racialised_ts, "pool_bucket_key", "approach", APPROACH_LABELS).drop(columns=["n_reviews"]))

plot_prevalence_bands(pool_connotation, CONNOTATION_LABELS,
                       f"Connotation prevalence by indoor/outdoor season (racialised subset, N={len(racialised_ts):,})",
                       colors={"Negative": "#C44E52", "Neutral": "#DD8452", "Positive": "#55A868"},
                       xlabel="season block (chronological)", ylim=(0, 1), figsize=(16, 6.5))
plot_prevalence_bands(pool_context, CONTEXT_LABELS,
                       f"Context prevalence by indoor/outdoor season (racialised subset, N={len(racialised_ts):,})",
                       colors={"generalised": "#4C72B0", "incident": "#DD8452"},
                       xlabel="season block (chronological)", ylim=(0, 1), figsize=(16, 6.5))
plot_prevalence_bands(pool_approach, APPROACH_LABELS,
                       f"Approach prevalence by indoor/outdoor season (racialised subset, N={len(racialised_ts):,})",
                       colors={"Explicit": "#55A868", "Implicit": "#DD8452", "Coded": "#8172B2"},
                       xlabel="season block (chronological)", ylim=(0, 1), figsize=(16, 6.5))

In [ ]:
# ============================================================================
# SECTION 5 — METEOROLOGICAL SEASON (4-way), chronological
# ============================================================================
racialised_post_cutoff = racialised_ts[racialised_ts["review_year"] >= CUTOFF_YEAR].copy()
SEASON_MONTH_ORDER = {"Winter": 0, "Spring": 1, "Summer": 2, "Autumn": 3}

n_by_season = racialised_post_cutoff.groupby(["review_year", "meteorological_season"]).size().reset_index(name="n_reviews")
n_by_season["season_order"] = n_by_season["meteorological_season"].map(SEASON_MONTH_ORDER)
n_by_season = n_by_season.sort_values(["review_year", "season_order"]).reset_index(drop=True)
n_by_season["label"] = n_by_season["review_year"].astype(str) + "-" + n_by_season["meteorological_season"]
n_by_season["season_bucket_key"] = list(zip(n_by_season["review_year"], n_by_season["meteorological_season"]))
n_by_season = n_by_season.set_index("season_bucket_key")
n_by_season["x_pos"] = range(len(n_by_season))

print(f"\nseason buckets: {len(n_by_season)} — check n_reviews for thin early-year cells:")
print(n_by_season[["label", "n_reviews"]].to_string())

racialised_post_cutoff["season_bucket_key"] = list(zip(racialised_post_cutoff["review_year"], racialised_post_cutoff["meteorological_season"]))

season_connotation = n_by_season.join(prevalence_with_ci_by_group(racialised_post_cutoff, "season_bucket_key", "connotation", CONNOTATION_LABELS).drop(columns=["n_reviews"]))
season_context = n_by_season.join(prevalence_with_ci_by_group(racialised_post_cutoff, "season_bucket_key", "context", CONTEXT_LABELS).drop(columns=["n_reviews"]))
season_approach = n_by_season.join(prevalence_with_ci_by_group(racialised_post_cutoff, "season_bucket_key", "approach", APPROACH_LABELS).drop(columns=["n_reviews"]))

plot_prevalence_bands(season_connotation, CONNOTATION_LABELS,
                       f"Connotation prevalence by season, chronological (racialised subset, post-{CUTOFF_YEAR})",
                       colors={"Negative": "#C44E52", "Neutral": "#DD8452", "Positive": "#55A868"},
                       xlabel="year-season", ylim=(0, 1), figsize=(16, 6.5))
plot_prevalence_bands(season_context, CONTEXT_LABELS,
                       f"Context prevalence by season, chronological (racialised subset, post-{CUTOFF_YEAR})",
                       colors={"generalised": "#4C72B0", "incident": "#DD8452"},
                       xlabel="year-season", ylim=(0, 1), figsize=(16, 6.5))
plot_prevalence_bands(season_approach, APPROACH_LABELS,
                       f"Approach prevalence by season, chronological (racialised subset, post-{CUTOFF_YEAR})",
                       colors={"Explicit": "#55A868", "Implicit": "#DD8452", "Coded": "#8172B2"},
                       xlabel="year-season", ylim=(0, 1), figsize=(16, 6.5))

In [ ]:

# ============================================================================
# SECTION 6 — SUMMER vs OTHER MONTHS, chronological
# ============================================================================
racialised_post_cutoff["summer_vs_other"] = np.where(
    racialised_post_cutoff["meteorological_season"] == "Summer", "Summer", "Other months"
)

n_by_summer = racialised_post_cutoff.groupby(["review_year", "summer_vs_other"]).size().reset_index(name="n_reviews")
GROUP_ORDER = {"Summer": 0, "Other months": 1}
n_by_summer["group_order"] = n_by_summer["summer_vs_other"].map(GROUP_ORDER)
n_by_summer = n_by_summer.sort_values(["review_year", "group_order"]).reset_index(drop=True)
n_by_summer["label"] = n_by_summer["review_year"].astype(str) + "-" + n_by_summer["summer_vs_other"]
n_by_summer["summer_bucket_key"] = list(zip(n_by_summer["review_year"], n_by_summer["summer_vs_other"]))
n_by_summer = n_by_summer.set_index("summer_bucket_key")
n_by_summer["x_pos"] = range(len(n_by_summer))

print(f"\nsummer-vs-other buckets: {len(n_by_summer)}")
print(n_by_summer[["label", "n_reviews"]].to_string())

racialised_post_cutoff["summer_bucket_key"] = list(zip(racialised_post_cutoff["review_year"], racialised_post_cutoff["summer_vs_other"]))

summer_connotation = n_by_summer.join(prevalence_with_ci_by_group(racialised_post_cutoff, "summer_bucket_key", "connotation", CONNOTATION_LABELS).drop(columns=["n_reviews"]))
summer_context = n_by_summer.join(prevalence_with_ci_by_group(racialised_post_cutoff, "summer_bucket_key", "context", CONTEXT_LABELS).drop(columns=["n_reviews"]))
summer_approach = n_by_summer.join(prevalence_with_ci_by_group(racialised_post_cutoff, "summer_bucket_key", "approach", APPROACH_LABELS).drop(columns=["n_reviews"]))

plot_prevalence_bands(summer_connotation, CONNOTATION_LABELS,
                       f"Connotation prevalence: Summer vs Other months (racialised subset, post-{CUTOFF_YEAR})",
                       colors={"Negative": "#C44E52", "Neutral": "#DD8452", "Positive": "#55A868"},
                       xlabel="year-group", ylim=(0, 1), figsize=(16, 6.5))
plot_prevalence_bands(summer_context, CONTEXT_LABELS,
                       f"Context prevalence: Summer vs Other months (racialised subset, post-{CUTOFF_YEAR})",
                       colors={"generalised": "#4C72B0", "incident": "#DD8452"},
                       xlabel="year-group", ylim=(0, 1), figsize=(16, 6.5))
plot_prevalence_bands(summer_approach, APPROACH_LABELS,
                       f"Approach prevalence: Summer vs Other months (racialised subset, post-{CUTOFF_YEAR})",
                       colors={"Explicit": "#55A868", "Implicit": "#DD8452", "Coded": "#8172B2"},
                       xlabel="year-group", ylim=(0, 1), figsize=(16, 6.5))

In [ ]:
racialised_ts = ts_social[(ts_social["racialised"] == "yes") & ~ts_social["review_timestamp"].isnull()].copy()
racialised_ts["review_year"] = racialised_ts["review_timestamp_est"].dt.year
racialised_ts.info()

In [ ]:
for col, expected_labels in [
    ("connotation", ["Negative", "Neutral", "Positive"]),
    ("context", ["generalised", "incident"]),
    ("approach", ["Explicit", "Implicit", "Coded"]),
]:
    print(f"\n=== {col} ===")
    # what types of cells exist?
    print("cell types:", racialised_ts[col].apply(type).value_counts().to_dict())

    # sample a few actual values, whatever shape they're in
    print("sample values:")
    print(racialised_ts[col].head(5).tolist())

    # if lists, what are the unique elements actually inside them?
    exploded = racialised_ts[col].explode()
    print("unique values found (exploded):", exploded.dropna().unique()[:15])

In [ ]:
# --- 0. multi-label-safe matching (since a review can carry >1 label per dimension) ---
import numpy as np

def label_match(cell, label):
    if isinstance(cell, (list, tuple, set, np.ndarray)):
        return label in cell
    if isinstance(cell, str):
        return cell == label
    return False # NaN or unexpected type

# --- 1. bucket setup (same pattern as the racialised-rate script) ---
CUTOFF_YEAR = 2018  # racialised subset is much smaller (N=7,768), so a later cutoff than the 2016 used for the overall rate

racialised_ts["review_year"] = racialised_ts["review_timestamp_est"].dt.year
racialised_ts["review_month"] = racialised_ts["review_timestamp_est"].dt.month

def make_bucket(row):
    if row["review_year"] < CUTOFF_YEAR:
        return (f"pre-{CUTOFF_YEAR}", 0)
    return (str(row["review_year"]), row["review_month"])

racialised_ts["bucket_key"] = racialised_ts.apply(make_bucket, axis=1)

def build_bucket_base(df, cutoff_year, thin_threshold=30):
    n_by_bucket = df.groupby("bucket_key").size()
    idx = n_by_bucket.reset_index(name="n_reviews")
    idx["sort_year"] = idx["bucket_key"].apply(lambda k: -1 if k[0] == f"pre-{cutoff_year}" else int(k[0]))
    idx["sort_month"] = idx["bucket_key"].apply(lambda k: k[1])
    idx = idx.sort_values(["sort_year", "sort_month"]).drop(columns=["sort_year", "sort_month"])
    idx["label"] = idx["bucket_key"].apply(lambda k: f"pre-{cutoff_year}" if k[0] == f"pre-{cutoff_year}" else f"{k[0]}-{int(k[1]):02d}")
    idx = idx.set_index("bucket_key")

    # trim trailing thin bucket(s) — usually the current in-progress month
    n_trailing_thin = 0
    while idx["n_reviews"].iloc[-(n_trailing_thin + 1)] < thin_threshold:
        n_trailing_thin += 1
    if n_trailing_thin > 0:
        print(f"dropping {n_trailing_thin} trailing thin bucket(s):")
        print(idx.iloc[-n_trailing_thin:][["label", "n_reviews"]])
        idx = idx.iloc[:-n_trailing_thin].copy()

    idx["x_pos"] = range(len(idx))
    return idx

bucket_base = build_bucket_base(racialised_ts, CUTOFF_YEAR)
print(f"final buckets: {len(bucket_base)}")
print(bucket_base[["label", "n_reviews"]])

# - prevalence + Wilson CI per label, multi-label aware ---
def prevalence_with_ci_by_bucket(df, col, labels, base_index):
    rows = []
    for bucket_key, g in df.groupby("bucket_key"):
        n = len(g)
        row = {"bucket_key": bucket_key, "n_reviews": n}
        for lab in labels:
            k = g[col].apply(lambda cell: label_match(cell, lab)).sum()
            rate, ci_lo, ci_hi = wilson(k, n)
            row[f"{lab}_rate"] = rate
            row[f"{lab}_ci_lo"] = ci_lo
            row[f"{lab}_ci_hi"] = ci_hi
        rows.append(row)
    out = pd.DataFrame(rows).set_index("bucket_key").drop(columns=["n_reviews"])
    out = out.reindex(base_index.index)
    return base_index.join(out)



CONNOTATION_LABELS = ["Negative", "Neutral", "Positive"]
CONTEXT_LABELS = ["generalised", "incident"]
APPROACH_LABELS = ["Explicit", "Implicit", "Coded"]

# --- recompute each dimension with CIs ---
connotation_monthly = prevalence_with_ci_by_bucket(racialised_ts, "connotation", CONNOTATION_LABELS, bucket_base)
context_monthly = prevalence_with_ci_by_bucket(racialised_ts, "context", CONTEXT_LABELS, bucket_base)
approach_monthly = prevalence_with_ci_by_bucket(racialised_ts, "approach", APPROACH_LABELS, bucket_base)



# --- recompute each dimension with CIs ---
connotation_monthly = prevalence_with_ci_by_bucket(racialised_ts, "connotation", CONNOTATION_LABELS, bucket_base)
context_monthly = prevalence_with_ci_by_bucket(racialised_ts, "context", CONTEXT_LABELS, bucket_base)
approach_monthly = prevalence_with_ci_by_bucket(racialised_ts, "approach", APPROACH_LABELS, bucket_base)

# --- generic plot function, now with error bars ---
def plot_dimension_with_ci(combined, labels, title, colors=None):
    if colors is None:
        palette = plt.cm.tab10.colors
        colors = {lab: palette[i % len(palette)] for i, lab in enumerate(labels)}

    fig, (ax_rate, ax_n) = plt.subplots(
        2, 1, figsize=(15, 7.5), gridspec_kw={"height_ratios": [2.5, 1]}
    )

    for lab in labels:
        rate = combined[f"{lab}_rate"]
        ci_lo = combined[f"{lab}_ci_lo"]
        ci_hi = combined[f"{lab}_ci_hi"]
        ax_rate.plot(combined["x_pos"], rate, marker="o", markersize=3, linewidth=1,
                     label=lab, color=colors[lab])
        ax_rate.fill_between(combined["x_pos"], ci_lo, ci_hi, color=colors[lab], alpha=0.15, linewidth=0)

    ax_rate.set_ylabel("share of racialised reviews")
    ax_rate.set_title(title)
    ax_rate.set_ylim(0, 1)
    ax_rate.legend(fontsize=8)
    ax_rate.grid(True, alpha=0.3)

    ax_n.bar(combined["x_pos"], combined["n_reviews"], color="#4C72B0", width=0.8)
    ax_n.set_ylabel("n racialised reviews")
    ax_n.set_xlabel(f"year-month (pre-{CUTOFF_YEAR} bucketed)")

    step = max(1, len(combined) // 20)
    tick_pos = combined["x_pos"].iloc[::step].tolist()
    tick_labels = combined["label"].iloc[::step].tolist()
    ax_n.set_xticks(tick_pos)
    ax_n.set_xticklabels(tick_labels, rotation=45, ha="right")
    ax_n.tick_params(axis="x", which="major", bottom=True, top=False, direction="out", length=8, width=1.2, color="black")

    ymin, ymax = ax_n.get_ylim()
    tick_len = (ymax - ymin) * 0.04
    for pos in tick_pos:
        ax_n.plot([pos, pos], [ymin - tick_len, ymin], color="black", linewidth=1.2, clip_on=False, zorder=10)
        ax_rate.axvline(pos, color="gray", linestyle="--", linewidth=0.6, alpha=0.4, zorder=0)
        ax_n.axvline(pos, color="gray", linestyle="--", linewidth=0.6, alpha=0.4, zorder=0)

    fig.tight_layout()
    fig.subplots_adjust(bottom=0.2, hspace=0.15)
    plt.savefig(FIGURES_DIR / f"ci_{slugify(title)}.png", dpi=150, bbox_inches="tight")
    plt.show()

# --- 4. run for each dimension ---
plot_dimension_with_ci(connotation_monthly, CONNOTATION_LABELS,
                        f"Connotation prevalence by month (racialised subset, N={len(racialised_ts):,})",
                        colors={"Negative": "#C44E52", "Neutral": "#DD8452", "Positive": "#55A868"})
plot_dimension_with_ci(context_monthly, CONTEXT_LABELS,
                        f"Context prevalence by month (racialised subset, N={len(racialised_ts):,})",
                        colors={"generalised": "#4C72B0", "incident": "#DD8452"})
plot_dimension_with_ci(approach_monthly, APPROACH_LABELS,
                        f"Approach prevalence by month (racialised subset, N={len(racialised_ts):,})",
                        colors={"Explicit": "#55A868", "Implicit": "#DD8452", "Coded": "#8172B2"})

In [ ]:
# --- combined "dominant + incident" plot: Negative, Generalised, Incident, Explicit ---
combined4 = bucket_base.copy()

combined4["Negative_rate"] = connotation_monthly["Negative_rate"]
combined4["Negative_ci_lo"] = connotation_monthly["Negative_ci_lo"]
combined4["Negative_ci_hi"] = connotation_monthly["Negative_ci_hi"]

combined4["generalised_rate"] = context_monthly["generalised_rate"]
combined4["generalised_ci_lo"] = context_monthly["generalised_ci_lo"]
combined4["generalised_ci_hi"] = context_monthly["generalised_ci_hi"]

combined4["incident_rate"] = context_monthly["incident_rate"]
combined4["incident_ci_lo"] = context_monthly["incident_ci_lo"]
combined4["incident_ci_hi"] = context_monthly["incident_ci_hi"]

combined4["Explicit_rate"] = approach_monthly["Explicit_rate"]
combined4["Explicit_ci_lo"] = approach_monthly["Explicit_ci_lo"]
combined4["Explicit_ci_hi"] = approach_monthly["Explicit_ci_hi"]

series_to_plot = ["Negative", "generalised", "incident", "Explicit"]
colors4 = {"Negative": "#C44E52", "generalised": "#4C72B0", "incident": "#DD8452", "Explicit": "#55A868"}

plot_dimension_with_ci(
    combined4.rename(columns={f"{s}_rate": f"{s}_rate" for s in series_to_plot}),  # already named correctly
    series_to_plot,
    f"Negative, Generalised, Explicit — prevalence by month (racialised subset, N={len(racialised_ts):,})",
    colors=colors4
)

In [ ]:
series_to_plot_3 = ["Negative", "generalised", "Explicit"]
colors3 = {"Negative": "#C44E52", "generalised": "#4C72B0", "Explicit": "#55A868"}

# find the actual min across ci_lo values to set a sensible floor
ci_lo_cols = [f"{s}_ci_lo" for s in series_to_plot_3]
y_floor = combined4[ci_lo_cols].min().min()
y_floor = max(0, y_floor - 0.05)  # small padding, don't go below 0

fig, (ax_rate, ax_n) = plt.subplots(
    2, 1, figsize=(15, 7.5), gridspec_kw={"height_ratios": [2.5, 1]}
)

for lab in series_to_plot_3:
    rate = combined4[f"{lab}_rate"]
    ci_lo = combined4[f"{lab}_ci_lo"]
    ci_hi = combined4[f"{lab}_ci_hi"]
    ax_rate.plot(combined4["x_pos"], rate, marker="o", markersize=3, linewidth=1, label=lab, color=colors3[lab])
    ax_rate.fill_between(combined4["x_pos"], ci_lo, ci_hi, color=colors3[lab], alpha=0.15, linewidth=0)

ax_rate.set_ylabel("share of racialised reviews")
ax_rate.set_title(f"Negative, Generalised, Explicit — prevalence by month (racialised subset, N={len(racialised_ts):,})")
ax_rate.set_ylim(y_floor, 1.02)   # <-- zoomed instead of fixed (0,1)
ax_rate.legend(fontsize=8)
ax_rate.grid(True, alpha=0.3)

ax_n.bar(combined4["x_pos"], combined4["n_reviews"], color="#4C72B0", width=0.8)
ax_n.set_ylabel("n racialised reviews")
ax_n.set_xlabel(f"year-month (pre-{CUTOFF_YEAR} bucketed)")

step = max(1, len(combined4) // 20)
tick_pos = combined4["x_pos"].iloc[::step].tolist()
tick_labels = combined4["label"].iloc[::step].tolist()
ax_n.set_xticks(tick_pos)
ax_n.set_xticklabels(tick_labels, rotation=45, ha="right")
ax_n.tick_params(axis="x", which="major", bottom=True, top=False, direction="out", length=8, width=1.2, color="black")

ymin, ymax = ax_n.get_ylim()
tick_len = (ymax - ymin) * 0.04
for pos in tick_pos:
    ax_n.plot([pos, pos], [ymin - tick_len, ymin], color="black", linewidth=1.2, clip_on=False, zorder=10)
    ax_rate.axvline(pos, color="gray", linestyle="--", linewidth=0.6, alpha=0.4, zorder=0)
    ax_n.axvline(pos, color="gray", linestyle="--", linewidth=0.6, alpha=0.4, zorder=0)

fig.tight_layout()
fig.subplots_adjust(bottom=0.2, hspace=0.15)
plt.savefig(FIGURES_DIR / "top3_dominant_labels_monthly_zoomed_raw_ci.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# identify the buckets in that window
window_mask = combined4["label"].between("2020-09", "2021-07")  # string comparison works for YYYY-MM format
window = combined4.loc[window_mask, ["label", "n_reviews", "Negative_rate", "Negative_ci_lo", "Negative_ci_hi",
                                       "generalised_rate", "generalised_ci_lo", "generalised_ci_hi",
                                       "Explicit_rate", "Explicit_ci_lo", "Explicit_ci_hi"]]
print(window.to_string())

In [ ]:
# find which bucket_key corresponds to a specific label, e.g. the extreme spike
target_label = "2021-03"  # adjust based on what `window` printout shows as most extreme
target_bucket = combined4.loc[combined4["label"] == target_label].index[0]

raw_reviews = racialised_ts.loc[racialised_ts["bucket_key"] == target_bucket,
                                  ["review_timestamp_est", "connotation", "context", "approach"]]
print(f"n reviews in {target_label}: {len(raw_reviews)}")
print(display(raw_reviews))

In [ ]:
N_THRESHOLD = 30  # adjust based on what you consider "too thin to trust"

def plot_dimension_with_ci_band_filtered(combined, labels, title, colors=None, n_threshold=N_THRESHOLD):
    combined_filtered = combined[combined["n_reviews"] >= n_threshold].copy()
    combined_filtered["x_pos"] = range(len(combined_filtered))  # renumber x positions after dropping

    dropped = combined[combined["n_reviews"] < n_threshold]
    if len(dropped) > 0:
        print(f"excluding {len(dropped)} bucket(s) with n < {n_threshold}:")
        print(dropped[["label", "n_reviews"]].to_string())

    if colors is None:
        palette = plt.cm.tab10.colors
        colors = {lab: palette[i % len(palette)] for i, lab in enumerate(labels)}

    fig, (ax_rate, ax_n) = plt.subplots(
        2, 1, figsize=(15, 7.5), gridspec_kw={"height_ratios": [2.5, 1]}
    )

    for lab in labels:
        rate = combined_filtered[f"{lab}_rate"]
        ci_lo = combined_filtered[f"{lab}_ci_lo"]
        ci_hi = combined_filtered[f"{lab}_ci_hi"]
        ax_rate.plot(combined_filtered["x_pos"], rate, marker="o", markersize=3, linewidth=1, label=lab, color=colors[lab])
        ax_rate.fill_between(combined_filtered["x_pos"], ci_lo, ci_hi, color=colors[lab], alpha=0.15, linewidth=0)

    ci_lo_cols = [f"{s}_ci_lo" for s in labels]
    y_floor = max(0, combined_filtered[ci_lo_cols].min().min() - 0.05)

    ax_rate.set_ylabel("share of racialised reviews")
    ax_rate.set_title(f"{title} (excluding n < {n_threshold})")
    ax_rate.set_ylim(y_floor, 1.02)
    ax_rate.legend(fontsize=8)
    ax_rate.grid(True, alpha=0.3)

    ax_n.bar(combined_filtered["x_pos"], combined_filtered["n_reviews"], color="#4C72B0", width=0.8)
    ax_n.set_ylabel("n racialised reviews")
    ax_n.set_xlabel(f"year-month (pre-{CUTOFF_YEAR} bucketed, thin months dropped)")

    step = max(1, len(combined_filtered) // 20)
    tick_pos = combined_filtered["x_pos"].iloc[::step].tolist()
    tick_labels = combined_filtered["label"].iloc[::step].tolist()
    ax_n.set_xticks(tick_pos)
    ax_n.set_xticklabels(tick_labels, rotation=45, ha="right")
    ax_n.tick_params(axis="x", which="major", bottom=True, top=False, direction="out", length=8, width=1.2, color="black")

    ymin, ymax = ax_n.get_ylim()
    tick_len = (ymax - ymin) * 0.04
    for pos in tick_pos:
        ax_n.plot([pos, pos], [ymin - tick_len, ymin], color="black", linewidth=1.2, clip_on=False, zorder=10)
        ax_rate.axvline(pos, color="gray", linestyle="--", linewidth=0.6, alpha=0.4, zorder=0)
        ax_n.axvline(pos, color="gray", linestyle="--", linewidth=0.6, alpha=0.4, zorder=0)

    fig.tight_layout()
    fig.subplots_adjust(bottom=0.2, hspace=0.15)
    plt.savefig(FIGURES_DIR / f"cifilt_{slugify(title)}.png", dpi=150, bbox_inches="tight")
    plt.show()

plot_dimension_with_ci_band_filtered(combined4, series_to_plot_3,
    f"Negative, Generalised, Explicit — prevalence by month (racialised subset, N={len(racialised_ts):,})",
    colors=colors3)

**Caveat.** `theme` has no human inter-annotator-agreement check yet (see `inter_annotator_agreement.ipynb`) — treat any pattern in the theme breakdown below as exploratory, not evidentiary.


In [ ]:

racialised_ts["year_bucket"] = np.where(
    racialised_ts["review_year"] < 2018, "pre-2018", racialised_ts["review_year"].astype(int).astype(str)
)
YEAR_ORDER = ["pre-2018", "2018", "2019", "2020", "2021", "2022", "2023", "2024", "2025", "2026"]

print("racialised reviews per year_bucket:")
print(racialised_ts["year_bucket"].value_counts().reindex(YEAR_ORDER))


def prevalence_by_year(df, dim, categories):
    """Share of reviews in each year_bucket carrying each category label (multi-label -> explode)."""
    n_per_year = df.groupby("year_bucket").size()
    exploded = df[[dim, "year_bucket"]].explode(dim)
    out = {}
    for cat in categories:
        cnt = exploded.loc[exploded[dim] == cat].groupby("year_bucket").size()
        out[cat] = (cnt / n_per_year).reindex(YEAR_ORDER)
    return pd.DataFrame(out)

**Note.** `Crowd` (the single largest theme -- see the dimension-content-overview section later) declines steadily from ~60% of racialised reviews in 2018-2019 to ~40% by 2025, while `Staff` and `Clothing` drift upward — a shift in which theme dominates, not just noise around a flat share, though see the cross-dimension check below (§1.4f) before reading too much into it in isolation.

**Note.** `incident`-framed reviews (a specific event, as opposed to a `generalised` complaint) rise steadily from ~27% in 2018-2019 to ~49% by 2025 — a fairly steady trend, not a noisy one.


**Note.** `Negative` connotation share rises from ~85% (2018-2020) to ~92-94% by 2025-2026 — a modest but fairly steady climb, at the expense of both `Neutral` and `Positive`.


**Caveat.** `approach` has near-chance-level human-human inter-annotator agreement (κ≈0-0.09, see `inter_annotator_agreement.ipynb`) — treat any pattern in the approach breakdown below as exploratory, not evidentiary, more so than the other dimensions on this page.


**Note.** `Explicit` share drifts down somewhat (~82% in 2018-2019 to ~75-76% by 2024-2025) with `Implicit` and `Coded` picking up the difference, but the year-to-year movement is noisier than `context`'s or `connotation`'s — consistent with the low-IAA caveat above; treat this one as the least trustworthy of the four series here.



In [ ]:
# assumes racialised_ts has a real datetime column for each review, e.g. review_timestamp_est
racialised_ts["review_year"] = racialised_ts["review_timestamp_est"].dt.year
racialised_ts["review_month"] = racialised_ts["review_timestamp_est"].dt.month

# bucket pre-2018 into one group, same logic as your monthly rate analysis
CUTOFF_YEAR = 2018

def make_bucket(row):
    if row["review_year"] < CUTOFF_YEAR:
        return ("pre-2018", 0)
    return (str(row["review_year"]), row["review_month"])

racialised_ts["bucket_key"] = racialised_ts.apply(make_bucket, axis=1)

# helper: prevalence share per bucket for a given column + set of labels
def prevalence_by_bucket(df, col, labels):
    out = df.groupby("bucket_key").apply(
        lambda g: pd.Series({lab: (g[col] == lab).mean() for lab in labels})
    )
    return out

# n per bucket, for the thinness check
n_by_bucket = racialised_ts.groupby("bucket_key").size()

def build_bucket_index(n_series):
    idx = n_series.reset_index()
    idx["sort_year"] = idx["bucket_key"].apply(lambda k: -1 if k[0] == "pre-2018" else int(k[0]))
    idx["sort_month"] = idx["bucket_key"].apply(lambda k: k[1])
    idx = idx.sort_values(["sort_year", "sort_month"]).drop(columns=["sort_year", "sort_month"])
    idx["label"] = idx["bucket_key"].apply(lambda k: "pre-2018" if k[0] == "pre-2018" else f"{k[0]}-{int(k[1]):02d}")
    idx["x_pos"] = range(len(idx))
    return idx.set_index("bucket_key")

bucket_index = build_bucket_index(n_by_bucket)
bucket_index["n_reviews"] = n_by_bucket

print(f"total buckets: {len(bucket_index)}")
print(f"buckets with n < 30: {(bucket_index['n_reviews'] < 30).sum()}")
print(bucket_index[bucket_index["n_reviews"] < 30][["label", "n_reviews"]])

## Cross-dimension check: do these move together, or independently?

Do `theme`, `context`, `connotation`, and `approach` drift over time as one coherent signal, or independently? We take one representative series per dimension (`theme`'s top category share, `connotation`'s Negative share, `context`'s incident share, `approach`'s Explicit share) over the 8 years with adequate per-year N (2018-2025; `pre-2018` and the partial `2026` are excluded from the correlation itself, though not from the plots above) and compute the Spearman correlation matrix across these yearly series.


In [ ]:
CORR_YEARS = ["2018", "2019", "2020", "2021", "2022", "2023", "2024", "2025"]
top_theme = TOP_THEMES[0]
cross_dim = pd.DataFrame({
    f"theme_{top_theme}_share": theme_prev.loc[CORR_YEARS, top_theme],
    "connotation_negative_share": connotation_prev.loc[CORR_YEARS, "Negative"],
    "context_incident_share": context_prev.loc[CORR_YEARS, "incident"],
    "context_generalised_share": context_prev.loc[CORR_YEARS, "generalised"],
    "approach_explicit_share": approach_prev.loc[CORR_YEARS, "Explicit"],
    "approach_implicit_share": approach_prev.loc[CORR_YEARS, "Implicit"],
    "approach_coded_share": approach_prev.loc[CORR_YEARS, "Coded"],
    
})
print(display(cross_dim))

cross_dim_corr = cross_dim.corr(method="spearman")
print("\nSpearman correlation across the 8 yearly points (2018-2025):")
print(display(cross_dim_corr.round(3)))

fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(cross_dim_corr, annot=True, fmt=".2f", cmap="RdBu_r", vmin=-1, vmax=1, ax=ax,
            cbar_kws={"label": "Spearman rho"})
ax.set_title("Cross-dimension yearly co-movement (2018-2025, n=8)")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "cross_dimension_yearly_comovement_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
# --- 1. compute monthly prevalence for every series needed in the heatmap ---
theme_monthly = prevalence_with_ci_by_bucket(racialised_ts, "theme", TOP_THEMES, bucket_base)

# --- 2. assemble one combined table with just the rate columns we want to correlate ---
monthly_corr_df = bucket_base[["label", "n_reviews"]].copy()
monthly_corr_df["theme_Crowd_share"] = theme_monthly["Crowd_rate"]
monthly_corr_df["connotation_negative_share"] = connotation_monthly["Negative_rate"]
monthly_corr_df["context_incident_share"] = context_monthly["incident_rate"]
monthly_corr_df["context_generalised_share"] = context_monthly["generalised_rate"]
monthly_corr_df["approach_explicit_share"] = approach_monthly["Explicit_rate"]
monthly_corr_df["approach_implicit_share"] = approach_monthly["Implicit_rate"]
monthly_corr_df["approach_coded_share"] = approach_monthly["Coded_rate"]

# --- 3. drop thin months ---
N_THRESHOLD = 30
before = len(monthly_corr_df)
monthly_corr_df_filtered = monthly_corr_df[monthly_corr_df["n_reviews"] >= N_THRESHOLD].copy()
after = len(monthly_corr_df_filtered)
print(f"dropped {before - after} month(s) with n < {N_THRESHOLD}; {after} months remain")

# --- 4. compute Spearman correlation matrix on the filtered monthly shares ---
share_cols = ["theme_Crowd_share", "connotation_negative_share", "context_incident_share",
              "context_generalised_share", "approach_explicit_share", "approach_implicit_share",
              "approach_coded_share"]

corr_monthly = monthly_corr_df_filtered[share_cols].corr(method="spearman")
print(corr_monthly.round(2))

# --- 5. plot heatmap ---
fig, ax = plt.subplots(figsize=(12, 10))
im = ax.imshow(corr_monthly, cmap="RdBu_r", vmin=-1, vmax=1)

ax.set_xticks(range(len(share_cols)))
ax.set_yticks(range(len(share_cols)))
ax.set_xticklabels(share_cols, rotation=90)
ax.set_yticklabels(share_cols)

# explicitly disable gridlines — overrides any style rcParams (same issue as the earlier tick fix)
ax.grid(False)
ax.set_axisbelow(False)

for i in range(len(share_cols)):
    for j in range(len(share_cols)):
        val = corr_monthly.iloc[i, j]
        text_color = "white" if abs(val) > 0.5 else "black"
        ax.text(j, i, f"{val:.2f}", ha="center", va="center", color=text_color)

cbar = fig.colorbar(im, ax=ax)
cbar.set_label("Spearman rho")
ax.set_title(f"Cross-dimension monthly co-movement (n={after} months, n_reviews≥{N_THRESHOLD}/month)")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "cross_dimension_monthly_comovement_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()

## Formal significance tests: is the trend, and the season effect, real?

Everything above (the yearly trend, the indoor/outdoor split, the summer/other split) has been shown via Wilson confidence-interval bands -- convincing, but not the same as an actual hypothesis test with a p-value. We close WHEN with the formal versions of exactly the three questions those subsections raised, run on the overall racialised rate first, then on every individual `theme`/`context`/`connotation`/`approach` label -- which lets us answer "which dimensional labels actually stand out" with numbers instead of eyeballing line charts.

**Method.** For the yearly trend we fit a logistic regression of `racialised` (0/1) on year (centered, continuous) -- this is the standard trend-test approach for a rate observed at several time points, equivalent in spirit to a Cochran-Armitage trend test, and gives a direct p-value for "is the log-odds moving with year." We fit it twice: once on the full 2016-2025 window, and once restricted to 2021-2025 specifically, since that's the window you flagged as visually steeper. For indoor/outdoor and summer/other we use a straightforward chi-square test of independence (racialised x season), pooling across all years -- the same test already used throughout this notebook for categorical associations.


In [ ]:
import statsmodels.formula.api as smf
from statsmodels.stats.proportion import proportions_ztest

corpus["review_year_est"] = corpus["review_timestamp_est"].dt.year
ts_all = corpus.loc[social_mask & corpus["review_timestamp_est"].notna()].copy()
ts_all["review_year_est"] = ts_all["review_timestamp_est"].dt.year
ts_all["is_racialised"] = (ts_all["racialised"] == "yes").astype(int)

# --- trend test 1: full 2016-2025 window ---
trend_df = ts_all[ts_all["review_year_est"].between(2016, 2025)].copy()
trend_df["year_c"] = trend_df["review_year_est"] - trend_df["review_year_est"].mean()
model_full = smf.logit("is_racialised ~ year_c", data=trend_df).fit(disp=0)
coef_full, p_full = model_full.params["year_c"], model_full.pvalues["year_c"]
print(f"2016-2025 trend: log-odds/year = {coef_full:.4f}, p = {p_full:.2e}")

# --- trend test 2: post-2021 window specifically ---
post_df = ts_all[ts_all["review_year_est"].between(2021, 2025)].copy()
post_df["year_c"] = post_df["review_year_est"] - post_df["review_year_est"].mean()
model_post = smf.logit("is_racialised ~ year_c", data=post_df).fit(disp=0)
coef_post, p_post = model_post.params["year_c"], model_post.pvalues["year_c"]
print(f"2021-2025 trend: log-odds/year = {coef_post:.4f}, p = {p_post:.2e}  "
      f"({coef_post / coef_full:.1f}x steeper than the full-period slope)")

# --- simple, easily-quotable endpoint test: 2021 vs 2025 ---
n2021, k2021 = (ts_all["review_year_est"] == 2021).sum(), ((ts_all["review_year_est"] == 2021) & (ts_all["is_racialised"] == 1)).sum()
n2025, k2025 = (ts_all["review_year_est"] == 2025).sum(), ((ts_all["review_year_est"] == 2025) & (ts_all["is_racialised"] == 1)).sum()
z, p_z = proportions_ztest([k2021, k2025], [n2021, n2025])
print(f"\n2021 rate: {k2021:,}/{n2021:,} = {k2021/n2021:.3%}   2025 rate: {k2025:,}/{n2025:,} = {k2025/n2025:.3%}")
print(f"two-proportion z-test, 2021 vs. 2025: z={z:.2f}, p={p_z:.2e}")

# --- indoor/outdoor and summer/other, pooled across all years (chi-square) ---
ts_social_actual["is_racialised"] = (ts_social_actual["racialised"] == "yes").astype(int)
ts_social_actual["summer_vs_other"] = np.where(ts_social_actual["meteorological_season"] == "Summer", "Summer", "Other")

for label, col in [("indoor/outdoor", "pool_season"), ("summer/other", "summer_vs_other")]:
    ct = pd.crosstab(ts_social_actual[col], ts_social_actual["is_racialised"])
    chi2, p, dof, _ = stats.chi2_contingency(ct)
    n = ct.values.sum()
    v = np.sqrt(chi2 / (n * (min(ct.shape) - 1)))
    rates = (ct[1] / ct.sum(axis=1) * 100).round(2)
    print(f"\n{label}: chi2={chi2:.1f}, dof={dof}, p={p:.2e}, Cramer's V={v:.4f}")
    print(rates)


**Takeaway -- yes, all three are real, formally significant effects.**

**Yearly trend.** The 2016-2025 trend is a highly significant increase (logistic-regression year coefficient = 0.0824 log-odds/year, p=6.8e-65). Restricting to 2021-2025 specifically, the slope is **2.7x steeper** (0.2216 log-odds/year, p=5.4e-92) -- confirming this isn't just "the overall trend, zoomed in," it's a genuine acceleration. The simplest, most quotable version: **0.77% (2021) -> 2.30% (2025) is a highly significant increase** (two-proportion z-test, z=-18.16, p=1.2e-73).

**Indoor vs. outdoor.** Significant: chi2=313.8, p=3.3e-70. Outdoor season 1.80% vs. indoor 1.19%.

**Summer vs. other.** Significant: chi2=358.7, p=5.5e-80. Summer 1.92% vs. other seasons 1.25%.

**One honest caveat that applies to all three**: Cramér's V for both season tests is tiny (0.025-0.027) despite the very small p-values -- with over a million reviews, even a trivial association clears significance easily. The p-value confirms these effects are *real* (not noise), but the *size* of the seasonal effect is modest; the year-on-year trend, by contrast, is both significant and substantively large (rate roughly tripled).

### A sharper version: is each season *trending upward on its own*, and do the two trends actually differ?

The indoor/outdoor and summer/other tests above are a single pooled comparison collapsing all years together -- they answer "is the average rate different between seasons," not "does each season have its own year-on-year trend, and is one trend genuinely steeper than the other." We test both directly: a trend model fit *separately* within each season, and a single combined model with a `year x season` interaction term, whose significance is the formal test for "the trends actually differ, not just both happen to be positive."


In [ ]:
ts_social_actual["review_year_est"] = ts_social_actual["review_timestamp_est"].dt.year
trend_window = ts_social_actual[ts_social_actual["review_year_est"].between(2016, 2025)].copy()
trend_window["year_c"] = trend_window["review_year_est"] - trend_window["review_year_est"].mean()

print("--- indoor/outdoor: trend within each season, separately ---")
for season in ["indoor", "outdoor"]:
    sub = trend_window[trend_window["pool_season"] == season]
    m = smf.logit("is_racialised ~ year_c", data=sub).fit(disp=0)
    print(f"{season:8s}: n={len(sub):,}, year coef={m.params['year_c']:.4f}, p={m.pvalues['year_c']:.2e}")

m_int = smf.logit("is_racialised ~ year_c * C(pool_season)", data=trend_window).fit(disp=0)
inter_coef, inter_p = m_int.params["year_c:C(pool_season)[T.outdoor]"], m_int.pvalues["year_c:C(pool_season)[T.outdoor]"]
print(f"interaction (does outdoor's trend differ from indoor's?): coef={inter_coef:.4f}, p={inter_p:.2e}")

print("\n--- summer/other: trend within each group, separately ---")
for grp in ["Summer", "Other"]:
    sub = trend_window[trend_window["summer_vs_other"] == grp]
    m = smf.logit("is_racialised ~ year_c", data=sub).fit(disp=0)
    print(f"{grp:8s}: n={len(sub):,}, year coef={m.params['year_c']:.4f}, p={m.pvalues['year_c']:.2e}")

m_int2 = smf.logit("is_racialised ~ year_c * C(summer_vs_other)", data=trend_window).fit(disp=0)
inter_coef2, inter_p2 = m_int2.params["year_c:C(summer_vs_other)[T.Summer]"], m_int2.pvalues["year_c:C(summer_vs_other)[T.Summer]"]
print(f"interaction (does Summer's trend differ from Other's?): coef={inter_coef2:.4f}, p={inter_p2:.2e}")


**Takeaway -- yes to both: each season is independently trending upward, and the trends are significantly different from each other, not just two positive slopes that happen to differ.**

**Indoor/outdoor.** `indoor` alone has a significant upward trend (coef=0.0368/year, p=4.2e-06); `outdoor` alone does too, and **~2.9x steeper** (coef=0.1049/year, p=1.3e-64). The `year x season` interaction term -- the actual formal test for "is outdoor's trend steeper than indoor's, or could that gap be chance" -- is itself significant (coef=0.0681, p=1.5e-11). So this isn't "both are rising, coincidentally at different rates": outdoor's faster rise is a real, distinct effect on top of the general year-on-year increase.

**Summer/other.** Same shape: `Summer` alone (coef=0.1134/year, p=4.8e-59) is **~2.4x steeper** than `Other` months alone (coef=0.0476/year, p=3.1e-12), and the interaction term confirms this difference is real (coef=0.0658, p=1.7e-11).

**How to read this alongside the pooled test above**: the earlier pooled chi-square (outdoor 1.80% vs. indoor 1.19%, averaged over all years) tells you the seasons differ *on average*; this section tells you *why* in more depth -- both seasons are riding the same general upward trend, but the outdoor/summer side of it is accelerating faster, which is itself part of what's driving the widening gap between them, not a separate phenomenon.

### A third angle: is the indoor/outdoor (and summer/other) gap significant *within each individual year*, and is that itself consistent over time?

The interaction test above is a single number summarizing "does the gap grow over the whole 2016-2025 window" -- it can be significant even if the gap actually appeared abruptly at some point rather than growing smoothly throughout. The more granular version: test indoor vs. outdoor (and Summer vs. Other) **separately within each year**, so we can see not just *whether* there's an overall interaction but *when* the within-year gap actually became real.


In [ ]:
def per_year_season_test(df, group_col, groups, year_col="review_year_est"):
    rows = []
    for yr, g in df.groupby(year_col):
        ct = pd.crosstab(g[group_col], g["is_racialised"])
        if ct.shape != (2, 2):
            continue
        chi2, p, dof, _ = stats.chi2_contingency(ct)
        n = ct.values.sum()
        v = np.sqrt(chi2 / (n * (min(ct.shape) - 1)))
        rate_a = ct.loc[groups[0], 1] / ct.loc[groups[0]].sum() * 100
        rate_b = ct.loc[groups[1], 1] / ct.loc[groups[1]].sum() * 100
        rows.append({
            "year": yr, f"{groups[0]}_rate": rate_a, f"{groups[1]}_rate": rate_b,
            f"n_{groups[0]}": ct.loc[groups[0]].sum(), f"n_{groups[1]}": ct.loc[groups[1]].sum(),
            "chi2": chi2, "p": p, "V": v,
        })
    return pd.DataFrame(rows)

io_yearly = per_year_season_test(trend_window, "pool_season", ["indoor", "outdoor"])
print("Indoor vs. outdoor, tested separately within each year:")
print(io_yearly.round(4).to_string(index=False))
print(f"\nyears with a significant (p<0.05) within-year indoor/outdoor gap: {io_yearly.loc[io_yearly['p']<0.05,'year'].tolist()}")

so_yearly = per_year_season_test(trend_window, "summer_vs_other", ["Summer", "Other"])
print("\n\nSummer vs. Other, tested separately within each year:")
print(so_yearly.round(4).to_string(index=False))
print(f"\nyears with a significant (p<0.05) within-year Summer/Other gap: {so_yearly.loc[so_yearly['p']<0.05,'year'].tolist()}")

fig1, ax1 = plt.subplots(figsize=(12, 12))
ax1.plot(io_yearly["year"], io_yearly["outdoor_rate"], marker="o", color="#C44E52", label="outdoor", linewidth=2)
ax1.plot(io_yearly["year"], io_yearly["indoor_rate"], marker="o", color="#4C72B0", label="indoor", linewidth=2)
sig_years_io = io_yearly.loc[io_yearly["p"] < 0.05, "year"]
for yr in sig_years_io:
    ax1.axvspan(yr - 0.4, yr + 0.4, color="gray", alpha=0.1)
ax1.set_xlabel("year", fontsize=30, fontweight="bold")
ax1.set_ylabel("racialised rate (%)", fontsize=30, fontweight="bold")
ax1.tick_params(axis="both", labelsize=30, width=2, length=8, direction="out", top=False, right=False)
leg1 = ax1.legend(fontsize=30, edgecolor="black", facecolor="white")
leg1.get_frame().set_linewidth(2)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "indoor_outdoor_rate_by_year.png", dpi=150, bbox_inches="tight")
plt.show()

fig2, ax2 = plt.subplots(figsize=(12, 12))
ax2.plot(so_yearly["year"], so_yearly["Summer_rate"], marker="o", color="#C44E52", label="Summer", linewidth=2)
ax2.plot(so_yearly["year"], so_yearly["Other_rate"], marker="o", color="#4C72B0", label="Other months", linewidth=2)
sig_years_so = so_yearly.loc[so_yearly["p"] < 0.05, "year"]
for yr in sig_years_so:
    ax2.axvspan(yr - 0.4, yr + 0.4, color="gray", alpha=0.1)
ax2.set_xlabel("year", fontsize=30, fontweight="bold")
ax2.set_ylabel("racialised rate (%)", fontsize=30, fontweight="bold")
ax2.tick_params(axis="both", labelsize=30, width=2, length=8, direction="out", top=False, right=False)
leg2 = ax2.legend(fontsize=30, edgecolor="black", facecolor="white")
leg2.get_frame().set_linewidth(2)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "summer_other_rate_by_year.png", dpi=150, bbox_inches="tight")
plt.show()

**Takeaway -- the gap is not a stable, longstanding feature. It is recent, and it has been growing every year since 2022.**

Indoor/outdoor and summer/other show *almost* the same pattern of which years are significant, with one small exception worth naming honestly rather than glossing over. Indoor/outdoor: `[2018, 2019, 2022, 2023, 2024, 2025]` significant, with **2016, 2017, 2020, 2021 all non-significant** (p=0.21-0.94) -- a clean three-phase shape. Summer/other: the same six years significant, **plus 2020** (p=0.040, gap=0.26pp) -- but 2020 is the weakest result in the entire table by both measures (Cramér's V=0.0113, the second-smallest of any year, and a gap noticeably smaller than either the adjoining 2018-2019 opening (0.53-0.59pp) or the 2022+ reopening (0.54pp+)). This reads as a borderline case that happens to clear p<0.05 with a very large sample, not a real break in the three-phase story:

1. **2016-2017**: no detectable seasonal gap at all (indoor/outdoor rates within 0.02-0.4 percentage points of each other, p>0.24).
2. **2018-2019**: a real, significant gap opens up (~0.4-0.6pp, p<0.001) -- then **substantially closes** in 2020-2021: indoor/outdoor is fully non-significant both years (p=0.21, 0.37); summer/other is non-significant in 2021 (p=0.36) and only marginally, weakly significant in 2020 (p=0.040, smallest effect size of any significant year in the table). 2020-2021 is also when overall review volume patterns shifted (visible in the `n_indoor`/`n_outdoor` and `n_Summer`/`n_Other` columns) -- consistent with, though not proof of, a COVID-era disruption to normal pool-going and reviewing patterns temporarily weakening the seasonal signal.
3. **2022 onward**: the gap re-emerges and **widens every single year** -- indoor/outdoor difference grows from 0.61pp (2022) to 1.14pp (2025); Cramér's V climbs steadily (0.0262 -> 0.0405 -> 0.0380 -> 0.0384 for indoor/outdoor; 0.0235 -> 0.0375 -> 0.0400 -> 0.0467 for Summer/other). By 2025, `Summer` sits at 3.05% vs. `Other months`' 1.64% -- essentially double, and both the largest gap and the largest individual rates seen anywhere in this yearly breakdown.

**Put together with the interaction test above**: the significant `year x season` interaction isn't describing a gap that's been slowly, steadily widening since 2016 -- it's really describing this specific post-2021 acceleration. The seasonal effect is a *recent and intensifying* phenomenon, not a stable structural feature of the data, which is an important nuance for how "outdoor/summer pools attract more racialised discourse" should be framed: true now, and increasingly so, but not true throughout the whole observation window. The 2020 summer/other borderline case doesn't change this reading -- it's the weakest result in the table, not a second "gap present" phase.

### The same three tests, per dimensional label -- which ones stand out?

We repeat all three tests -- post-2021 trend, indoor/outdoor, summer/other -- for every `theme` (restricted to the top-5, per the established convention above), `context`, `connotation`, and `approach` label, on the actually-parsed-timestamp racialised subset (N=5,693, same restriction as the month/season analysis above, for the same estimation-artifact reason). This directly answers "are there dimensional labels that stand out": rather than reading 12 separate line charts, we rank every label by effect size across all three tests at once.


In [ ]:
rdf = corpus.loc[corpus["racialised"] == "yes"].copy()
rdf = rdf[rdf["review_timestamp"].notna()].copy()  # actual, non-estimated timestamps only
rdf["review_year"] = rdf["review_timestamp_est"].dt.year
rdf["review_month"] = rdf["review_timestamp_est"].dt.month
rdf["meteorological_season"] = rdf["review_month"].map(SEASON_MAP)
rdf["pool_season"] = np.where(rdf["review_month"].between(5, 9), "outdoor", "indoor")
rdf["summer_vs_other"] = np.where(rdf["meteorological_season"] == "Summer", "Summer", "Other")
print(f"racialised, actual-timestamp subset for these tests: {len(rdf):,}")

def label_match(cell, label):
    return label in cell if isinstance(cell, (list, tuple, set, np.ndarray)) else cell == label

DIM_LABELS = {
    "theme": TOP_THEMES,  # top-5, as established earlier in this section
    "context": ["generalised", "incident"],
    "connotation": ["Negative", "Neutral", "Positive"],
    "approach": ["Explicit", "Implicit", "Coded"],
}

post_df = rdf[rdf["review_year"].between(2021, 2025)].copy()
post_df["year_c"] = post_df["review_year"] - post_df["review_year"].mean()

records = []
for dim, labels in DIM_LABELS.items():
    for lab in labels:
        rdf[f"_has_{lab}"] = rdf[dim].apply(lambda c: label_match(c, lab)).astype(int)
        post_df[f"_has_{lab}"] = post_df[dim].apply(lambda c: label_match(c, lab)).astype(int)

        m = smf.logit(f"_has_{lab} ~ year_c", data=post_df).fit(disp=0)
        trend_coef, trend_p = m.params["year_c"], m.pvalues["year_c"]

        ct1 = pd.crosstab(rdf["pool_season"], rdf[f"_has_{lab}"])
        chi1, p1, _, _ = stats.chi2_contingency(ct1)
        v1 = np.sqrt(chi1 / (ct1.values.sum() * (min(ct1.shape) - 1)))

        ct2 = pd.crosstab(rdf["summer_vs_other"], rdf[f"_has_{lab}"])
        chi2_, p2, _, _ = stats.chi2_contingency(ct2)
        v2 = np.sqrt(chi2_ / (ct2.values.sum() * (min(ct2.shape) - 1)))

        records.append({
            "dim": dim, "label": lab,
            "trend_coef_2021_25": trend_coef, "trend_p": trend_p,
            "indoor_pct": ct1.loc["indoor", 1] / ct1.loc["indoor"].sum() * 100,
            "outdoor_pct": ct1.loc["outdoor", 1] / ct1.loc["outdoor"].sum() * 100,
            "io_p": p1, "io_V": v1,
            "summer_pct": ct2.loc["Summer", 1] / ct2.loc["Summer"].sum() * 100,
            "other_pct": ct2.loc["Other", 1] / ct2.loc["Other"].sum() * 100,
            "so_p": p2, "so_V": v2,
        })

sig_table = pd.DataFrame(records)
pd.set_option("display.width", 160)
print(sig_table.round(4).to_string(index=False))

print("\nsignificant (p<0.05) post-2021 trend, sorted by |coefficient| (steepest first):")
print(sig_table[sig_table["trend_p"] < 0.05].sort_values("trend_coef_2021_25", key=abs, ascending=False)
      [["dim", "label", "trend_coef_2021_25", "trend_p"]].to_string(index=False))


**Takeaway -- `Clothing` is still the clear standout, but the significant-label set changed underneath it.** Ranking the significant post-2021 trends by steepness: **`Clothing` (theme)** still has one of the largest coefficients of all 13 labels tested (+0.212, p=6.5e-12), just behind `connotation`'s `Negative` (+0.216, p=1.8e-09) -- and `Clothing` *still* has the largest effect size on both season tests (summer vs. other: V=0.088, the single largest V in the whole table; indoor vs. outdoor: V=0.070, the largest among theme labels). All three point the same way: **`Clothing`-themed racialised discourse is rising faster than almost any other label since 2021, and is heavily concentrated in summer/outdoor months** (19.4% of racialised reviews in summer carry it, vs. 12.9% in other months -- a ~1.5x gap).

Beyond `Clothing`, the significant post-2021 trends are: `Crowd` declining (-0.066, p=0.004), `context`'s `incident`/`generalised` shift continuing (+0.111/-0.097, both p<0.001), and `connotation` continuing to polarize toward `Negative` (+0.216) at the expense of `Neutral` (-0.177) and `Positive` (-0.129), all p<0.02. `Belonging`, previously a borderline-significant riser, is **no longer significant** in this window (p=0.084) -- worth dropping from the "standout labels" list rather than repeating the old finding uncritically.

**The one real change from before: all three `approach` labels are now significant** (`Coded` +0.103 p=0.007, `Implicit` +0.071 p=0.018, `Explicit` -0.070 p=0.009) -- previously none were. This is exactly the situation the low-inter-annotator-agreement caveat exists for: `approach` has near-chance human-human agreement (κ≈0-0.09), so a newly-significant trend here is more likely a symptom of the corpus growing large enough to make even label noise "significant" than a real behavioural shift -- read this result with *more* caution than the others in this table, not less, precisely because it's now statistically significant.

For the season splits: two labels are now significantly *more* prevalent indoors than outdoors -- `Etiquette` (33.6% vs. 28.6%, p<0.0001), as before, worth noting since it cuts against a naive "crowded summer = more etiquette complaints" assumption, and now also `Explicit` approach (80.1% vs. 76.9%, p=0.001). `Implicit` approach shifts the other way (higher in summer/outdoor, p<0.001) -- consistent with `Explicit` and `Implicit` moving in opposite directions, but again all three `approach` findings should be read through the low-IAA caveat above.

### Draft: paper write-up tables (season significance + dimension trend)

Draft LaTeX for the paper's WHEN section. Updated after the `gmaps_poi` DB refresh (timestamp coverage jumped from 69.5% to 99.5% pre-recovery, 100.00% after) -- every number below is recomputed on the rebuilt `master_corpus_final.parquet`, not the pre-refresh corpus. Two tables built from the significance tests just above: (1) overall racialised rate by season (indoor/outdoor, summer/other), (2) per-dimension-label breakdown across both season splits plus the 2021-2025 trend. Source file: `when_section_significance_tables.tex` in the session scratchpad -- move it into the actual paper repo when ready.

```latex
% Requires: \usepackage{booktabs} \usepackage{array}
% All numbers pulled directly from descriptive_analysis.ipynb's executed output
% (cells a34d63d7 / e0f55cc5 / 423a2b89, "Formal significance tests" section),
% recomputed after the 2026-08 gmaps_poi DB refresh.
% Significance stars: * p<0.05, ** p<0.01, *** p<0.001 (unmarked = not significant).

% ------------------------------------------------------------------
% Table 1 -- overall racialised rate by season (single column)
% ------------------------------------------------------------------
% Chi-square test of independence (racialised x season), pooled across all
% years 2016-2025, actually-parsed-timestamp social subset (n=501,162).
\begin{table}[htbp]
\centering
\footnotesize
\setlength{\tabcolsep}{4pt}
\caption{Racialised rate by season, pooled across all years (chi-square test of independence).}
\label{tab:racialised-rate-season}
\begin{tabular}{@{} l r r r r r @{}}
\toprule
Group & Rate (\%) & $n$ & $\chi^2$ & $p$ & V \\
\midrule
Indoor & 1.19 & 200{,}672 & 313.8 & $<0.001$ & 0.025 \\
Outdoor & 1.80 & 300{,}490 & & & \\
\midrule
Summer & 1.92 & 151{,}106 & 358.7 & $<0.001$ & 0.027 \\
Other & 1.25 & 350{,}056 & & & \\
\bottomrule
\end{tabular}

\vspace{2pt}
\raggedright\footnotesize
Note: both effects are highly significant but small in size (Cram\'er's V $\approx 0.03$) -- with over a million reviews, even a modest association clears significance easily; read the $p$-values as confirming the effects are real, not as a claim about their size.
\end{table}

% ------------------------------------------------------------------
% Table 2 -- per-label breakdown: season splits + post-2021 trend
% ------------------------------------------------------------------
\begin{table}[htbp]
\centering
\footnotesize
\setlength{\tabcolsep}{3pt}
\caption{Content-dimension labels: prevalence by season and 2021--2025 trend (racialised subset, actually-parsed timestamps only, $n=7{,}630$). \texttt{theme} restricted to its top-5 labels by overall prevalence. Superscripts mark significance of that pair's/coefficient's own test.}
\label{tab:dimension-season-trend}
\begin{tabular}{@{} l l r r r r r @{}}
\toprule
Dim. & Label & In.\ \% & Out.\ \% & Sum.\ \% & Oth.\ \% & $\beta_{\text{year}}$ \\
\midrule
Theme & Crowd & 45.6$^{*}$ & 47.0$^{*}$ & 48.9$^{**}$ & 45.3$^{**}$ & $-0.066^{**}$ \\
& Etiquette & 33.6$^{***}$ & 28.6$^{***}$ & 28.5$^{***}$ & 32.4$^{***}$ & $-0.039$ \\
& Staff & 26.6 & 25.3 & 24.5$^{*}$ & 27.0$^{*}$ & \phantom{-}0.023 \\
& Belonging & 19.8 & 19.3 & 19.4 & 19.6 & \phantom{-}0.049 \\
& Clothing & 12.7$^{***}$ & 18.2$^{***}$ & 19.4$^{***}$ & 12.9$^{***}$ & \phantom{-}0.212$^{***}$ \\
\midrule
Context & Generalised & 83.5 & 83.6 & 84.0 & 83.1 & $-0.097^{***}$ \\
& Incident & 40.8 & 41.5 & 41.4 & 41.2 & \phantom{-}0.111$^{***}$ \\
\midrule
Connotation & Negative & 88.1 & 88.5 & 88.8 & 87.9 & \phantom{-}0.216$^{***}$ \\
& Neutral & 9.5 & 8.3 & 8.0 & 9.5$^{*}$ & $-0.177^{***}$ \\
& Positive & 3.8 & 4.7 & 4.7 & 4.0 & $-0.129^{*}$ \\
\midrule
Approach & Explicit & 80.2$^{**}$ & 76.9$^{**}$ & 76.8$^{**}$ & 79.4$^{**}$ & $-0.070^{**}$ \\
& Implicit & 13.4$^{***}$ & 17.3$^{***}$ & 17.4$^{***}$ & 14.4$^{***}$ & \phantom{-}0.071$^{*}$ \\
& Coded & 9.6 & 9.7 & 9.9 & 9.4 & \phantom{-}0.103$^{**}$ \\
\bottomrule
\end{tabular}

\vspace{2pt}
\raggedright\footnotesize
$^{*}p<0.05$, $^{**}p<0.01$, $^{***}p<0.001$ (unmarked = not significant). A star on In.\%/Out.\% marks the indoor/outdoor chi-square for that label; a star on Sum.\%/Oth.\% marks the summer/other chi-square; a star on $\beta_{\text{year}}$ marks that label's own 2021--2025 trend p-value. In./Out.\ and Sum./Oth.\ are the share of racialised reviews carrying that label within each season group (not the overall racialised rate). $\beta_{\text{year}}$ is the year-centered logistic-regression coefficient (log-odds/year), restricted to 2021--2025. \textbf{Changed since the DB refresh}: all three \texttt{approach} labels are now significant on the trend where none were before -- read with extra caution given \texttt{approach}'s near-chance inter-annotator agreement (documented separately); \texttt{Belonging}'s previously-borderline trend (p=0.049) is no longer significant (p=0.084) and should be dropped from any "standout labels" list.
\end{table}

```

## Bridging to WHERE: are all constituencies contributing to the post-2021 rise?

One more question the trend above raises directly: is the 2021-2025 increase a broad, nationwide shift, or is it concentrated in a handful of constituencies that simply didn't have any racialised reviews until recently (which would make it look like a national trend while really being a geographic one)? We check this by classifying every constituency by *when* its racialised reviews fall relative to 2021.


In [ ]:
rdf_all_years = corpus.loc[corpus["racialised"] == "yes"].copy()
rdf_all_years["review_year"] = rdf_all_years["review_timestamp_est"].dt.year
print(f"racialised reviews with a resolvable year (actual + estimated): "
      f"{rdf_all_years['review_year'].notna().sum():,} / {len(rdf_all_years):,}")

contrib = rdf_all_years.groupby("constituency_code")["review_year"].agg(
    n="size", n_pre2021=lambda s: (s < 2021).sum(), n_post2021=lambda s: (s >= 2021).sum(),
    min_year="min", max_year="max",
)
contrib["has_pre"] = contrib["n_pre2021"] > 0
contrib["has_post"] = contrib["n_post2021"] > 0

n_zero = 299 - len(contrib)
n_both = ((contrib["has_pre"]) & (contrib["has_post"])).sum()
n_pre_only = ((contrib["has_pre"]) & (~contrib["has_post"])).sum()
n_post_only = ((~contrib["has_pre"]) & (contrib["has_post"])).sum()

print(f"\nconstituencies with zero racialised reviews ever: {n_zero} / 299")
print(f"constituencies with racialised reviews BOTH before and from 2021 onward: {n_both} / 299 ({n_both/299:.1%})")
print(f"constituencies with racialised reviews ONLY before 2021: {n_pre_only}")
print(f"constituencies with racialised reviews ONLY from 2021 onward: {n_post_only}")

# const_agg (built in WHERE) doesn't exist yet at this point in the notebook -- pull
# constituency/state from `political`, already loaded in Section 0, instead
meta_cols = ["constituency", "state"]
political_meta = political.set_index("constituency_code")[meta_cols]
pre_only = contrib[(contrib["has_pre"]) & (~contrib["has_post"])].join(political_meta)
post_only = contrib[(~contrib["has_pre"]) & (contrib["has_post"])].join(political_meta)

print("\nconstituencies with racialised reviews ONLY before 2021:")
print(pre_only[["constituency", "state", "n", "min_year", "max_year"]].to_string())
print("\nconstituencies with racialised reviews ONLY from 2021 onward:")
print(post_only[["constituency", "state", "n", "min_year", "max_year"]].to_string())


**Takeaway -- yes, essentially every constituency contributes to both periods; the rise is nationwide, not a handful of newly-appearing places.** Of 299 constituencies, **288 (96.3%) have racialised reviews both before and from 2021 onward**. Only **1 constituency (`Berlin-Pankow`) has racialised reviews exclusively before 2021** -- and it's a single review from 2020, already flagged elsewhere in this notebook as the thinnest constituency in the corpus (1 racialised review total). **10 constituencies have racialised reviews only from 2021 onward** -- all thin (2-14 reviews), scattered across 7 different states with no shared pattern, and only 2 of the 10 (`Lübeck`, `Trier`) clear the `reliable_n` bar. **No constituency has zero racialised reviews ever.**

This rules out the concerns you might reasonably have: the 2021-2025 rise is not an artifact of a few constituencies suddenly starting to appear in the data, and it is not concentrated in one region -- it's a broad, essentially universal pattern across German constituencies, which makes the trend test above more trustworthy as a genuine national shift rather than a compositional artifact of *which* constituencies happen to have data in which years.

### Pre-2021 vs. post-2021: is the 2021 dip explained by fewer non-German-origin reviewers, and how has each constituency's rate actually moved?

Two related questions raised by the yearly trend above. **First:** the low point sits at **2021** (0.77%) — is that a real dip in racialising discourse, or simply a dip in how many non-German-origin reviewers were writing at all that year (plausible as a COVID-era effect on foreign-national travel/mobility, distinct from a change in discourse itself)? If non-German-origin reviewers -- the group WHO shows racialises more often -- made up an unusually small share of reviewers specifically in 2021, that alone would produce a lower racialised rate without anything about the discourse having changed. **Second,** independent of 2021 specifically: which constituencies actually moved (in either direction) between the pre-2021 and post-2021 eras, and by how much? This uses each constituency's *overall* racialised rate (not split by reviewer ethnicity -- that would double-count the first question), split at the same 2021 cutoff the trend test above already established as the inflection point.

In [ ]:
# --- composition check: does 2021 have an unusually small non-German-origin share? ---
covid_check = corpus.loc[corpus["social"] == "yes"].copy()
covid_check["review_year"] = covid_check["review_timestamp_est"].dt.year
covid_check = covid_check[covid_check["review_year"].between(2018, 2025)]
covid_check["german_status"] = np.select(
    [covid_check["ethnicity_country_name"] == "Germany", covid_check["ethnicity_country_name"].isna()],
    ["German", "Unknown"], default="Non-German",
)

comp_by_year = covid_check.groupby("review_year")["german_status"].value_counts(normalize=True).unstack()
n_by_year = covid_check.groupby("review_year").size()
print("German-status share of social=yes reviews, by year:")
print(comp_by_year.round(4))
print("\nn social reviews per year:")
print(n_by_year)

rate_by_year = covid_check.groupby("review_year")["racialised"].apply(lambda s: (s == "yes").mean())

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(9, 6), sharex=True)
ax1.plot(rate_by_year.index, rate_by_year.values * 100, marker="o", color="#C44E52")
ax1.set_ylabel("racialised rate (%)")
ax1.set_title("Racialised rate vs. reviewer name-origin composition, by year")
ax1.axvline(2021, color="gray", linestyle="--", linewidth=1)
ax2.plot(comp_by_year.index, comp_by_year["Non-German"] * 100, marker="o", color="#4C72B0", label="Non-German")
ax2.plot(comp_by_year.index, comp_by_year["German"] * 100, marker="o", color="#55A868", label="German")
ax2.axvline(2021, color="gray", linestyle="--", linewidth=1)
ax2.set_ylabel("share of social reviews (%)")
ax2.set_xlabel("year")
ax2.legend(fontsize=9)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "covid_composition_check_by_year.png", dpi=150, bbox_inches="tight")
plt.show()

print("\nNon-German share, 2021 vs. the mean of the two adjacent years (2020, 2022):")
adjacent_mean = comp_by_year.loc[[2020, 2022], "Non-German"].mean()
print(f"2021: {comp_by_year.loc[2021, 'Non-German']:.4f}  |  2020/2022 mean: {adjacent_mean:.4f}")


**Takeaway -- no, the 2021 dip is not explained by fewer non-German-origin reviewers; composition is essentially flat, and it's review *volume*, not composition, that actually collapses in 2020-2021.**

Non-German-origin share of `social=yes` reviews is **50.3% in 2021** -- almost identical to 2020 (50.7%) and only modestly below the 2020/2022 average (51.9%). There's no 2021-specific dip in this share at all: if anything, the whole 2018-2021 window (50.3-51.7%) sits a couple points below 2022-2025 (53.0-55.4%), a gradual drift that starts well before 2021 and continues well after it -- not a composition anomaly isolated to the one low-rate year. What *does* move sharply in 2020-2021 is raw review volume -- **33,191 (2020) and 38,093 (2021) social reviews, vs. 46,451-80,124 in every surrounding year** -- a genuine COVID-era collapse in how many reviews got written at all. But the racialised rate is a *proportion* (racialised / social), so a volume drop alone can't move it unless who's still writing, among the smaller pool that did, shifted too -- and it barely did. **This specific hypothesis doesn't hold**: the 2021 dip looks like less racialising discourse per reviewer during the pandemic (plausibly less crowding, fewer close-contact frictions, less socializing at all), not fewer non-German-origin reviewers around to produce it.

In [ ]:
# --- constituency-level rate change, pre-2021 vs. post-2021 ---
prepost = corpus.loc[corpus["social"] == "yes"].copy()
prepost["review_year"] = prepost["review_timestamp_est"].dt.year
prepost["period"] = np.where(prepost["review_year"] < 2021, "pre_2021", "post_2021")

prepost_agg = (
    prepost.groupby(["constituency_code", "period"])
    .agg(social_reviews=("racialised", "size"), racialised_reviews=("racialised", lambda s: (s == "yes").sum()))
    .unstack("period")
)
prepost_agg.columns = [f"{a}_{b}" for a, b in prepost_agg.columns]
prepost_agg = prepost_agg.fillna(0)

MIN_PERIOD_N = 10  # same reliability bar WHERE settles on below, applied here per-period
reliable_both = (
    (prepost_agg["racialised_reviews_pre_2021"] >= MIN_PERIOD_N)
    & (prepost_agg["racialised_reviews_post_2021"] >= MIN_PERIOD_N)
)
print(f"constituencies with >= {MIN_PERIOD_N} racialised reviews in BOTH periods: "
      f"{reliable_both.sum()} / {len(prepost_agg)}")

rel = prepost_agg.loc[reliable_both].copy()
rel["pre_rate"] = rel["racialised_reviews_pre_2021"] / rel["social_reviews_pre_2021"]
rel["post_rate"] = rel["racialised_reviews_post_2021"] / rel["social_reviews_post_2021"]
rel["rate_diff_pp"] = (rel["post_rate"] - rel["pre_rate"]) * 100

# Cohen's h: standard effect-size measure for a difference between two proportions
# (arcsine-transformed, so it's comparable across constituencies at very different base rates)
rel["cohens_h"] = 2 * np.arcsin(np.sqrt(rel["post_rate"])) - 2 * np.arcsin(np.sqrt(rel["pre_rate"]))

def two_prop_p(row):
    _, p = proportions_ztest(
        [row["racialised_reviews_post_2021"], row["racialised_reviews_pre_2021"]],
        [row["social_reviews_post_2021"], row["social_reviews_pre_2021"]],
    )
    return p

rel["p"] = rel.apply(two_prop_p, axis=1)

name_map = political[["constituency_code", "constituency", "state"]].drop_duplicates()
rel = rel.reset_index().merge(name_map, on="constituency_code", how="left")

print(f"\nproportion tests significant at p<0.05: {(rel['p'] < 0.05).sum()} / {len(rel)}")

cols = ["constituency", "state", "pre_rate", "post_rate", "rate_diff_pp", "cohens_h", "p"]
print("\nLargest INCREASES (post-2021 higher), ranked by |Cohen's h|:")
print(rel.sort_values("cohens_h", ascending=False).head(10)[cols].round(4).to_string(index=False))
print("\nLargest DECREASES (post-2021 lower), ranked by |Cohen's h|:")
print(rel.sort_values("cohens_h").head(10)[cols].round(4).to_string(index=False))

fig, ax = plt.subplots(figsize=(9, 6))
ax.scatter(rel["pre_rate"] * 100, rel["post_rate"] * 100, alpha=0.6, color="#4C72B0")
lim = max(rel["pre_rate"].max(), rel["post_rate"].max()) * 100 * 1.1
ax.plot([0, lim], [0, lim], color="gray", linestyle="--", linewidth=1)
ax.set_xlabel("pre-2021 racialised rate (%)")
ax.set_ylabel("post-2021 racialised rate (%)")
ax.set_title(f"Constituency racialised rate: pre-2021 vs. post-2021\n"
             f"(n={len(rel)} constituencies, >= {MIN_PERIOD_N} racialised reviews/period)")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "constituency_rate_pre_post_2021_scatter.png", dpi=150, bbox_inches="tight")
plt.show()


**Takeaway -- real movement in specific constituencies, but no dominant nationwide reshuffling: only 65/299 constituencies clear >=10 racialised reviews in both periods to test at all, and just 8 of those 65 show a statistically significant shift.**

Ranking by Cohen's h (the arcsine-based effect size, comparable across constituencies at very different base rates): the significant **increases** are `Hannover-Land II` (1.3%->3.1%, h=0.124, p=0.011), `Berlin-Friedrichshain-Kreuzberg – Prenzlauer Berg Ost` (2.8%->4.9%, h=0.113, p=0.035), `Lörrach – Müllheim` (4.2%->6.7%, h=0.108, p=0.001), and two Cologne constituencies, `Köln III` (1.5%->3.0%, h=0.106, p=0.002) and `Köln I` (1.5%->3.0%, h=0.099, p=0.008). The significant **decreases** are `Berlin-Neukölln` (7.6%->3.5%, h=-0.181, p=0.033), `Duisburg II` (10.8%->5.9%, h=-0.177, p<0.0001 -- the single strongest result in the table), and `Berlin-Tempelhof-Schöneberg` (6.7%->3.2%, h=-0.161, p=0.040).

Two things stand out about *which* constituencies moved. **First**, every significant riser started from a low-to-moderate pre-2021 base (1.3-4.2%), while every significant faller started from one of the *highest* pre-2021 rates anywhere in the corpus (6.7-10.8%) -- `Duisburg II`'s 10.8% pre-2021 rate was already the kind of outlier value that mechanically tends to regress toward the mean in a second period, so at least part of this "decrease" pattern is a familiar reversion effect layered on top of whatever real change is also happening, not proof these specific places got safer. **Second**, direction doesn't track state or city: Berlin has two of the three significant fallers *and* one of the five risers, and Nordrhein-Westfalen has both risers (`Köln I`/`Köln III`) and a faller (`Duisburg II`) among its own constituencies -- consistent with this notebook's earlier finding that region/state isn't a strong driver of this discourse. **Bottom line**: the post-2021 national rise established by the formal trend test above is broad, but the constituency-level picture underneath it is not one-directional -- a handful of places are genuinely diverging from the national trend in both directions, and it's specifically the previously-highest-rate constituencies that are pulling back down.

# WHERE

How the racialised-review rate is distributed across German constituencies: physical
coverage (pools/reviews), the sharper reliability question of how many *racialised* reviews
back each rate, prevalence with confidence intervals, choropleths, venue-level concentration
and volume-vs-rate sensitivity checks, and a geometric border-constituency proxy.

## Coverage: pools and text reviews per constituency and state

We build the constituency-level aggregate table now (`const_agg`) and reuse it for the rest of Part 1 — one canonical table rather than recomputing aggregates ad hoc in each section. Critically, we start from the **full list of 299 constituencies** (not just the ones that happen to appear in the corpus) so constituencies with zero pools show up as zero, not as missing rows — a left join here, not an inner join, is the whole point: it's the only way to see the coverage gap at all.

In [ ]:
# canonical 299-constituency backbone comes from the political CSV (already verified 299 rows,
# consistent naming with the shapefile/join in section 1)
backbone = political[["state", "constituency_code", "constituency"]].copy()

agg = (
    corpus.groupby("constituency_code")
    .agg(
        pool_count=("unique_identifier", "nunique"),
        total_text_reviews=("review_id", "count"),
        social_reviews=("social", lambda s: (s == "yes").sum()),
        racialised_reviews=("racialised", lambda s: (s == "yes").sum()),
    )
    .reset_index()
)

const_agg = backbone.merge(agg, on="constituency_code", how="left")
for col in ["pool_count", "total_text_reviews", "social_reviews", "racialised_reviews"]:
    const_agg[col] = const_agg[col].fillna(0).astype(int)

# pools that ever got a social review -- the correct denominator for "share of pools with a
# racialised review", since racialised reviews are a subset of social ones (a pool with zero
# social reviews can never register as racialised).
social_pools = (
    corpus.loc[corpus["social"] == "yes"]
    .groupby("constituency_code")["unique_identifier"].nunique()
    .rename("pools_with_social")
)
const_agg = const_agg.merge(social_pools, on="constituency_code", how="left")
const_agg["pools_with_social"] = const_agg["pools_with_social"].fillna(0).astype(int)

n_zero_pool = (const_agg["pool_count"] == 0).sum()
n_zero_racialised = (const_agg["racialised_reviews"] == 0).sum()
print(f"constituencies with zero pools: {n_zero_pool} / 299")
print(f"constituencies with zero racialised reviews: {n_zero_racialised} / 299")
print()
print("pool_count per constituency:")
print(const_agg["pool_count"].describe())
print()
print("total_text_reviews per constituency:")
print(const_agg["total_text_reviews"].describe())

In [ ]:
# Coverage thresholds: how many constituencies would be usable at different minimum-N bars?
# This is purely descriptive (physical venue/review coverage) — it does NOT define what counts
# as a "reliable" racialised-rate estimate. That's a distinct question, addressed properly in
# the racialised-reviews-coverage subsection below using social_reviews/racialised_reviews
# directly, since pool_count and total_text_reviews
# are only loose proxies for what actually drives the rate's confidence interval.
for thresh in [1, 2, 5, 10, 20]:
    n = (const_agg["pool_count"] >= thresh).sum()
    print(f"constituencies with >= {thresh:>2} pools: {n:>3} / 299 ({n / 299:.1%})")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(const_agg["pool_count"], bins=40, color="#4C72B0")
axes[0].set_title("Pools per constituency")
axes[0].set_xlabel("pool_count")
axes[1].hist(const_agg["total_text_reviews"], bins=40, color="#DD8452")
axes[1].set_title("Text reviews per constituency")
axes[1].set_xlabel("total_text_reviews")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "pools_and_text_reviews_per_constituency_hist.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
# State-level is a natural fallback grain wherever constituency-level is too thin
state_agg = (
    const_agg.groupby("state")
    .agg(
        n_constituencies=("constituency_code", "count"),
        pool_count=("pool_count", "sum"),
        total_text_reviews=("total_text_reviews", "sum"),
        social_reviews=("social_reviews", "sum"),
        racialised_reviews=("racialised_reviews", "sum"),
    )
    .sort_values("pool_count", ascending=False)
)
state_agg


**Takeaway.** Coverage is better than we assumed going in: every one of the 299 constituencies has at least 4 pools and at least 678 text reviews — there are **no true zero-coverage constituencies** anywhere in Germany. Pool coverage in particular is close to a non-issue (every constituency clears even a 4-pool floor), so it is not the binding constraint on what's analytically reliable — that question is about the racialised-review counts specifically, addressed next in the following subsection. At state level, coverage ranges from Bremen (2 constituencies, 25 pools) to Nordrhein-Westfalen (64 constituencies, 809 pools) — city-states and small states will naturally have noisier rate estimates and this should be kept in mind whenever state is used as the unit of comparison.

## Coverage of racialised reviews specifically

The coverage subsection above showed pool/text-review coverage is essentially a non-issue — every constituency clears even a low bar. But that's the wrong thing to gate on: the spatial join already guarantees every constituency has *some* data, and downstream (in the prevalence-with-CIs subsection onward) we're not estimating "does this constituency have pools", we're estimating a **rate** — racialised / social reviews — whose Wilson confidence interval width is driven directly by `social_reviews` (the denominator, i.e. the effective sample size for the proportion) and `racialised_reviews` (the numerator, i.e. how many actual "successes" the estimate rests on). A constituency can have plenty of pools and text reviews and still have a rate estimate resting on only 1-2 racialised reviews, which is a fundamentally different (and worse) reliability problem than "not enough pools." We check that directly here, and **redefine `reliable_n` on that basis** — replacing the pool/text-review-based version from the coverage subsection above, which we now drop since it wasn't measuring the thing that actually matters.

In [ ]:
print("social_reviews per constituency (the rate's denominator):")
print(const_agg["social_reviews"].describe())
print()
print("racialised_reviews per constituency (the rate's numerator):")
print(const_agg["racialised_reviews"].describe())
print()

for thresh in [1, 2, 5, 10, 20]:
    n = (const_agg["racialised_reviews"] >= thresh).sum()
    print(f"constituencies with >= {thresh:>2} racialised reviews: {n:>3} / 299 ({n / 299:.1%})")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(const_agg["social_reviews"], bins=40, color="#55A868")
axes[0].set_title("Social reviews per constituency (rate denominator)")
axes[0].set_xlabel("social_reviews")
axes[1].hist(const_agg["racialised_reviews"], bins=range(0, const_agg["racialised_reviews"].max() + 2), color="#C44E52")
axes[1].set_title("Racialised reviews per constituency (rate numerator)")
axes[1].set_xlabel("racialised_reviews")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "social_and_racialised_reviews_per_constituency_hist.png", dpi=150, bbox_inches="tight")
plt.show()

# reliability now defined on what actually drives the rate's CI width, not on pools/total text reviews.
# MIN_SOCIAL_REVIEWS: a floor on the denominator. MIN_RACIALISED_REVIEWS: revised upward from the
# standard rare-event rule-of-thumb minimum of >=5 "successes" to >=10, for a meaningfully tighter
# Wilson CI on the rate estimate (see the '1.2 addendum' markdown cell below for the exclusion count).
MIN_SOCIAL_REVIEWS = 100
MIN_RACIALISED_REVIEWS = 10

const_agg["reliable_n"] = (
    (const_agg["social_reviews"] >= MIN_SOCIAL_REVIEWS)
    & (const_agg["racialised_reviews"] >= MIN_RACIALISED_REVIEWS)
)
n_reliable = const_agg["reliable_n"].sum()
print(f"\nconstituencies meeting the revised reliability bar (>={MIN_SOCIAL_REVIEWS} social reviews AND "
      f">={MIN_RACIALISED_REVIEWS} racialised reviews): {n_reliable} / 299 ({n_reliable / 299:.1%})")

# how different is this from the old, pool/text-review-based bar?
old_reliable = (const_agg["pool_count"] >= 5) & (const_agg["total_text_reviews"] >= 100)
print(f"old (pool/text-review-based) bar would have flagged: {old_reliable.sum()} / 299 as reliable")
print(f"constituencies where the two bars disagree: {(old_reliable != const_agg['reliable_n']).sum()}")

n_below_bar = (~const_agg["reliable_n"]).sum()
print(f"\nMIN_SOCIAL_REVIEWS never actually binds here: minimum observed social_reviews is "
      f"{const_agg['social_reviews'].min()}, already well above {MIN_SOCIAL_REVIEWS}. The revised bar's "
      f"{n_below_bar} exclusions are driven entirely by too few racialised_reviews, not by denominator size.")

print(f"\nThe {n_below_bar} constituencies below the revised bar (<{MIN_RACIALISED_REVIEWS} racialised reviews):")
print(const_agg.loc[~const_agg["reliable_n"], ["constituency", "state", "social_reviews", "racialised_reviews"]]
      .sort_values("racialised_reviews").to_string(index=False))


**Takeaway.** This confirms the concern directly: `social_reviews` was never actually the binding constraint (minimum is 307, far above any reasonable denominator floor), but `racialised_reviews` is a real bottleneck for a genuinely different set of constituencies. At the original `>=5` racialised-reviews bar, 11/299 (3.7%) constituencies fell short, led by `Berlin-Pankow` with just **1** racialised review out of 382 social reviews. We revise this bar upward to `>=10` (code above) for a meaningfully tighter Wilson CI on the rate estimate — see the reliability-bar revision directly below for the exclusion count and named examples at the revised bar. The old pool/text-review-based bar and the racialised/social-review-based bar (at the revised `>=10` cut) **disagree on 48 constituencies** — it was neither a subset nor a superset of the right answer, it was measuring a different thing that happened to overlap only partially. From here on, `reliable_n` in this notebook reflects this racialised/social-review-based definition at `>=10`, not the pool-count one — anywhere a choropleth or robustness check flags "unreliable," it now means "too few racialised reviews to trust this rate," which is the caveat that actually matters for everything built on top of `racialised_rate`.

## Revising the reliability bar to `>=10` racialised reviews

Raising `MIN_RACIALISED_REVIEWS` from 5 to 10 moves the exclusion count from **11/299 (3.7%)** to **52/299 (17.4%)** — the sensitivity sweep two cells above already previewed this (the `>=10` row). The 41 newly-excluded constituencies (5-9 racialised reviews, previously counted as reliable) include some places that are otherwise well-covered by reviews overall, e.g. `Münster` (5 racialised / 634 social), `Ulm` (7 / 984), `Stuttgart I` (9 / 785), `Würzburg` (9 / 1,102), and `Rosenheim` (9 / 1,595) — a reminder that `social_reviews` volume and `racialised_reviews` volume are genuinely different constraints (as the cell above already notes; it just bites harder at the tighter bar). Everything downstream (choropleth hatching in the Choropleths subsection, the `is_land_border_constituency` flag in the
border-constituency subsection, and the final saved `const_agg.parquet`) now uses this `>=10` definition.


## Prevalence, with confidence intervals

A bare proportion (racialised / social reviews) is misleading on its own for a rare event measured on constituencies with anywhere from ~700 to ~21,000 reviews — the same point estimate means something very different backed by 700 reviews vs. 21,000. We use the **Wilson score interval** rather than the normal approximation: it stays inside [0, 1] and is far better behaved for small samples and proportions near zero, both of which apply here (racialised rate is ~0.5-2%, and several constituencies have modest review counts). We define the rate as **racialised / social**, i.e. among reviews already judged to be about the social experience of the venue (rather than / total reviews, which would dilute the rate with reviews about e.g. opening hours or facilities that were never in scope for racialised-discourse classification).

In [ ]:
# wilson() and its proportion_confint import are hoisted into the shared Part 1 setup
# above (WHEN's temporal cells need it too, and now run before this section).
overall_rate, overall_lo, overall_hi = wilson(
    (corpus["racialised"] == "yes").sum(), (corpus["social"] == "yes").sum()
)
print(f"overall racialised rate (of social=yes reviews): {overall_rate:.3%}  "
      f"[95% CI {overall_lo:.3%}, {overall_hi:.3%}]")

# per-state rates with CI
state_rate = state_agg.apply(
    lambda r: pd.Series(wilson(r["racialised_reviews"], r["social_reviews"]), index=["rate", "ci_lo", "ci_hi"]),
    axis=1,
)
state_rate = state_agg.join(state_rate).sort_values("rate", ascending=False)
state_rate[["n_constituencies", "social_reviews", "racialised_reviews", "rate", "ci_lo", "ci_hi"]]


In [ ]:
# constituency-level rate + CI, added onto the canonical const_agg table
rates = const_agg.apply(
    lambda r: pd.Series(wilson(r["racialised_reviews"], r["social_reviews"]), index=["racialised_rate", "ci_lo", "ci_hi"]),
    axis=1,
)
const_agg = pd.concat([const_agg, rates], axis=1)
const_agg["ci_width"] = const_agg["ci_hi"] - const_agg["ci_lo"]

print("Widest 5 CIs (least reliable rate estimates):")
print(const_agg.sort_values("ci_width", ascending=False)
      [["constituency", "state", "social_reviews", "racialised_reviews", "racialised_rate", "ci_width"]]
      .head(5).to_string(index=False))

print("\nNarrowest 5 CIs (most reliable rate estimates):")
print(const_agg.sort_values("ci_width")
      [["constituency", "state", "social_reviews", "racialised_reviews", "racialised_rate", "ci_width"]]
      .head(5).to_string(index=False))

      
FULL_W = 12.0    # match \textwidth (confirm your class)
FONT = 10

plot_df = const_agg.sort_values("racialised_rate").reset_index(drop=True)
fig, ax = plt.subplots(figsize=(FULL_W, FULL_W * 5 / 14))   # keeps your original 14:5 aspect
colors = np.where(plot_df["reliable_n"], "#4C72B0", "#C44E52")
ax.errorbar(
    range(len(plot_df)), plot_df["racialised_rate"],
    yerr=[plot_df["racialised_rate"] - plot_df["ci_lo"], plot_df["ci_hi"] - plot_df["racialised_rate"]],
    fmt="none", ecolor=colors, alpha=0.5, linewidth=1,
)
ax.scatter(range(len(plot_df)), plot_df["racialised_rate"], c=colors, s=10, zorder=3)
ax.axhline(overall_rate, color="black", linestyle="--", linewidth=1, label=f"overall rate ({overall_rate:.2%})")
ax.set_xlabel("constituencies, sorted by racialised rate", fontsize=FONT, fontweight="bold")
ax.set_ylabel("racialised rate (of social reviews)", fontsize=FONT, fontweight="bold")
ax.tick_params(axis="both", labelsize=FONT-1, width=0.8, length=3)
ax.set_title(f"Racialised rate per constituency with 95% Wilson CI\n(red = below reliability bar: <{MIN_RACIALISED_REVIEWS} racialised reviews)", fontsize=FONT)
leg = ax.legend(fontsize=FONT-1, edgecolor="black", facecolor="white", frameon=True)
leg.get_frame().set_linewidth(0.8)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "racialised_rate_per_constituency_wilson_ci.png", dpi=300, bbox_inches="tight")
plt.show()


**Takeaway.** The overall racialised rate is **1.54% of social reviews [95% CI 1.51%, 1.58%]** — a tight interval at this scale, so the headline prevalence number is solid. State-level rates range roughly 4x, from Mecklenburg-Vorpommern (0.81%) up to Saarland/Berlin/Bremen (~3.0-3.1%), and these top three are small states/city-states, so their CIs are wider — worth naming as "elevated, but on a smaller base" rather than a flat ranking. At constituency level the widest CIs cluster in Ruhr-area NRW constituencies (Essen II, Bochum I, Recklinghausen I) and Berlin-Neukölln — but these aren't just noisy from small N; several (e.g. Bochum I at 526 reviews, 5.7% rate) have *both* reasonable sample size and a genuinely high rate, which is a more interesting signal than sampling noise and worth flagging as a candidate area for closer qualitative follow-up. The narrowest, most reliable estimates sit in the 0.25-0.7% range in mid-size rural/semi-rural constituencies.

## Choropleths

Three separate maps onto the constituency polygons from Section 1: pool coverage
(`pools_with_social`), the social-review rate (`social_reviews / total_text_reviews`), and the
racialised rate itself. For the rate map specifically, constituencies below our `reliable_n` bar
(<10 racialised reviews, revised upward from an earlier <5 cut — see the reliability-bar
subsection above) are hatched rather than removed — removing them would imply "no data" when the
honest message is "data exists but the estimate is noisy."

In [ ]:
# national pool coverage: how many of the 5,528 pools ever get a social/racialised review at all?
n_pools_total = corpus["unique_identifier"].nunique()
n_pools_with_social = corpus.loc[social_mask, "unique_identifier"].nunique()
n_pools_with_racialised = racialised_df["unique_identifier"].nunique()
print(f"pools nationally: {n_pools_total:,}")
print(f"pools with >=1 social review: {n_pools_with_social:,} ({n_pools_with_social / n_pools_total:.1%})")
print(f"pools with >=1 racialised review: {n_pools_with_racialised:,} ({n_pools_with_racialised / n_pools_total:.1%})")

map_df = constituencies[["WKR_NR", "geometry"]].rename(columns={"WKR_NR": "constituency_code"}).merge(
    const_agg, on="constituency_code", how="left"
)


In [ ]:
import textwrap

def format_legend(numbered_names, width=20):
    lines = []
    for num, name in numbered_names:
        prefix = f"{num}: "
        wrapped = textwrap.wrap(name, width=width)
        lines.append(prefix + wrapped[0])
        for cont in wrapped[1:]:
            lines.append(" " * len(prefix) + cont)
    return "\n".join(lines)

fig, ax = plt.subplots(figsize=(7, 8))
map_df.plot(column="pools_with_social", cmap="Blues", legend=True, ax=ax, edgecolor="white", linewidth=0.1)
state_boundaries = map_df.dissolve(by="state")
state_boundaries.boundary.plot(ax=ax, color="black", linewidth=0.9)

top8_pools = map_df.sort_values("pools_with_social", ascending=False).head(8).reset_index(drop=True)
for num, row in top8_pools.iterrows():
    pt = row.geometry.representative_point()
    ax.annotate(str(num + 1), xy=(pt.x, pt.y), fontsize=8, fontweight="bold", color="black",
                ha="center", va="center",
                bbox=dict(boxstyle="circle,pad=0.25", fc="white", ec="black", linewidth=0.6))

legend_text = format_legend([(i + 1, row["constituency"]) for i, row in top8_pools.iterrows()])
ax.text(0.85, 0.32, legend_text, transform=ax.transAxes, fontsize=6.5, ha="left", va="bottom",
        bbox=dict(boxstyle="round,pad=0.4", fc="white", ec="black", linewidth=0.6))

ax.axis("off")
plt.tight_layout()
fig.savefig(FIGURES_DIR / "pools_with_social_choropleth.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
map_df["social_rate"] = map_df["social_reviews"] / map_df["total_text_reviews"]
import textwrap

def format_legend(numbered_names, width=20):
    # right-aligned box (ha="right"): matplotlib right-aligns each line independently,
    # so continuation lines need no manual indent -- ha handles the alignment.
    lines = []
    for num, name in numbered_names:
        wrapped = textwrap.wrap(name, width=width)
        lines.append(f"{num}: {wrapped[0]}")
        lines.extend(wrapped[1:])
    return "\n".join(lines)

fig, ax = plt.subplots(figsize=(7, 8))
map_df.plot(column="social_rate", cmap="Greens", legend=True, ax=ax, edgecolor="white", linewidth=0.1)
state_boundaries = map_df.dissolve(by="state")
state_boundaries.boundary.plot(ax=ax, color="black", linewidth=0.9)

top8_social = map_df.sort_values("social_rate", ascending=False).head(8).reset_index(drop=True)
for num, row in top8_social.iterrows():
    pt = row.geometry.representative_point()
    ax.annotate(str(num + 1), xy=(pt.x, pt.y), fontsize=8, fontweight="bold", color="black",
                ha="center", va="center",
                bbox=dict(boxstyle="circle,pad=0.25", fc="white", ec="black", linewidth=0.6))

legend_text = format_legend([(i + 1, row["constituency"]) for i, row in top8_social.iterrows()])
ax.text(0.98, 0.28, legend_text, transform=ax.transAxes, fontsize=6.5, ha="right", va="bottom",
        bbox=dict(boxstyle="round,pad=0.4", fc="white", ec="black", linewidth=0.6))

ax.axis("off")
plt.tight_layout()
fig.savefig(FIGURES_DIR / "social_rate_choropleth.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# Every constituency actually named in the WHERE-section prose (not just top-N by rate, which
# would miss cases like Berlin-Pankow, Trier, Mansfeld that are discussed for other reasons --
# coverage edge case, post-2021-only, dominant-pool outlier -- so a reader can locate each one
# spatially. Numbered markers + an on-map legend box, bottom corner, long names wrapped.
named_constituencies = [
    "Offenburg", "Duisburg II", "Gelsenkirchen", "Lörrach – Müllheim", "Essen II",
    "Berlin-Pankow", "Lübeck", "Trier", "Mansfeld",
]

missing = set(named_constituencies) - set(map_df["constituency"])
if missing:
    print(f"WARNING: not found in map_df, check spelling: {missing}")

import textwrap

def format_legend(numbered_names, width=20):
    # right-aligned box (ha="right"): matplotlib right-aligns each line independently,
    # so continuation lines need no manual indent -- ha handles the alignment.
    lines = []
    for num, name in numbered_names:
        wrapped = textwrap.wrap(name, width=width)
        lines.append(f"{num}: {wrapped[0]}")
        lines.extend(wrapped[1:])
    return "\n".join(lines)

fig, ax = plt.subplots(figsize=(7, 8))
map_df.plot(column="racialised_rate", cmap="Reds", legend=True, ax=ax, edgecolor="white", linewidth=0.1)
unreliable = map_df.loc[~map_df["reliable_n"]]
unreliable.plot(ax=ax, facecolor="none", edgecolor="black", hatch="///", linewidth=0.6)
state_boundaries = map_df.dissolve(by="state")
state_boundaries.boundary.plot(ax=ax, color="black", linewidth=0.9)

for num, name in enumerate(named_constituencies, start=1):
    row = map_df.loc[map_df["constituency"] == name].iloc[0]
    pt = row.geometry.representative_point()
    ax.annotate(str(num), xy=(pt.x, pt.y), fontsize=8, fontweight="bold", color="black",
                ha="center", va="center",
                bbox=dict(boxstyle="circle,pad=0.25", fc="white", ec="black", linewidth=0.6))

legend_text = format_legend([(i, name) for i, name in enumerate(named_constituencies, start=1)])
ax.text(0.98, 0.28, legend_text, transform=ax.transAxes, fontsize=6.5, ha="right", va="bottom",
        bbox=dict(boxstyle="round,pad=0.4", fc="white", ec="black", linewidth=0.6))

ax.axis("off")
plt.tight_layout()
fig.savefig(FIGURES_DIR / "rr_choropleth_labeled.png", dpi=300, bbox_inches="tight")
plt.show()

**Takeaway.** Three maps, three layers of the same underlying data. Pool coverage
(`pools_with_social`) is patchy but not sparse anywhere critical — the most pool-dense
constituencies (`Nordfriesland – Dithmarschen Nord`, 53; `Ansbach`, 48; `Erzgebirgskreis I`, 48)
are large rural areas accumulating many small venues, not data errors. The social-review rate
map shows which constituencies' reviews skew toward the social-experience content this whole
analysis depends on — highest in `Bottrop – Recklinghausen III` (57.0% of 2,785 reviews),
`Berlin-Marzahn-Hellersdorf` (53.1%), and `Dahme-Spreewald – Teltow-Fläming III` (51.6% of
21,490) — with `Duisburg II` also appearing here (49.1%), an early hint that its high racialised
rate isn't a small-sample artifact. The racialised-rate map is the substantive one: it is **not**
a diffuse, noisy-looking pattern — it shows a concentrated cluster of high, well-supported rates
in western Germany, led by `Offenburg` (8.6% of 1,724 social reviews), `Duisburg II` (7.6% of
2,316), `Gelsenkirchen` (6.0%), and `Lörrach – Müllheim` (5.7% of 3,975 reviews) — all several
times the 1.54% national rate, and all with sample sizes large enough that this isn't sampling
noise. Combined with the Ruhr-area constituencies flagged in the prevalence-with-CIs subsection
above, there's a recognisable geography here (Ruhr valley NRW + southwestern Baden-Württemberg
border constituencies) that is a genuine spatial pattern, not scattered outliers.

## Spatial autocorrelation: is the clustering in the map above real?

Moved here from what was originally drafted as a Part 2 section: testing this needs only `racialised_rate` and constituency geometry, both already available at this point, not the socioeconomic/political data Part 2 introduces -- no reason to make a reader wait for Part 2 to find out whether the racialised-rate choropleth above is showing a real pattern or just a suggestive color scale. The earlier ICWSM prototype hand-rolled Moran's I with an ad hoc distance-weight threshold, despite already importing `pysal`/`esda` for exactly this purpose. We use the actual library this time: **Queen contiguity weights** (constituencies that share a border or a single point are neighbours) built directly from the Wahlkreis polygons, row-standardized, with **Global Moran's I** tested via permutation (999 random relabellings) rather than a normal-theory p-value, since we can't be sure the normality assumption holds for a rate this skewed.

In [ ]:
from libpysal.weights import Queen
from esda.moran import Moran, Moran_Local

moran_gdf = map_df.dropna(subset=["racialised_rate"]).reset_index(drop=True)

w = Queen.from_dataframe(moran_gdf, use_index=False)
n_islands = len(w.islands)
print(f"constituencies with zero contiguity neighbours (islands): {n_islands}")
if n_islands:
    print(moran_gdf.loc[w.islands, "constituency"].to_list())
w.transform = "r"

moran = Moran(moran_gdf["racialised_rate"].values, w, permutations=999)
print(f"\nGlobal Moran's I = {moran.I:.4f}")
print(f"expected I under spatial randomness = {moran.EI:.4f}")
print(f"p-value (999 permutations) = {moran.p_sim:.4f}")
print(f"z-score = {moran.z_sim:.2f}")


In [ ]:
# Local Moran's I (LISA) to identify *which* constituencies drive the global clustering
from matplotlib.patches import Patch

lisa = Moran_Local(moran_gdf["racialised_rate"].values, w, permutations=999, seed=42)
sig = lisa.p_sim < 0.05

cluster_labels = np.array(["Not significant"] * len(moran_gdf), dtype=object)
quadrant_names = {1: "High-High (hotspot)", 2: "Low-High", 3: "Low-Low (coldspot)", 4: "High-Low"}
for q, name in quadrant_names.items():
    cluster_labels[(lisa.q == q) & sig] = name
moran_gdf["lisa_cluster"] = cluster_labels

print(moran_gdf["lisa_cluster"].value_counts())
print("\nHigh-High (hotspot) constituencies:")
print(moran_gdf.loc[moran_gdf["lisa_cluster"] == "High-High (hotspot)", ["constituency", "state", "racialised_rate"]]
      .sort_values("racialised_rate", ascending=False).to_string(index=False))

fig, ax = plt.subplots(figsize=(7, 8))
palette = {
    "High-High (hotspot)": "#C44E52", "Low-Low (coldspot)": "#4C72B0",
    "Low-High": "#8FBADE", "High-Low": "#E8A9AC", "Not significant": "#E8E8E8",
}
for label, color in palette.items():
    subset = moran_gdf[moran_gdf["lisa_cluster"] == label]
    if len(subset):
        subset.plot(ax=ax, color=color, edgecolor="white", linewidth=0.2)
ax.legend(handles=[Patch(facecolor=c, label=l) for l, c in palette.items()], loc="lower left", fontsize=8)
ax.set_title("LISA cluster map: racialised rate\n(Queen contiguity, 999 permutations, p<0.05)")
ax.axis("off")
plt.tight_layout()
fig.savefig(FIGURES_DIR / "lisa_clusters.png", dpi=130)
plt.show()


**Takeaway.** The clustering is real, not an artifact of the choropleth's color scale: **Global Moran's I = 0.28** (expected value under spatial randomness ≈ 0, z = 7.91, p = 0.001 over 999 permutations) — a moderate, highly significant positive spatial autocorrelation. This confirms what the map above suggested visually and replaces the earlier prototype's ad hoc, hand-rolled Moran's I with the proper library implementation.

The LISA breakdown identifies exactly two coherent **High-High hotspot clusters**, both matching the constituencies already flagged above: a **Ruhr-area NRW cluster** (`Gelsenkirchen`, `Bochum I`, `Essen II`/`III`, `Duisburg I`, `Mülheim – Essen I`, `Herne – Bochum II`, `Oberhausen – Wesel III`, `Mettmann II`, and neighbours) and a **Berlin cluster** (`Berlin-Neukölln`, `Berlin-Mitte`, `Berlin-Tempelhof-Schöneberg`, `Berlin-Friedrichshain-Kreuzberg`, `Berlin-Treptow-Köpenick`, `Berlin-Charlottenburg-Wilmersdorf`, plus the adjoining Brandenburg constituency `Dahme-Spreewald – Teltow-Fläming III`), with a smaller pair (`Rastatt`, `Emmendingen – Lahr`) extending the earlier Offenburg/Lörrach signal in southwestern Baden-Württemberg. **Low-Low coldspots** are broad and diffuse — most of Schleswig-Holstein/Mecklenburg-Vorpommern in the north, a Saxony-Anhalt/Thuringia block in the center-east, and most of Bavaria south of Nuremberg — i.e. the "low" side of the pattern is a general background rate across large, mostly rural regions, while the "high" side is concentrated in specific dense, post-industrial or major-metropolitan constituencies. This is a genuinely paper-worthy geographic finding on its own. **Part 2** (`02_socioeconomic_political_relations.ipynb`) later connects these same hotspots to the deprivation/political predictors it introduces — the hotspots sit in exactly the kind of higher-deprivation, higher-foreign-population-share urban constituencies Part 2 finds correlated with the rate.

## Concentration: how many *pools*, not just reviews, drive a constituency's racialised rate?

The reliability-bar subsection above checked whether a constituency has enough racialised reviews to trust its rate — but not *where within the constituency* those reviews sit. Two constituencies can have identical `racialised_reviews` counts and tell completely different stories: one where the reviews are spread across most of the constituency's pools (a pattern that plausibly reflects something about the *area*), and one where they all sit at one or two specific venues (a pattern that may really be about those *venues*, not the constituency as a whole). This is a sharper, venue-level version of the ecological-fallacy problem: if a "high-rate constituency" is really "one pool with a bad pattern," reading it as evidence about the surrounding area is even shakier than the aggregate framing already implies.

For each constituency we compute: how many of its pools have *at least one* racialised review (`pools_with_racialised`, and as a share of `pools_with_social` — pools that could plausibly have a racialised review at all, since a racialised review is by definition also a social one), and how concentrated the racialised reviews are within it (`top_pool_share_of_racialised` — the share of a constituency's racialised reviews contributed by its single most-implicated pool). A low share of affected pools + a high top-pool share together would mean "one venue is doing most of the work"; a high share of affected pools + a low top-pool share would mean "this is genuinely spread across the area."

In [ ]:
# racialised_df already built earlier (shared Part 1 setup); reused here for pool-level counts
# pool-level racialised counts, then aggregated up to constituency level
pool_racialised = (
    racialised_df.groupby(["constituency_code", "unique_identifier"]).size().reset_index(name="pool_racialised_count")
)
pool_agg = pool_racialised.groupby("constituency_code").agg(
    pools_with_racialised=("unique_identifier", "nunique"),
    top_pool_racialised_count=("pool_racialised_count", "max"),
)

const_agg = const_agg.merge(pool_agg, on="constituency_code", how="left")
const_agg["pools_with_racialised"] = const_agg["pools_with_racialised"].fillna(0).astype(int)
# denominator is pools_with_social, not pool_count: a pool with zero social reviews can never
# have a racialised review, so it shouldn't count against a constituency's "share affected".
const_agg["pct_pools_with_racialised"] = (
    const_agg["pools_with_racialised"] / const_agg["pools_with_social"].replace(0, np.nan)
)
const_agg["top_pool_share_of_racialised"] = (
    const_agg["top_pool_racialised_count"] / const_agg["racialised_reviews"]
)

print("\nShare of a constituency's social-active pools that have >=1 racialised review:")
print(const_agg["pct_pools_with_racialised"].describe())
print()
print("Share of a constituency's racialised reviews contributed by its single most-implicated pool:")
print(const_agg["top_pool_share_of_racialised"].describe())

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(const_agg["pct_pools_with_racialised"], bins=30, color="#4C72B0")
axes[0].set_title("Share of social-active pools with >=1 racialised review")
axes[0].set_xlabel("pct_pools_with_racialised")
axes[0].set_ylabel("number of constituencies")
axes[1].hist(const_agg["top_pool_share_of_racialised"], bins=30, color="#C44E52")
axes[1].set_title("Top-pool share of a constituency's racialised reviews")

plt.tight_layout()
plt.savefig(FIGURES_DIR / "pool_share_and_top_pool_share_racialised_hist.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# racialised_rate (Wilson point estimate) now already exists on const_agg -- computed in the
# Prevalence-with-CIs subsection above, which now runs before this one.
rho_pct, p_pct = stats.spearmanr(const_agg["racialised_rate"], const_agg["pct_pools_with_racialised"])
rho_top, p_top = stats.spearmanr(const_agg["racialised_rate"], const_agg["top_pool_share_of_racialised"])
print(f"racialised rate vs. pct_pools_with_racialised: rho={rho_pct:.3f}, p={p_pct:.2e}")
print(f"racialised rate vs. top_pool_share_of_racialised: rho={rho_top:.3f}, p={p_top:.2e}")

# the specific high-rate constituencies flagged in the prevalence/choropleth subsections above
# (checked here by name)
hotspot_names = [
    "Offenburg", "Duisburg II", "Gelsenkirchen", "Lörrach – Müllheim", "Bochum I",
    "Berlin-Neukölln", "Essen II", "Recklinghausen I",
]
cols = ["constituency", "state", "pool_count", "racialised_reviews",
        "pools_with_racialised", "pct_pools_with_racialised", "top_pool_share_of_racialised", "racialised_rate"]
print("\nConcentration profile of the flagged high-rate constituencies:")
print(const_agg[const_agg["constituency"].isin(hotspot_names)][cols]
      .sort_values("racialised_rate", ascending=False).to_string(index=False))

**Takeaway.** Concentration is real but the pattern is reassuring, not worrying. On average, only **40% of a constituency's social-active pools have any racialised review at all** (median 37.5%), and its single most-implicated pool typically accounts for **~48% of its racialised reviews** (median 47%, ranging all the way to 100% for a few constituencies with very few affected pools) — so no, this was not a negligible detail: a meaningful share of a constituency's racialised-review count often does rest on one or two specific venues, exactly the scenario in your hypothetical.

But the key relationship goes the *reassuring* way: **racialised rate correlates strongly with how many pools are affected** (ρ=0.65, p≪0.001), not with how concentrated the top pool is (ρ=0.09, not significant). In other words, **high-rate constituencies tend to have racialised discourse spread across many of their pools, not concentrated in one "bad venue"** — checking the specific hotspots named earlier confirms this directly: `Gelsenkirchen` (90% of its pools affected) and `Essen II` (83%) are genuinely broad-based patterns, and even the more concentrated-looking cases (`Duisburg II`, `Essen II` at 65-71% top-pool share) have very few pools in total (6), so one venue naturally carries a large share simply because there isn't much to spread across — that's a small-denominator effect, not evidence of a single outlier driving an otherwise-quiet constituency.

## Is the dominant pool unusual in *rate*, or just in *volume*?

The concentration subsection above showed concentration exists but isn't what drives high constituency rates. One more check in the same spirit: when one pool contributes the largest share of a constituency's racialised reviews, is that because the pool has an unusually **elevated racialised rate** (something specific to that venue), or simply because it's the **biggest, most-reviewed pool** in the constituency (so it naturally contributes more of everything, racialised included, without being disproportionately "worse")? These have very different implications — the first is a venue-specific finding, the second is just an artifact of review volume.

For each constituency's single most-implicated ("dominant") pool (the same one identified in the concentration check above, which on average contributes ~48% of a constituency's racialised reviews), we compare it directly against **the rest of its own constituency**, not against pools nationally: its own rate (racialised/social) vs. the combined rate of every other pool in the same constituency, tested with a two-proportion test, and its share of the constituency's total social-review volume. This answers the actual question at the right grain — is *this* pool unusual for *its own area* — without pulling in unrelated pools from other constituencies as a reference population, and without inventing a second reliability threshold: we reuse the existing `reliable_n` bar (247/299 constituencies) already established above.

In [ ]:
# Comparing the dominant pool against pools nationally answered a different question -- "is this
# pool unusual among all German pools" -- and needed an arbitrary reliability floor to keep that
# reference population meaningful, which excluded 62% of constituencies. The real question is
# whether the dominant pool is unusual *for its own constituency*: compare it directly against the
# rest of that constituency's pools combined, reusing the existing reliable_n bar (247/299) instead
# of inventing a pool-level one.
from statsmodels.stats.proportion import proportions_ztest

# the actual dominant pool (not just its count) for each constituency
dominant_idx = pool_racialised.groupby("constituency_code")["pool_racialised_count"].idxmax()

pool_social = corpus.loc[corpus["social"] == "yes"].groupby("unique_identifier").size().rename("dominant_social_reviews")

dominant_pools = pool_racialised.loc[dominant_idx].set_index("constituency_code")
dominant_pools = dominant_pools.join(pool_social, on="unique_identifier")

reliable_const = const_agg.loc[
    const_agg["reliable_n"], ["constituency_code", "constituency", "state", "social_reviews", "racialised_reviews"]
].set_index("constituency_code")
dp = dominant_pools.join(reliable_const, how="inner")

dp["dominant_rate"] = dp["pool_racialised_count"] / dp["dominant_social_reviews"]
dp["rest_racialised"] = dp["racialised_reviews"] - dp["pool_racialised_count"]
dp["rest_social"] = dp["social_reviews"] - dp["dominant_social_reviews"]

n_single_pool = (dp["rest_social"] <= 0).sum()
print(f"reliable constituencies analysed: {len(dp)}")
print(f"single-pool constituencies (no 'rest of constituency' to compare against, excluded): {n_single_pool}")
dp = dp.loc[dp["rest_social"] > 0].copy()

dp["rest_rate"] = dp["rest_racialised"] / dp["rest_social"]
dp["volume_share"] = dp["dominant_social_reviews"] / dp["social_reviews"]

def two_prop_p(row):
    return proportions_ztest(
        [row["pool_racialised_count"], row["rest_racialised"]],
        [row["dominant_social_reviews"], row["rest_social"]],
    )[1]

dp["p_value"] = dp.apply(two_prop_p, axis=1)
dp["significantly_elevated"] = (dp["p_value"] < 0.05) & (dp["dominant_rate"] > dp["rest_rate"])
dp["big"] = dp["volume_share"] > 0.5

# Wilson CIs on each side (the `wilson()` helper defined earlier in Part 1), reported alongside
# the z-test for interpretability -- note these are two different checks: non-overlapping CIs
# implies significance, but overlapping CIs does NOT imply non-significance (CI-overlap is the
# more conservative, weaker test). The z-test above remains the actual significance criterion;
# this just shows how much stricter a bare CI-overlap check would be.
dp["dominant_ci_lo"], dp["dominant_ci_hi"] = zip(*dp.apply(
    lambda r: wilson(r["pool_racialised_count"], r["dominant_social_reviews"])[1:], axis=1))
dp["rest_ci_lo"], dp["rest_ci_hi"] = zip(*dp.apply(
    lambda r: wilson(r["rest_racialised"], r["rest_social"])[1:], axis=1))
dp["ci_non_overlapping"] = dp["dominant_ci_lo"] > dp["rest_ci_hi"]

print(f"\n  of the {dp['significantly_elevated'].sum()} z-test-significant dominant pools, "
      f"{(dp['significantly_elevated'] & dp['ci_non_overlapping']).sum()} "
      f"also have non-overlapping Wilson CIs (the stricter check)")

print(f"\nof {len(dp)} constituencies with a comparable 'rest of constituency':")
print(f"  dominant pool's rate significantly higher than the rest of its own constituency: "
      f"{dp['significantly_elevated'].sum()} ({dp['significantly_elevated'].mean():.1%})")
print(f"  dominant pool contributes >50% of the constituency's social-review volume: "
      f"{dp['big'].sum()} ({dp['big'].mean():.1%})")

quadrants = pd.Series(
    np.select(
        [dp["big"] & dp["significantly_elevated"], dp["big"] & ~dp["significantly_elevated"],
         ~dp["big"] & dp["significantly_elevated"], ~dp["big"] & ~dp["significantly_elevated"]],
        ["big pool, significantly elevated rate", "big pool, not significantly elevated",
         "small pool, significantly elevated rate", "small pool, not significantly elevated"],
        default="unknown",
    )
).value_counts()
print("\nWhich kind of pool is 'dominant' in its constituency?")
print(quadrants)

small_elevated = dp.loc[~dp["big"] & dp["significantly_elevated"]].sort_values("dominant_rate", ascending=False)
print(f"\nThe {len(small_elevated)} 'small pool, significantly elevated rate' constituencies:")
print(small_elevated[["constituency", "state", "pool_racialised_count", "dominant_social_reviews",
                       "dominant_rate", "dominant_ci_lo", "dominant_ci_hi", "rest_rate", "volume_share",
                       "p_value", "ci_non_overlapping"]].to_string(index=False))

fig, ax = plt.subplots(figsize=(6, 5))
colors = np.where(dp["significantly_elevated"], "#C44E52", "#4C72B0")
ax.errorbar(
    dp["rest_rate"] * 100, dp["dominant_rate"] * 100,
    yerr=[(dp["dominant_rate"] - dp["dominant_ci_lo"]) * 100, (dp["dominant_ci_hi"] - dp["dominant_rate"]) * 100],
    xerr=[(dp["rest_rate"] - dp["rest_ci_lo"]) * 100, (dp["rest_ci_hi"] - dp["rest_rate"]) * 100],
    fmt="none", ecolor=colors, alpha=0.3, linewidth=0.8,
)
ax.scatter(dp["rest_rate"] * 100, dp["dominant_rate"] * 100, c=colors, alpha=0.6, s=20, zorder=3)
lims = [0, max(dp["dominant_ci_hi"].max(), dp["rest_ci_hi"].max()) * 100 * 1.05]
ax.plot(lims, lims, color="grey", linestyle="--", linewidth=0.8)
ax.set_xlabel("rate of the rest of the constituency's pools (%)")
ax.set_ylabel("dominant pool's own rate (%)")
ax.set_title("Dominant pool's rate vs. the rest of its own constituency\n(red = significantly elevated, p<0.05)")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "dominant_pool_vs_rest_of_constituency_rate.png", dpi=150, bbox_inches="tight")
plt.show()

**Takeaway.** Comparing each constituency's dominant pool directly against the rest of its own constituency (a two-proportion test, `reliable_n` constituencies only, n=247 -- no single-pool constituencies to exclude) gives a cleaner and more concerning picture than either earlier version.

**40% of dominant pools (99/247) have a rate that is statistically significantly higher than the rest of their own constituency's pools combined.** Splitting by volume share: **54/247 (22%) of dominant pools carry more than half their constituency's social-review volume** ("big"); of those, only 9/247 (3.6%) are *also* significantly elevated -- for most big dominant pools, their size alone explains their contribution, consistent with the earlier finding. But **90/247 (36%) are dominant pools that carry a minority of the volume and are still significantly elevated** -- a real venue-specific effect, not a volume artifact, and a much larger share than either earlier pass suggested. `Lübeck` and `Mansfeld` (5/21 = 23.8% each) are legitimately on this list: a proper significance test says a 23.8% rate against a ~0.7% constituency baseline is extreme regardless of the small raw count (p≈2e-18) -- the earlier percentile-based versions either flagged this kind of case on a shaky basis (no minimum-count floor) or excluded it altogether (an arbitrary national floor); testing directly against the constituency's own baseline handles both problems at once, with no extra threshold needed. Cross-checked against Wilson CIs (the same interval used throughout Part 1): 60 of the 99 z-test-significant dominant pools also have non-overlapping CIs against the rest of their constituency -- the z-test is the correct primary criterion here (non-overlapping CIs implies significance, but overlapping CIs doesn't imply the reverse), so the other 39 are not spurious, just below the more conservative CI-overlap bar. `Lübeck`/`Mansfeld` clear both: their dominant-pool CI (10.6-45.1%) sits entirely above the rest-of-constituency rate.

In [ ]:
# flag these constituencies on const_agg for downstream robustness checks, without recomputing
# pool-level stats. Definition updated to match the corrected
# within-constituency test: dominant pool carries a minority of constituency volume AND is
# significantly elevated vs. the rest of the constituency (was: national percentile-based).
const_agg["dominant_pool_is_small_high_rate"] = const_agg["constituency_code"].isin(small_elevated.index)
print(f"flagged on const_agg: {const_agg['dominant_pool_is_small_high_rate'].sum()} constituencies")

### Draft: paper write-up table (WHERE section findings)

One table: constituency-level correlation/significance tests (pool-concentration, dominant-pool test, border distance). Trimmed to what the prose actually cites -- dropped the old prevalence/reliability summary table (redundant with Fig. 1 + the opening paragraph) and the uncorrected "national boundary incl. coastline" border-distance rows (not referenced in text; only the corrected land-neighbour measure is). Source file: `where_section_findings_tables.tex` (session scratchpad, not reachable from your side -- this cell is the copy to actually use). Move into the paper repo when ready.

```latex
% Requires: \usepackage{booktabs} \usepackage{array}
% All numbers pulled directly from descriptive_analysis.ipynb's WHERE section
% takeaway cells (Concentration / Is the dominant pool unusual / Geometric proxy subsections).
% Single-column table. Test column is a wrapping p{} column (not plain l) and labels are
% shortened -- the previous version's long Test strings (e.g. "Dominant pool sig. elevated
% vs. rest of constituency") forced the table past \linewidth in a two-column layout.

\begin{table}[htbp]
\centering
\footnotesize
\setlength{\tabcolsep}{3pt}
\caption{WHERE section: correlation and significance tests (constituency level; reliable\_n = $\geq$10 racialised reviews, 247/299).}
\label{tab:where-correlations}
\begin{tabular}{@{} p{3.0cm} r l r c @{}}
\toprule
Test & $\rho$ & $p$ & $n$ & Sig. \\
\midrule
Rate vs.\ \% pools w/ racialised review$^\dagger$ & 0.65 & $\ll 0.001$ & 299 & *** \\
Rate vs.\ top-pool share$^\P$ & 0.09 & n.s. & 299 & ns \\
Dominant pool sig.\ elevated$^\ddagger$ & -- & -- & 247 & 99/247 (40\%) \\
Border dist.$^\S$, all constituencies & $-0.055$ & 0.341 & 299 & ns \\
Border dist.$^\S$, reliable\_n only & $-0.000$ & 0.996 & 247 & ns \\
\bottomrule
\end{tabular}

\vspace{2pt}
\raggedright\footnotesize
$^{***}p<0.001$, ns = not significant. $^\dagger$Share of a constituency's social-active pools (not all pools) with $\geq$1 racialised review. $^\P$Share of a constituency's racialised reviews contributed by its single most-implicated pool. $^\ddagger$Per-constituency two-proportion $z$-test vs.\ rest of constituency, not a single correlation; of these 99, 54/247 (22\%) overall also carry $>$50\% of the constituency's review volume (``big''), while 90/247 (36\%) are minority-volume pools that are still significantly elevated -- see text. $^\S$Distance to nearest land-neighbour country.
\end{table}
```

## Geometric proxy: how close is each constituency to a neighbouring country's border?

Using constituency polygons from Section 1 and actual country polygons (Natural Earth, bundled
with this environment's `pyogrio` install) for Germany's 9 land neighbours — France, Poland,
Austria, Switzerland, Luxembourg, Belgium, Netherlands, Denmark, Czechia — we compute
`distance_to_neighbor_country_km` (straight-line distance to the nearest neighbouring-country
polygon) and the binary `is_land_border_constituency` (within 150m of one). We deliberately
measure distance to a neighbouring country's polygon, not to Germany's own external boundary
line: the latter also includes the North Sea and Baltic coastlines, which would incorrectly put
a coastal-but-inland constituency like `Bremen II – Bremerhaven` at the same 0km as a genuine
cross-border constituency like `Offenburg`. Both columns are saved to `const_agg`. This is a bivariate, purely descriptive check — it does
not control for anything (state, foreign-population share, urban/rural, etc.).

In [ ]:
world_path = "/home/ethel/Desktop/swimming_pool_project/.venv/lib/python3.12/site-packages/pyogrio/tests/fixtures/naturalearth_lowres/naturalearth_lowres.shp"
world = gpd.read_file(world_path)
NEIGHBOURS = ["France", "Poland", "Austria", "Switzerland", "Luxembourg", "Belgium", "Netherlands", "Denmark", "Czechia"]
neighbour_gdf = world[world["name"].isin(NEIGHBOURS)].to_crs(constituencies.crs)
print(f"neighbouring countries matched: {sorted(neighbour_gdf['name'].tolist())} ({len(neighbour_gdf)}/9)")

neighbour_union = neighbour_gdf.union_all()
dist_by_wkr = constituencies.set_index("WKR_NR").geometry.distance(neighbour_union) / 1000
const_agg["distance_to_neighbor_country_km"] = const_agg["constituency_code"].map(dist_by_wkr)
const_agg["is_land_border_constituency"] = const_agg["distance_to_neighbor_country_km"] < 0.15

n_land_border = const_agg["is_land_border_constituency"].sum()
print(f"land-border constituencies: {n_land_border} / 299 ({n_land_border / 299:.1%})")

# sanity check: known hotspots on the France/Switzerland border should be True and near 0km; a
# genuinely coastal-but-inland seat and clearly interior constituencies should be False
check_names = ["Offenburg", "Lörrach – Müllheim", "Bremen II – Bremerhaven", "Frankfurt am Main I", "Kassel"]
print(const_agg.loc[const_agg["constituency"].isin(check_names),
                     ["constituency", "state", "is_land_border_constituency", "distance_to_neighbor_country_km"]]
      .to_string(index=False))

# Does racialised_rate actually vary smoothly with distance to a land border? Spearman rather
# than Pearson, for the same reason as elsewhere in this notebook: racialised_rate is a skewed
# proportion, not a normally-distributed quantity.
rho_all, p_all = stats.spearmanr(const_agg["distance_to_neighbor_country_km"], const_agg["racialised_rate"])
rho_rel, p_rel = stats.spearmanr(
    const_agg.loc[const_agg["reliable_n"], "distance_to_neighbor_country_km"],
    const_agg.loc[const_agg["reliable_n"], "racialised_rate"],
)
print(f"\ndistance_to_neighbor_country_km vs. racialised_rate: all 299 rho={rho_all:.3f} p={p_all:.3f} | "
      f"reliable_n only (n={const_agg['reliable_n'].sum()}) rho={rho_rel:.3f} p={p_rel:.3f}")

# coarse near/mid/far split on fixed, round-number breaks (a plain quantile split doesn't work
# cleanly here: the land-border constituencies all sit near 0km, which collapses qcut's first
# two bin edges together)
const_agg["dist_band"] = pd.cut(
    const_agg["distance_to_neighbor_country_km"], bins=[-0.1, 30, 70, 300],
    labels=["<=30km", "30-70km", ">70km"],
)
band_rates = const_agg.groupby("dist_band", observed=True).apply(
    lambda g: pd.Series(wilson(g["racialised_reviews"].sum(), g["social_reviews"].sum()),
                         index=["rate", "ci_lo", "ci_hi"])
)
band_rates["n_constituencies"] = const_agg.groupby("dist_band", observed=True).size()
print("\nracialised rate by distance-to-land-border band (pooled numerator/denominator, not averaged per-constituency):")
print(band_rates)

fig, ax = plt.subplots(figsize=(8, 5))
colors = np.where(const_agg["reliable_n"], "#4C72B0", "#C44E52")
ax.scatter(const_agg["distance_to_neighbor_country_km"], const_agg["racialised_rate"], c=colors, s=18, alpha=0.7)
for name in ["Offenburg", "Lörrach – Müllheim", "Duisburg II", "Gelsenkirchen"]:
    row = const_agg.loc[const_agg["constituency"] == name]
    if len(row):
        ax.annotate(name, xy=(row["distance_to_neighbor_country_km"].iloc[0], row["racialised_rate"].iloc[0]),
                    fontsize=7, xytext=(4, 4), textcoords="offset points")
ax.set_xlabel("distance to nearest land-neighbour country (km)")
ax.set_ylabel("racialised rate (of social reviews)")
ax.set_title(f"Racialised rate vs. distance to land border (rho={rho_all:.2f}, p={p_all:.2f}, n=299)\n"
             "blue = reliable_n, red = below reliability bar")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "racialised_rate_vs_distance_to_land_border_scatter.png", dpi=150, bbox_inches="tight")
plt.show()

**Takeaway.** Racialised rate does not track distance to a land-neighbour border: the
correlation is weak and not significant (all 299 constituencies: rho=-0.055, p=0.341;
`reliable_n` only: rho=-0.000, p=0.996). The banded view shows why it isn't monotonic:
`<=30km` sits near the national rate, `30-70km` runs *above* it, `>70km` runs *below* it —
because the two single highest-rate constituencies nationally, `Duisburg II` (7.6%) and
`Gelsenkirchen` (6.0%), are Ruhr-area NRW constituencies far from any land border (28-36km
inland), while the constituencies genuinely closest to a border nationally (Schleswig-Holstein,
near Denmark) sit at or below the national rate. Different border segments pull in opposite
directions and cancel out in an aggregate correlation — the high-rate cluster shown in the
choropleth (`Offenburg`, `Lörrach – Müllheim`, near France/Switzerland) is specific to that
stretch, not a general border-proximity effect. `is_land_border_constituency` /
`distance_to_neighbor_country_km` are saved to `const_agg` as covariates, not as a validated
standalone explanation.

# WHO

Before anything ethnicity- or language-based: a look at reviewer identity itself -- how much
of the corpus comes from repeat reviewers (and whether racialised reviews are structurally
different there), and how completely the name-to-country classifier actually covers the
reviewer names it was given, since every comparison in this section rests on that coverage.
Then the reviewer-ethnicity name-origin signal proper: a data-quality check on `org_review_lang`,
then composition of racialised=yes vs. racialised=no reviews using a refined name-origin
categorization built once and reused throughout -- a genuine "Other" separated from a genuine
"Unknown", plus an auditable "Arab League" aggregate -- extended to a three-way comparison
across the entire corpus / social=yes / racialised=yes grains, then a review-language facet,
a German vs. non-German cut against the four content dimensions
(`theme`/`context`/`connotation`/`approach`) refined further by specific language (within
Non-German reviewers) and by top ethnicity, then consolidated into four theoretically-motivated
groups (`Germany`/`Arab`/`Türkiye`/`Europe`) and checked against East/West/Berlin geography.
A closing section then brings this together with the constituency-level rate from WHERE to ask
what's actually driving it.

## Reviewer identity structure: how unique are reviewers and names, and how well does the classifier cover them?

Two things worth establishing before any ethnicity/language analysis: `reviewer_id` is not
equally "one row per person" in every grain -- some reviewers post many reviews, and that
repeat-reviewer share could differ between racialised and non-racialised discourse in a way
that would bias composition comparisons. And `ethnicity_country_name` comes from a name-level
lookup (reviewer names were deduplicated before being sent to the prediction tool, so it was
never asked to classify the same string twice) -- worth knowing not just the overall
classified/unclassified split, but whether that coverage differs for names that repeat across
multiple reviewers versus names that appear once, and whether racialised reviews look any
different from the corpus at large on either front.

In [ ]:
def reviewer_id_breakdown(label, df):
    g = df.dropna(subset=["reviewer_id"]).groupby("reviewer_id").size()
    singleton = (g == 1).sum()
    repeat_counts = g[g > 1]
    excess_row_share = 1 - len(g) / len(df)
    print(f"--- {label} ---")
    print(f"unique reviewer_id: {len(g):,} / {len(df):,} rows")
    print(f"  singleton (1 review): {singleton:,} ({singleton / len(g):.1%})")
    print(f"  repeat (>1 review):    {len(repeat_counts):,} ({len(repeat_counts) / len(g):.1%}), "
          f"{repeat_counts.sum():,} rows ({repeat_counts.sum() / len(df):.1%} of rows)")
    print(f"  excess-row share (rows beyond one-per-id): {excess_row_share:.1%}")
    if len(repeat_counts):
        print(f"  spread among repeat reviewers: median={repeat_counts.median():.0f}, mean={repeat_counts.mean():.1f}, "
              f"90th pct={repeat_counts.quantile(0.9):.0f}, max={repeat_counts.max():,}")
    print()
    return g

id_corpus = reviewer_id_breakdown("Entire corpus", corpus)
id_social = reviewer_id_breakdown("Social = yes", corpus.loc[corpus["social"] == "yes"])
id_racialised = reviewer_id_breakdown("Racialised = yes", corpus.loc[corpus["racialised"] == "yes"])

**Takeaway.** Racialised reviews are overwhelmingly one-off: **98.4% of racialised reviewer_ids have exactly one review** (only 1.6% repeat, accounting for just 3.4% of racialised rows), compared to 86.9% singleton at social grain and 78.8% at corpus grain. The excess-row share (rows beyond one-per-id) drops from 27.1% (corpus) to 15.8% (social) to **1.8%** (racialised) -- repeat-reviewer structure shrinks sharply and monotonically as you move toward the subset that actually matters for this analysis. Where repeat reviewers do exist, the spread is heavily right-skewed (median exactly 2 reviews even at corpus grain, a long thin tail of power-reviewers rather than a bulk of prolific ones) -- so the handful of reviewers with many reviews are not a meaningful confound for racialised-specific comparisons, since that grain barely has any repeat reviewers to begin with.

In [ ]:
def name_classification_breakdown(label, df):
    g = df.dropna(subset=["reviewer_name"]).groupby("reviewer_name").agg(
        n_rows=("reviewer_name", "size"),
        classified=("ethnicity_country_name", lambda s: s.notna().any()),
    )
    singleton = g[g["n_rows"] == 1]
    repeating = g[g["n_rows"] > 1]
    print(f"--- {label} ---")
    print(f"unique names: {len(g):,}")
    print(f"  singleton names: {len(singleton):,} -- classified {singleton['classified'].mean():.1%} "
          f"({singleton['classified'].sum():,} of {len(singleton):,})")
    print(f"  repeating names: {len(repeating):,} -- classified {repeating['classified'].mean():.1%} "
          f"({repeating['classified'].sum():,} of {len(repeating):,})")
    print()
    return g

names_corpus = name_classification_breakdown("Entire corpus", corpus)
names_social = name_classification_breakdown("Social = yes", corpus.loc[corpus["social"] == "yes"])
names_racialised = name_classification_breakdown("Racialised = yes", corpus.loc[corpus["racialised"] == "yes"])

**Takeaway.** Classification coverage sits around 95% for both singleton and repeating names at corpus grain (95.2%/95.3%) and social grain (95.0%/94.1%) -- being shared across multiple reviewers doesn't make a name meaningfully harder to classify at these grains. Racialised grain is lower overall (90.2%) and the gap between singleton (91.1%) and repeating (68.4%) names is much wider -- but repeating names are a tiny slice there (269 of 7,389 unique names), so this is a small-N pattern worth explaining directly rather than a robust trend on its own -- see the next two checks.

In [ ]:
rac = corpus.loc[corpus["racialised"] == "yes"]
g = rac.dropna(subset=["reviewer_name"]).groupby("reviewer_name")["reviewer_id"].agg(["nunique", "size"])
repeating = g[g["size"] > 1]
same_id = repeating[repeating["nunique"] == 1]
diff_id = repeating[repeating["nunique"] > 1]

print(f"racialised repeating names: {len(repeating)}")
print(f"  same reviewer_id every time (genuine repeat reviewer): {len(same_id)} names, {same_id['size'].sum()} rows")
print(f"  different reviewer_ids (coincidental name collision):  {len(diff_id)} names, {diff_id['size'].sum()} rows")
print()
print("most-shared names in the racialised set (by number of distinct reviewer_ids using that name):")
print(diff_id.sort_values("nunique", ascending=False).head(10))

**Takeaway.** Racialised "repeating names" are mostly *not* the same person reviewing twice: 161 of the 269 repeating names (414 of 648 rows) are **coincidental collisions** -- different real reviewers who happen to share a generic name -- versus 108 names (234 rows) that really are one reviewer posting multiple times. The collision list is dominated by bare initials (`M`, shared by 9 different reviewer_ids; `M M`, `S K`, `S` by 6-7 each) and common first-name-only entries (`Chris`, `Tom`, `Michael`, `Ben`), plus one literal placeholder -- `Vorname Nachname`, German for "Firstname Lastname" -- used by 6 different reviewer_ids who apparently never entered a real name. This is consistent with, not contradictory to, the reviewer-ID finding above: racialised reviewers are almost all one-off individually, and the small "repeating name" pool is mostly an artifact of generic/placeholder names, not evidence of a repeat-reviewer subpopulation.

In [ ]:
def unclassified_name_nature(label, df):
    g = df.dropna(subset=["reviewer_name"]).groupby("reviewer_name")["ethnicity_country_name"].apply(lambda s: s.notna().any())
    unclassified_names = g[~g].index.to_series()
    n = len(unclassified_names)
    word_count = unclassified_names.str.split().str.len()
    char_len = unclassified_names.str.len()
    single_word = (word_count == 1).sum()
    non_ascii = unclassified_names.apply(lambda s: any(ord(c) > 127 for c in s)).sum()
    print(f"--- {label}: unclassified names (n={n:,}) ---")
    print(f"  single-word (no space): {single_word:,} ({single_word / n:.1%})")
    print(f"  contains non-ASCII char: {non_ascii:,} ({non_ascii / n:.1%})")
    print(f"  avg word count: {word_count.mean():.2f}, avg char length: {char_len.mean():.1f}")
    top = df[df["reviewer_name"].isin(unclassified_names)]["reviewer_name"].value_counts().head(8)
    print(f"  most common unclassified name-strings:")
    print(top.to_string())
    print()

unclassified_name_nature("Entire corpus", corpus)
unclassified_name_nature("Social = yes", corpus.loc[corpus["social"] == "yes"])
unclassified_name_nature("Racialised = yes", corpus.loc[corpus["racialised"] == "yes"])

**Takeaway.** The classifier fails on the same kind of name everywhere -- this is not a racialised-specific problem. At every grain the most common unclassified strings are bare initials (`M`, `M K`, `A. S.`, `S. B.`) that simply can't be mapped to a country of origin, regardless of how the review is classified. If anything, racialised unclassified names are a *milder* case of the same pattern: less often single-word (37.1% vs. 52.9% corpus, 51.0% social) and less often non-ASCII (4.7% vs. 7.2% corpus, 6.7% social) -- just a much smaller absolute set (720 names) than the other two grains. The only genuinely distinct entries in the racialised unclassified list are one-offs (a concatenated no-space name, a gaming handle) rather than a pattern -- there's no separate failure mode specific to racialised reviews, just the general limit of name-to-country prediction on initials-only names, showing up proportionally less here than elsewhere.

### Draft: paper write-up tables (reviewer-ID structure + name classification)

Two small tables: (1) reviewer_id singleton/repeat structure by grain, (2) name classification coverage by grain, singleton vs. repeating names. Source cells: `whoid-code1`/`whoid-take1` (table 1), `whoid-code2`/`whoid-code3`/`whoid-take2`/`whoid-take3` (table 2).

```latex
% Requires: \usepackage{booktabs}

% ------------------------------------------------------------------
% Table 1 -- reviewer_id structure
% ------------------------------------------------------------------
\begin{table}[htbp]
\centering
\footnotesize
\setlength{\tabcolsep}{4pt}
\caption{Reviewer-\texttt{id} structure: singleton vs.\ repeat reviewers, by grain.}
\label{tab:reviewer-id-structure}
\begin{tabular}{@{} l r r r r @{}}
\toprule
Grain & Unique IDs & Singleton & Repeat & Excess-row \\
\midrule
Entire corpus     & 923{,}168 & 78.8\% & 21.2\% & 27.1\% \\
Social = yes      & 424{,}420 & 86.9\% & 13.1\% & 15.8\% \\
Racialised = yes  &   7{,}627 & 98.4\% &  1.6\% &  1.8\% \\
\bottomrule
\end{tabular}

\vspace{2pt}
\raggedright\footnotesize
Singleton/Repeat = share of unique reviewer\_ids with exactly one review vs.\ more than one. Excess-row = rows beyond one-per-\texttt{id}, as a share of all rows in that grain. Spread among repeat reviewers is heavily right-skewed at every grain (median 2 reviews throughout); 90th percentile / max: corpus 4 / 155, social 3 / 89, racialised 3 / 5.
\end{table}

% ------------------------------------------------------------------
% Table 2 -- name classification coverage
% ------------------------------------------------------------------
\begin{table}[htbp]
\centering
\footnotesize
\setlength{\tabcolsep}{4pt}
\caption{Name-to-country classification coverage, singleton vs.\ repeating reviewer names, by grain.}
\label{tab:name-classification-coverage}
\begin{tabular}{@{} l r r r @{}}
\toprule
Grain & Unique names & Singleton class.\ & Repeating class.\ \\
\midrule
Entire corpus     & 758{,}558 & 95.2\% & 95.3\% \\
Social = yes      & 356{,}220 & 95.0\% & 94.1\% \\
Racialised = yes  &   7{,}389 & 91.1\% & 68.4\%$^\dagger$ \\
\bottomrule
\end{tabular}

\vspace{2pt}
\raggedright\footnotesize
Singleton/Repeating class.\ = \% of unique names in that category with a country prediction. $^\dagger$Only 269 repeating names at racialised grain (vs.\ 194,909 corpus-wide) -- of these, 161 (414 rows) are coincidental collisions between different reviewers sharing a generic name (e.g.\ bare initials, or the literal placeholder \textit{Vorname Nachname}), not genuine repeat reviewers (108 names, 234 rows). Unclassified names follow the same pattern at every grain regardless -- dominated by bare-initials entries (\textit{M}, \textit{M K}, \textit{A. S.}) the classifier cannot map to a country; racialised unclassified names are if anything less often single-word (37.1\% vs.\ 52.9\% corpus) or non-ASCII (4.7\% vs.\ 7.2\% corpus), just a smaller absolute set (720 names).
\end{table}
```

## Reviewer ethnicity facet

**Key question:** does reviewer name-origin composition differ between racialised and non-racialised discourse, and does that shift differ by grain (entire corpus / social / racialised)?

`ethnicity_country_name` is a **name-to-country prediction**, not a verified identity — per the corpus README, 51,935 reviewer names were shared by more than one reviewer ID (affecting 313,200 rows), and age/gender predictions were intentionally excluded as unreliable. We treat this strictly as a name-origin signal for descriptive comparison, not as ground truth about who wrote which review.

Rather than a simple top-10-country cut with everything else dumped into one "Other/unknown" label, we build the categorization we actually want to read from the start: the top-10 non-Arab countries kept individually, an **Arab League aggregate** (individual Arab-League member-state name-origins are too small to clear a top-10 cut on their own, but scattered across ~20 countries they may still be a meaningful minority signal), a genuine **Other** (a real, non-top-10, non-Arab country), and a genuine **Unknown** (a missing prediction) — kept distinct rather than conflated into one catch-all. The top-10 cut is computed on the full `social == 'yes'` population (504,072 rows), not the much smaller racialised=yes subset, so a rare country's ranking doesn't wobble on small-sample noise.

We look at this three ways, in order: first the same categories across all three natural grains, to confirm filtering to `social == 'yes'` doesn't itself skew composition; then a direct test of whether composition differs specifically between racialised=yes and racialised=no (within social=yes); then, holding each group's own size fixed, whether a review from a given group is itself more likely to be racialised at all. This same categorization is reused, unchanged, later in this section for the language facet.

In [ ]:
# --- Arab League aggregate + Other/Unknown split: built once here (on the full social=yes
# population, so the categorization is stable enough to reuse everywhere else in this section)
ARAB_LEAGUE_VARIANTS = {
    "Algeria": ["Algeria", "People's Democratic Republic of Algeria"],
    "Bahrain": ["Bahrain", "Kingdom of Bahrain"],
    "Comoros": ["Comoros", "Union of the Comoros"],
    "Djibouti": ["Djibouti", "Republic of Djibouti"],
    "Egypt": ["Egypt", "Egypt, Arab Republic of"],
    "Iraq": ["Iraq", "Republic of Iraq"],
    "Jordan": ["Jordan", "Hashemite Kingdom of Jordan"],
    "Kuwait": ["Kuwait", "State of Kuwait"],
    "Lebanon": ["Lebanon", "Lebanese Republic"],
    "Libya": ["Libya", "Libyan Arab Jamahiriya", "State of Libya"],
    "Mauritania": ["Mauritania", "Islamic Republic of Mauritania"],
    "Morocco": ["Morocco", "Kingdom of Morocco"],
    "Oman": ["Oman", "Sultanate of Oman"],
    "Palestine": ["Palestine, State of", "State of Palestine", "Palestinian Territory, Occupied"],
    "Qatar": ["Qatar", "State of Qatar"],
    "Saudi Arabia": ["Saudi Arabia", "Kingdom of Saudi Arabia"],
    "Somalia": ["Somalia", "Federal Republic of Somalia"],
    "Sudan": ["Sudan", "Republic of the Sudan"],
    "Syria": ["Syrian Arab Republic", "Syria"],
    "Tunisia": ["Tunisia", "Tunisian Republic"],
    "United Arab Emirates": ["United Arab Emirates"],
    "Yemen": ["Yemen", "Republic of Yemen"],
}

def build_ethnicity_final(country_series, n_top=10):
    """Top-N non-Arab countries, kept individually, + an Arab-League aggregate + explicit
    Other (real, non-top-N, non-Arab country) / Unknown (missing prediction)."""
    present_values = set(country_series.dropna().unique())
    arab_value_map = {
        v: country
        for country, variants in ARAB_LEAGUE_VARIANTS.items()
        for v in variants
        if v in present_values
    }
    arab_mask = country_series.isin(arab_value_map.keys())

    non_arab_mask = ~arab_mask & country_series.notna()
    top_countries = country_series.loc[non_arab_mask].value_counts().head(n_top).index

    final = pd.Series("Other", index=country_series.index)
    final = final.mask(country_series.isin(top_countries), country_series)
    final = final.mask(arab_mask, "Arab")
    final = final.mask(country_series.isna(), "Unknown")
    return final, arab_value_map, list(top_countries)

social_df = corpus.loc[social_mask].copy()
n_ethnicity_missing = social_df["ethnicity_country_name"].isna().mean()
print(f"social=yes reviews missing an ethnicity_country_name prediction: {n_ethnicity_missing:.1%}")

social_df["ethnicity_final"], arab_value_map, top_countries = build_ethnicity_final(social_df["ethnicity_country_name"])
found_arab_countries = sorted(set(arab_value_map.values()))
print(f"Arab League countries matched in ethnicity_country_name: {len(found_arab_countries)}/22")

print(f"\ncategory counts (social=yes, N={len(social_df):,}):")
print(social_df["ethnicity_final"].value_counts())

### Top ethnicities: entire corpus vs. social vs. racialised reviews

Before comparing racialised=yes against racialised=no directly, worth seeing the same `ethnicity_final` categories at all three natural grains side by side: the **entire corpus** (1,265,806 reviews, including the ~40% not judged to be about the social experience at all), **social=yes** (504,072), and **racialised=yes** (7,768). This shows how much composition shifts as we narrow from "everyone reviewing a pool" down to "reviews flagged as racialised" -- the entire-vs-social columns serve as a sanity check that filtering to `social == 'yes'` doesn't itself skew composition much, so any real shift should show up between social and racialised specifically, tested directly in the next subsection.

We reuse the exact category scheme already fit above (`top_countries`, `arab_value_map`, both fit on the `social == 'yes'` population) and simply apply it to the full corpus too, rather than refitting a separate top-10 cut on 1,265,806 rows -- refitting here would risk a third, subtly different category list for the same underlying scheme.

In [ ]:
def apply_ethnicity_final(country_series, top_countries, arab_value_map):
    """Applies an already-fit ethnicity_final scheme (top_countries + arab_value_map, both fit
    on the social=yes population above) to another population, without refitting a new top-10 cut."""
    arab_mask = country_series.isin(arab_value_map.keys())
    final = pd.Series("Other", index=country_series.index)
    final = final.mask(country_series.isin(top_countries), country_series)
    final = final.mask(arab_mask, "Arab")
    final = final.mask(country_series.isna(), "Unknown")
    return final

corpus["ethnicity_final"] = apply_ethnicity_final(corpus["ethnicity_country_name"], top_countries, arab_value_map)
racialised_df["ethnicity_final"] = social_df.loc[racialised_df.index, "ethnicity_final"]

three_way = pd.DataFrame({
    "entire_corpus_pct": corpus["ethnicity_final"].value_counts(normalize=True) * 100,
    "social_pct": social_df["ethnicity_final"].value_counts(normalize=True) * 100,
    "racialised_pct": racialised_df["ethnicity_final"].value_counts(normalize=True) * 100,
}).fillna(0).sort_values("racialised_pct", ascending=False)

print(f"entire corpus: N={len(corpus):,}  |  social=yes: N={len(social_df):,}  |  racialised=yes: N={len(racialised_df):,}\n")
print(three_way.round(2))

three_way.plot(kind="barh", figsize=(8, 7))
plt.xlabel("% of reviews in that population")
plt.title("Ethnicity name-origin composition: entire corpus vs. social=yes vs. racialised=yes")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig(FIGURES_DIR / "ethnicity_composition_corpus_social_racialised.png", dpi=150, bbox_inches="tight")
plt.show()


**Takeaway.** The entire-corpus and social=yes columns are nearly identical (Germany 39.58% vs. 39.62%, United States 13.49% vs. 14.00%, Other 24.30% vs. 23.27%) -- confirming that filtering down to `social == 'yes'` does not itself introduce a meaningful compositional shift in reviewer name-origin. The real shift happens entirely between social and racialised -- confirmed formally in the next subsection, which tests it directly against racialised=no. Anchored against the full corpus too: **Germany drops sharply** to 27.63% of racialised reviews (vs. ~39.6% at the other two grains), **Other** actually rises slightly (23.27% -> 25.54%) to become nearly as large a category as Germany, **Unknown** rises from 7.74% to 11.07%, and **Arab** (1.92% -> 5.95%, ~3.1x) and **Türkiye** (1.81% -> 4.78%, ~2.6x) roughly triple. Every other top-10 country (Russian Federation, France, Indonesia, Poland, Philippines, Italy, Brazil) sits within a point or two of its social-grain share at all three grains -- the racialised-vs-not shift is concentrated in Germany, Arab, Türkiye, and (to a lesser extent) Unknown, not spread evenly across the whole category list.

### Does composition differ specifically between racialised=yes and racialised=no reviews?

The three-way comparison above shows *where* the shift concentrates (between social and racialised, not between corpus and social); this tests it formally, within `social == 'yes'`, with a chi-square test of independence -- review-level, not deduplicated by reviewer, since we care about the composition of discourse instances, not the reviewer population.

In [ ]:
comp = (
    social_df.groupby("racialised")["ethnicity_final"]
    .value_counts(normalize=True)
    .rename("share")
    .reset_index()
    .pivot(index="ethnicity_final", columns="racialised", values="share")
    .fillna(0)
    .sort_values("yes", ascending=False)
)
print("share of each racialised group's own reviews (each column sums to 1):")
print(comp)

# association test: is ethnicity_final distribution different for racialised=yes vs no?
ct = pd.crosstab(social_df["ethnicity_final"], social_df["racialised"])
chi2, p, dof, _ = stats.chi2_contingency(ct)
n = ct.values.sum()
cramers_v = np.sqrt(chi2 / (n * (min(ct.shape) - 1)))
print(f"\nchi2 = {chi2:.1f}, dof = {dof}, p = {p:.2e}, Cramer's V = {cramers_v:.4f}")

comp.plot(kind="barh", figsize=(8, 6))
plt.xlabel("share of reviews social=yes, by racialised=yes vs. no")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "ethnicity_composition_racialised_yes_vs_no.png", dpi=150, bbox_inches="tight")
plt.show()

**Takeaway.** The association is statistically significant (χ² = 1573.1, p ≪ .001) but the effect size is small (Cramér's V = 0.056) — with ~500K reviews, even a small compositional shift clears significance easily, so **the p-value alone is not the interesting number here; the category-level shifts are.** Three shifts stand out: reviewers with a Germany name-origin signal make up 39.8% of racialised=no reviews but only 27.6% of racialised=yes reviews; **Arab** rises from 1.9% to 5.9% (~3.2x) and **Türkiye** rises from 1.8% to 4.8% (~2.7x) — now that Arab is its own auditable category rather than scattered across ~20 individually-tiny countries, its elevation is visibly comparable in magnitude to the already-known Türkiye shift, not a footnote to it. Separating Other from Unknown also clarifies the picture: **Unknown** (missing prediction) is a modest 7.7% of all social reviews and rises only slightly for racialised=yes (7.7% -> 11.1%), while **Other** (a real, non-top-10, non-Arab country) is the largest single category after Germany in both groups (23.2% -> 25.5%) — so a conflated "Other/unknown" label would have mostly been describing real long-tail country signal, not missing data. We deliberately avoid interpreting this as "who racialised discourse targets" or "who writes it" — the ethnicity signal is a name-origin prediction with known collision issues, not a per-review reviewer identity — but the magnitude of the Arab and Türkiye shifts specifically looks like it's worth a closer, qualitative look in the dimension-level work already planned (`approach`/`context` labels), rather than stopping at this aggregate association. One methodological note already put to use above: computing the top-10 cut from the full social population instead of the small racialised-only subset is not just a stability preference — it changes the actual category list (e.g. `India`, which edges out `Brazil` in the 7,768-row racialised subset by a handful of reviews, drops out of the top-10 once ranked against the full 504,072-row population, where `Brazil` is clearly the more common name-origin) — reusing the subset-derived ranking later would have made the two grains of this facet inconsistent with each other.


### Data-quality check: reviews with a missing `org_review_lang`

Before the language facet later in this section, one loose end is worth checking: `org_review_lang` is null for a non-trivial slice of the corpus. The likely explanation is mechanical, not missing data -- Google Maps only populates `trans_review` (and, presumably, its language detector) when it detects the review is written in a language *different* from the display locale used at scrape time. If a review is already in English (the scrape's apparent display locale) there may be nothing to detect or translate, leaving `org_review_lang` null even though the review plainly has a language. We check this directly before relying on `org_review_lang` for anything below, rather than silently letting `pd.crosstab`/`value_counts` drop these rows later (which is what was happening implicitly until now).


In [ ]:
null_mask = corpus["org_review_lang"].isnull()
print(f"corpus rows with null org_review_lang: {null_mask.sum():,} / {len(corpus):,} ({null_mask.mean():.2%})")

# if org_review_lang is null because "nothing needed translating," org_review and trans_review
# should be identical -- check that directly rather than assuming it
same_text = corpus.loc[null_mask, "org_review"] == corpus.loc[null_mask, "trans_review"]
print(f"org_review == trans_review among null-lang rows: {same_text.sum():,} / {null_mask.sum():,} ({same_text.mean():.1%})")

# spot-check the text itself: a simple English-vs-German marker-word scan (not a real
# language detector, just a sanity check) confirms these are English writing, not a missing
# detection on some other language
import re
def has_marker(s, words):
    return any(re.search(rf"\b{w}\b", s, re.IGNORECASE) for w in words)
sample_text = corpus.loc[null_mask, "org_review"].dropna()
german_hit = sample_text.apply(lambda s: has_marker(s, ["der", "die", "das", "und", "sehr", "nicht", "ist"]))
english_hit = sample_text.apply(lambda s: has_marker(s, ["the", "and", "very", "not", "is", "we", "are", "good", "great"]))
print(f"share with a German marker word: {german_hit.mean():.1%}")
print(f"share with an English marker word: {english_hit.mean():.1%} (many of the rest are single-word reviews like \"Top\"/\"Super\")")

print(f"\nsocial=yes reviews affected: {(social_df['org_review_lang'].isnull()).sum():,}")
print(f"racialised=yes reviews affected: {(racialised_df['org_review_lang'].isnull()).sum():,} "
      "(these were previously silently dropped from the ethnicity x language crosstab below, "
      "since pd.crosstab excludes NaN by default)")

# --- correction: null org_review_lang -> "English" ---
corpus["org_review_lang"] = corpus["org_review_lang"].fillna("English")
social_df["org_review_lang"] = corpus.loc[social_df.index, "org_review_lang"]
racialised_df["org_review_lang"] = corpus.loc[racialised_df.index, "org_review_lang"]

print(f"\nnull org_review_lang after correction: {corpus['org_review_lang'].isnull().sum()}")
print(f"'English' now recorded for {(corpus['org_review_lang']=='English').sum():,} corpus-wide, "
      f"{(social_df['org_review_lang']=='English').sum():,} social=yes, "
      f"{(racialised_df['org_review_lang']=='English').sum():,} racialised=yes")


**Takeaway.** The hypothesis holds up completely: **100% of the 65,908 null-`org_review_lang` rows (5.21% of the corpus) have `org_review == trans_review`** -- i.e. Google Maps never ran a translation on them, consistent with them already being in the scrape's display language. The marker-word scan confirms it's specifically English text, not a missing detection on some other language: essentially no German markers (0.2%) vs. a clear English signature (48.1% contain an unambiguous English marker word; most of the remainder are short single-word reviews like "Top"/"Super"/"Must have" that are consistent with English but too short to trigger a multi-word regex). We correct this by filling `org_review_lang` with `"English"` wherever it was null, across `corpus`, `social_df`, and `racialised_df`.

This isn't just a cosmetic fix. It recovers **710 social=yes reviews and 17 racialised=yes reviews** that previously had no language recorded at all -- and since `pd.crosstab` (used throughout the language facet below) silently drops rows with a NaN in either dimension, those 17 racialised reviews weren't just uncategorized, they were **entirely absent** from the ethnicity x language contingency table before this fix, without any visible warning. English now appears as its own (small) category throughout the language facet below.

### Within-group racialised rate: is a review from a given ethnicity group more likely to be racialised at all?

The three-way comparison above answers "of racialised reviews, what share come from each
group" -- a composition question, sensitive to how big each group is in the corpus to begin
with. The complementary question is the individual-level analogue of the constituency-level
ethnicity-share correlation found later in this notebook (Bringing WHO and WHERE together):
**taking each `ethnicity_final` group's own total reviews as the denominator, what share of
*those* reviews are social=yes, and, of the social ones, what share are racialised=yes?** This
holds group size fixed and asks about the group's own base rate directly, rather than its share
of an already-rare event.

In [ ]:
group_stats = corpus.groupby("ethnicity_final").agg(
    total_reviews=("review_id", "size"),
    social_reviews=("social", lambda s: (s == "yes").sum()),
)
group_stats["pct_social"] = group_stats["social_reviews"] / group_stats["total_reviews"] * 100

social_only = corpus.loc[corpus["social"] == "yes"]
rac_by_group = social_only.groupby("ethnicity_final").agg(
    racialised_reviews=("racialised", lambda s: (s == "yes").sum()),
)
group_stats = group_stats.join(rac_by_group)

rates = group_stats.apply(
    lambda r: pd.Series(wilson(r["racialised_reviews"], r["social_reviews"]),
                         index=["racialised_rate", "ci_lo", "ci_hi"]),
    axis=1,
)
group_stats = group_stats.join(rates)
for col in ["racialised_rate", "ci_lo", "ci_hi"]:
    group_stats[col] = group_stats[col] * 100
group_stats = group_stats.rename(columns={"racialised_rate": "racialised_rate_pct"})

overall_rate_pct = (corpus["racialised"] == "yes").sum() / (corpus["social"] == "yes").sum() * 100
group_stats = group_stats.sort_values("racialised_rate_pct", ascending=False)

print(f"baseline (all reviewers combined) racialised rate: {overall_rate_pct:.3f}%\n")
print(group_stats[["total_reviews", "social_reviews", "pct_social",
                    "racialised_reviews", "racialised_rate_pct", "ci_lo", "ci_hi"]].round(3))

fig, ax = plt.subplots(figsize=(8, 5))
colors = np.where(group_stats["racialised_rate_pct"] > overall_rate_pct, "#C44E52", "#4C72B0")
y = np.arange(len(group_stats))
ax.barh(y, group_stats["racialised_rate_pct"], color="#4C72B0",
        xerr=[group_stats["racialised_rate_pct"] - group_stats["ci_lo"],
              group_stats["ci_hi"] - group_stats["racialised_rate_pct"]])
ax.set_yticks(y)
ax.set_yticklabels(group_stats.index)
ax.axvline(overall_rate_pct, color="black", linestyle="--", linewidth=1,
           label=f"overall rate ({overall_rate_pct:.2f}%)")
ax.set_xlabel("racialised rate within this group's own social reviews (%, 95% Wilson CI)")
# ax.set_title("Within-group racialised rate, by ethnicity_final")
ax.invert_yaxis()
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "within_group_racialised_rate.png", dpi=150, bbox_inches="tight")
plt.show()

**Takeaway -- yes, there is a real in-group effect, and it lines up exactly with the composition finding above, not just as an artifact of group size.** Against a baseline racialised rate of 1.541% (all reviewers pooled), **`Arab`-name-origin reviewers have their own reviews flagged racialised at 4.78%** (95% CI 4.38-5.23%) and **`Türkiye` at 4.06%** (3.68-4.49%) -- both roughly 3-4x the baseline, with confidence intervals nowhere near overlapping it. `Germany` sits at the *opposite* end, the single lowest group at **1.08%** (1.03-1.12%), clearly below baseline. Everything else in between forms a gradient rather than a sharp cutoff: `France` (2.44%), `Indonesia` (2.22%), `Unknown` (2.20%) sit moderately above baseline; `Poland`, `Russian Federation`, `Other`, `Philippines`, `Italy` cluster just above or around it (1.49-1.78%); `Brazil` and `United States` sit at or slightly below it (1.27-1.29%).

This directly answers the question the composition view alone couldn't: **it is not simply that `Arab`/`Türkiye` are large enough groups to dominate racialised-review composition -- a review actually written under an `Arab` or `Türkiye` name-origin is intrinsically several times more likely to be racialised than one written under a `Germany` name-origin**, holding each group's own size fixed. The ordering here (`Arab` > `Türkiye` > ... > `Germany` lowest) exactly matches the composition shift found in the three-way table above, which is reassuring: two different ways of asking the question (share of racialised reviews vs. own-group base rate) land on the same story, not two different ones.

### Top languages: entire corpus vs. social vs. racialised reviews

The three-way ethnicity comparison above raises the natural follow-up: does *review language* shift the same way across the same three grains, and is any shift statistically significant? We apply the same method as the ethnicity three-way table -- fit a top-10-language cut once (on `social == 'yes'`, for the same stability reason as before), then apply that fixed category list to all three grains rather than refitting per grain.


In [ ]:
N_TOP_LANG = 10
top_langs = social_df["org_review_lang"].value_counts().head(N_TOP_LANG).index.tolist()
print("top 10 languages (fit on social=yes):", top_langs)

def apply_lang_grouped(series, top_langs):
    return np.where(series.isin(top_langs), series, "Other language")

corpus["lang_grouped_all"] = apply_lang_grouped(corpus["org_review_lang"], top_langs)
social_df["lang_grouped_all"] = apply_lang_grouped(social_df["org_review_lang"], top_langs)
racialised_df["lang_grouped_all"] = apply_lang_grouped(racialised_df["org_review_lang"], top_langs)

three_way_lang = pd.DataFrame({
    "entire_corpus_pct": pd.Series(corpus["lang_grouped_all"]).value_counts(normalize=True) * 100,
    "social_pct": pd.Series(social_df["lang_grouped_all"]).value_counts(normalize=True) * 100,
    "racialised_pct": pd.Series(racialised_df["lang_grouped_all"]).value_counts(normalize=True) * 100,
}).fillna(0).sort_values("racialised_pct", ascending=False)

print(f"\nentire corpus: N={len(corpus):,}  |  social=yes: N={len(social_df):,}  |  racialised=yes: N={len(racialised_df):,}\n")
print(three_way_lang.round(2))

three_way_lang.plot(kind="barh", figsize=(8, 6))
plt.xlabel("% of reviews in that population")
plt.title("Review language composition: entire corpus vs. social=yes vs. racialised=yes")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig(FIGURES_DIR / "language_composition_corpus_social_racialised.png", dpi=150, bbox_inches="tight")
plt.show()

# significance: language x racialised(yes/no), within social=yes -- same design as the ethnicity test above
ct = pd.crosstab(social_df["lang_grouped_all"], social_df["racialised"])
chi2, p, dof, _ = stats.chi2_contingency(ct)
n = ct.values.sum()
cv = np.sqrt(chi2 / (n * (min(ct.shape) - 1)))
print(f"\nlanguage x racialised(yes/no), within social=yes: chi2={chi2:.1f}, dof={dof}, p={p:.2e}, Cramer's V={cv:.4f}")


**Takeaway.** Language behaves differently from ethnicity across these three grains -- worth flagging directly since it's the opposite of what the ethnicity three-way table showed. Entire-corpus and social=yes are **not** near-identical here: `German` rises from 88.58% (entire) to 95.19% (social) and `Other language` (which absorbs `English` -- not in the top-10 fit on social=yes, since it's overwhelmingly a *non-social* review language, per the earlier data-quality fix) drops from 6.44% to 0.83%. This makes sense mechanically: many non-social reviews are short, generic, non-German snippets ("Nice", "Clean", "Good") that get filtered out once you require a review to be about the venue's social experience specifically.

The social-to-racialised shift reproduces the ethnicity-facet finding from a different angle: `German` drops from 95.19% to 90.69% (-4.5pp), while `French` rises 0.91%->3.14% (~3.4x), `Arabic` 0.23%->0.68% (~3x), `Czech` 0.22%->0.62% (~2.8x), and `Polish` 0.54%->0.88% (~1.6x) -- the same France/Arab/Czech/Polish elevation already documented in the ethnicity x language cross-tab, now shown as a direct three-way comparison. The shift is **statistically significant** (language x racialised, within social=yes: chi2=645.4, dof=10, p=3.3e-132) but with a small effect size (Cramér's V=0.036) -- the now-familiar "large N makes a real but modest shift easily detectable" pattern seen throughout this notebook.

### Ethnicity signal x review language

If `ethnicity_country_name` reflects genuine reviewer-identity signal, we might expect some
correspondence with the language a review is actually written in — e.g. a Turkish name-origin
reviewer writing disproportionately more often in Turkish than the corpus baseline. We test this
directly: cross-tabulate the final ethnicity category (built above) against `org_review_lang`, and quantify
the association with Cramér's V (via a chi-square test of independence).

`org_review_lang` has 37 distinct values, most vanishingly rare (`review_id`-level review-language
tags on 1-7 reviews each). A contingency table with that many sparse columns is uninterpretable, so
we collapse any language with fewer than 10 reviews in this subset into `"Other language"` --
the same "don't force detail the data can't support" logic as the Other/Unknown split above.

In [ ]:
df = racialised_df.copy()  # ethnicity_final already attached via the three-way comparison above

LANG_MIN_N = 10

lang_counts = df["org_review_lang"].value_counts()
print("org_review_lang counts:")
print(lang_counts)

kept_langs = lang_counts[lang_counts >= LANG_MIN_N].index
df["lang_grouped"] = np.where(df["org_review_lang"].isin(kept_langs), df["org_review_lang"], "Other language")
print(f"\nlanguages kept individually (n >= {LANG_MIN_N}): {list(kept_langs)}")
print(f"\ngrouped language counts:")
print(df["lang_grouped"].value_counts())


In [ ]:
ct = pd.crosstab(df["ethnicity_final"], df["lang_grouped"])
print("ethnicity_final x lang_grouped contingency table:")
print(ct)

chi2, p, dof, _ = stats.chi2_contingency(ct)
n = ct.values.sum()
cramers_v = np.sqrt(chi2 / (n * (min(ct.shape) - 1)))
print(f"\nchi2 = {chi2:.1f}, dof = {dof}, p = {p:.2e}, Cramer's V = {cramers_v:.4f}")

# row-normalised: within each ethnicity category, what % of reviews are in each language
row_pct = ct.div(ct.sum(axis=1), axis=0)
print("\nrow-normalised (share of each ethnicity category's reviews, by language):")
print((row_pct * 100).round(1))


In [ ]:
row_order = [c for c in ct.sum(axis=1).sort_values(ascending=False).index]
row_pct_pct = row_pct.loc[row_order] * 100

fig, ax = plt.subplots(figsize=(11, 7))
sns.heatmap(
    row_pct_pct,
    annot=row_pct_pct.round(1).values,
    fmt=".1f",
    cmap="Greens",
    ax=ax,
    cbar_kws={"label": "row % (share of that ethnicity category's reviews)"},
)
ax.set_xlabel("review language (org_review_lang, grouped)")
ax.set_ylabel("ethnicity category (ethnicity_final)")
ax.set_title("Ethnicity signal x review language (% share of each ethnicity category's own reviews)")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "ethnicity_x_language_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()


**Takeaway.** This is *not* a mechanical artifact — the association is both statistically significant and, once judged against baseline rates rather than raw dominance, substantively real (Cramér's V = 0.240, a moderate effect on a 13x15 table; chi2 = 5349.8, dof = 168, p ≈ 0). German is the majority language in almost every row simply because this is a corpus of reviews on German pools, so raw row-dominance by German is expected and uninformative on its own — the meaningful comparison is each category's *own-language share relative to that language's corpus-wide baseline* (German 90.7%, French 3.1%, Russian 1.2%, Polish 0.9%, Arabic 0.7%, Turkish 0.3% of all 7,768 reviews). Against that baseline, several categories show a dramatic, genuine elevation: **France** reviews are 49.7% French-language (vs. 3.1% baseline, and nearly a 50/50 split with German rather than German-dominant), **Poland** reviews are 28.8% Polish (vs. 0.9%, ~33x), and **Russian Federation** reviews are 25.5% Russian (vs. 1.2%, ~22x) — these three categories alone are doing most of the work behind the Cramér's V. **Arab** (8.4% Arabic vs. 0.7% baseline, ~12x) and **Türkiye** (3.5% Turkish vs. 0.3%, ~13x) show the same direction of effect, smaller in absolute share but still a meaningful multiple of baseline. By contrast, categories with no obvious home language in Europe — Brazil, Indonesia, Philippines, United States, Unknown, and the residual Other — sit at 83-99% German, essentially indistinguishable from the corpus baseline, i.e. no detectable own-language elevation there (there is no single "Portuguese" or "Indonesian" tag with enough reviews in this subset to register, so this is partly a construct-availability limit, not necessarily evidence of no signal). **Read honestly: this looks like genuine identity signal for the European/Slavic/Turkish/Arab categories where a corresponding language actually exists in `org_review_lang`, concentrated in a handful of categories rather than uniform across all of them** — not a broad mechanical artifact, but also not a story that holds evenly across every category in the table.

**English, now that it's been recovered.** The data-quality fix earlier in this section (correcting the null-`org_review_lang` rows that were actually English but had never been detected as any language) adds English as a real, if small, category here: 17 of the 7,768 racialised reviews (0.2%). It doesn't change the story above -- 0.2% is nowhere near enough to move the Cramér's V or challenge any of the category-level findings -- but it matters for correctness: before the fix, these 17 rows were not just uncategorized, `pd.crosstab` silently dropped them from this entire table, so the contingency table above was quietly built on 7,751 rows instead of the full 7,768.


## How does racialised discourse differ between German and non-German name-origin reviewers?

**Key question:** does *who* the racialised review's name-origin signal points to relate to *what* the review actually says? We split the racialised=yes subset into **German** (`ethnicity_country_name == "Germany"`, the largest single category throughout this section) vs. **Non-German** (any other known name-origin -- Arab League members, Türkiye, the US, etc., pooled together) vs. **Unknown** (missing prediction, excluded from the comparison below since we can't classify what we don't know). This is a coarser cut than the full `ethnicity_final` scheme above -- deliberately so, since the four content dimensions (`theme`/`context`/`connotation`/`approach`) are already thin at N=7,768, and splitting further than a binary German/non-German cut would leave individual dimension labels (e.g. `Security`, a small `theme` category) too sparse per cell to read.

We look at this two ways: **individually** (does each of the four dimensions, on its own, differ by German-status?), then **collectively** (specific label *combinations* across dimensions -- e.g. `theme == Clothing` reviews framed with `approach == Explicit` vs. `Coded` -- since a real pattern can be invisible in each dimension's marginal distribution and only show up once you condition on a second dimension).


In [ ]:
german_mask = df["ethnicity_country_name"] == "Germany"
unknown_mask = df["ethnicity_country_name"].isna()
non_german_mask = ~german_mask & ~unknown_mask

df["german_status"] = np.select(
    [german_mask, non_german_mask, unknown_mask],
    ["German", "Non-German", "Unknown"],
    default="Unknown",
)
print("german_status counts (of all 7,768 racialised reviews):")
print(df["german_status"].value_counts())
print((df["german_status"].value_counts(normalize=True) * 100).round(1))

# Unknown (missing ethnicity_country_name prediction) can't be classified German/non-German --
# excluded from every comparison below, the same logic as excluding it from a country-level
# comparison in the ethnicity subsections above.
known = df[df["german_status"] != "Unknown"].copy()
baseline_german_pct = (known["german_status"] == "German").mean() * 100
print(f"\nknown-ethnicity racialised reviews used below: {len(known):,} / {len(df):,}")
print(f"baseline German share of known-ethnicity racialised reviews: {baseline_german_pct:.1f}%")


### Individually: does each dimension differ by German-status on its own?

For each dimension, we explode multi-label reviews (same approach as the theme x connotation check in the dimension-content-overview section below), cross-tabulate against `german_status`, and compare the **column-normalised %** (share *within* German reviews vs. share *within* non-German reviews) rather than raw counts, since the two groups are unequal in size (2,146 vs. 4,762). Cramér's V lets us compare effect size across the four dimensions directly, on the same 0-1 scale.


In [ ]:
DIMENSIONS = ["approach", "theme", "connotation", "context"]

dim_results = {}
for dim in DIMENSIONS:
    exploded = known[[dim, "german_status"]].explode(dim)
    ct = pd.crosstab(exploded[dim].to_numpy(), exploded["german_status"].to_numpy())
    chi2, p, dof_, _ = stats.chi2_contingency(ct)
    n = ct.values.sum()
    cv = np.sqrt(chi2 / (n * (min(ct.shape) - 1)))
    col_pct = ct.div(ct.sum(axis=0), axis=1) * 100
    dim_results[dim] = {"ct": ct, "col_pct": col_pct, "chi2": chi2, "p": p, "cramers_v": cv}
    gap = (col_pct["German"] - col_pct["Non-German"]).sort_values(key=abs, ascending=False)
    print(f"\n--- {dim}: chi2={chi2:.1f}, p={p:.2e}, Cramer's V={cv:.3f} ---")
    print(col_pct.round(1))
    print("biggest German - Non-German percentage-point gaps:")
    print(gap.round(1))

print("\n" + "=" * 60)
print("Cramer's V by dimension, largest effect first:")
print(pd.Series({d: r["cramers_v"] for d, r in dim_results.items()}).sort_values(ascending=False).round(3))


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 9))
for ax, dim in zip(axes.flat, DIMENSIONS):
    col_pct, cv, p = dim_results[dim]["col_pct"], dim_results[dim]["cramers_v"], dim_results[dim]["p"]
    gap = (col_pct["German"] - col_pct["Non-German"]).sort_values()
    colors = np.where(gap > 0, "#4C72B0", "#C44E52")
    ax.barh(gap.index.astype(str), gap.values, color=colors)
    ax.axvline(0, color="black", linewidth=0.8)
    ax.set_title(f"{dim} (Cramer's V={cv:.3f}, p={p:.1e})")
    ax.set_xlabel("German % - Non-German % (pct points)\nblue = more German, red = more Non-German")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "german_vs_nongerman_gap_by_dimension.png", dpi=150, bbox_inches="tight")
plt.show()


**Takeaway.** Of the 7,768 racialised reviews, **2,146 (27.6%) have a German name-origin signal, 4,762 (61.3%) a non-German one, and 860 (11.1%) are Unknown** (missing prediction, excluded below) -- giving a baseline German share of **31.1%** among the 6,908 reviews with a known origin.

`theme` is doing almost all of the work here. Cramér's V by dimension, largest first: **theme = 0.180**, context = 0.085, approach = 0.050, connotation = 0.032 -- all four clear p<.05 (unsurprising at N=6,908), but effect size makes clear this is really a `theme`/`context` story, not a four-way one. The direction is coherent across both: **German-authored racialised reviews skew toward `Crowd`** (32.6% vs. 23.7% of non-German, +8.9pp) **and `Etiquette`** (21.5% vs. 14.9%, +6.6pp) **themes, framed `generalised`** rather than incident-specific (72.9% vs. 64.2%). **Non-German-authored racialised reviews skew toward `Staff`** (17.3% vs. 9.5% German, i.e. -7.9pp) **and `Access`** (6.2% vs. 2.1%, -4.0pp) **themes, framed as a specific `incident`** more often (35.8% vs. 27.1%). `connotation` is nearly flat (86.8% vs. 87.0% Negative) -- both groups are overwhelmingly negative regardless of name-origin signal, so ethnicity doesn't track *how negative* the discourse is, only *what it's about* and *how concretely it's framed*.

Read plainly: German-origin racialised reviews lean toward broad, situational complaints ("too crowded", "people not following the rules") described in general terms, while non-German-origin racialised reviews lean toward complaints about a specific staff interaction or being restricted/denied access, described as a specific incident. That asymmetry -- generalised/environmental vs. incident/interpersonal -- is a more specific and more interesting story than "both groups write similarly negative reviews," and is worth carrying into the qualitative RQ3 work.

### Word clouds: what do German vs. non-German racialised reviews actually say?

The chi-square/Cramér's V comparisons above show *that* German-status shifts which dimension label a review carries; word clouds make *what's actually written* directly visible. Built on `trans_review` (the English machine translation, populated for every review regardless of original language) rather than `org_review` -- Non-German-origin reviewers write in many different source languages, so translating everything into one language first is what makes a single English stopword list, and a side-by-side comparison, meaningful at all. This is a discovery aid, not a formal test: word frequency says nothing about dimension-label accuracy or sentiment on its own.

Two layers, both restricted to `known` (the 6,908 racialised reviews with a resolved German/Non-German status): first, one German vs. one Non-German cloud pooling every racialised review regardless of dimension label; then, per dimension, one word-cloud pair per label that clears an `n>=20`-per-group floor (same thin-cell convention used elsewhere in this section) -- i.e. exactly the "German cloud for `generalised`, German cloud for `incident`, then the Non-German equivalents" structure requested for `context`, repeated for `theme`/`connotation`/`approach`. `pool`/`pools`/`swimming` are added to the library's standard English stopword list -- they appear in nearly every review regardless of group and carry no discriminating signal for this specific comparison. Each panel's top-15 words are also printed (pulled directly from the same frequencies the cloud renders), so the takeaway below can cite concrete words rather than relying on eyeballing an image.

In [ ]:
from wordcloud import WordCloud, STOPWORDS

# beyond the standard English list: venue-generic words (sauna/staff/water/lifeguard -- the
# swimming-pool-review equivalent of "pool"/"swimming" themselves) and vague filler/connector
# words (people/even/really/nice/day/simply/...) that showed up at 8-27% document frequency
# across ALL racialised reviews regardless of group or theme (checked directly against the
# corpus rather than guessed) -- dominating every cloud's top slots without discriminating
# anything, and specifically crowding out the concrete, on-topic words (hygiene/dirty/clean/
# unhygienic/racist/burkini/friendly/foreigners/...) that are actually worth reading.
GENERIC_FILLER = {
    "pool", "pools", "swimming", "sauna", "people", "even", "unfortunately", "area", "one", "go",
    "many", "time", "staff", "water", "lifeguard", "lifeguards", "really", "good", "great", "nice",
    "never", "place", "two", "completely", "back", "day", "simply", "always", "around", "want",
    "everything", "thing", "things", "guest", "guests",
    "price", "year", "well", "thermal", "u", "food", "bath", "lounger", "lot",
}
WC_STOPWORDS = STOPWORDS | GENERIC_FILLER

def make_wordcloud(texts, max_words=80):
    text = " ".join(t for t in texts.dropna() if isinstance(t, str))
    if not text.strip():
        return None
    return WordCloud(width=900, height=550, background_color="white", stopwords=WC_STOPWORDS,
                      collocations=False, max_words=max_words, random_state=0).generate(text)

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
for ax, grp in zip(axes, ["German", "Non-German"]):
    texts = known.loc[known["german_status"] == grp, "trans_review"]
    wc = make_wordcloud(texts)
    ax.imshow(wc, interpolation="bilinear")
    ax.axis("off")
    ax.set_title(f"{grp} (n={len(texts):,})", fontsize=13)
    print(f"{grp} top 15 words: {list(wc.words_.keys())[:15]}")
plt.suptitle("All racialised reviews, pooled across every dimension -- German vs. Non-German", y=1.02)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "wordcloud_overall_german_vs_nongerman.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
MIN_WC_N = 20  # same thin-cell floor used elsewhere in this section (e.g. the combination scan below)

for dim in DIMENSIONS:
    exploded = known[[dim, "german_status", "trans_review"]].explode(dim)
    label_counts = exploded.groupby([dim, "german_status"]).size().unstack(fill_value=0)
    valid_labels = label_counts[
        (label_counts.get("German", 0) >= MIN_WC_N) & (label_counts.get("Non-German", 0) >= MIN_WC_N)
    ].index.tolist()
    skipped = [l for l in label_counts.index if l not in valid_labels]
    if skipped:
        print(f"{dim}: skipping {skipped} (n<{MIN_WC_N} in at least one group)")

    fig, axes = plt.subplots(2, len(valid_labels), figsize=(5 * len(valid_labels), 8), squeeze=False)
    for j, lab in enumerate(valid_labels):
        for i, grp in enumerate(["German", "Non-German"]):
            texts = exploded.loc[(exploded[dim] == lab) & (exploded["german_status"] == grp), "trans_review"]
            wc = make_wordcloud(texts)
            ax = axes[i, j]
            ax.imshow(wc, interpolation="bilinear")
            ax.axis("off")
            ax.set_title(f"{grp} -- {lab} (n={len(texts)})", fontsize=10)
            top_words = list(wc.words_.keys())[:10] if wc else []
            print(f"{dim} / {lab} / {grp} (n={len(texts)}) top 10: {top_words}")
    plt.suptitle(f"{dim}: German vs. Non-German word clouds by label", y=1.02, fontsize=14)
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / f"wordcloud_{dim}_by_label_german_vs_nongerman.png", dpi=150, bbox_inches="tight")
    plt.show()


**Takeaway -- with generic filler stripped out, the real split concentrates in direct accusations of racism/unfriendliness in specific labels; several apparent differences from the first pass (burkini, hygiene vocabulary) turn out to be shared once the ranking isn't crowded out by "people"/"staff"/"water"/"sauna".**

Overall pooled top-15s now diverge more visibly: German leans toward facility/amenity vocabulary (`room`, `price`, `thermal`, `bath`, `lounger`, `food`), while Non-German surfaces `racist`, `foreigner`, `unfriendly`, `allowed` directly in its overall top-15 -- words that never cracked the German list at all. Three places this sharpens further:

- **`approach=Implicit`**: `racist` is the single most common word in Non-German-authored reviews (n=762) -- and doesn't appear anywhere in German Implicit's top-10 (n=319). The cleanest split in the whole scan.
- **`theme=Access`**: `racist` again appears only in Non-German (n=518), not in German (n=81, thin).
- **`approach=Coded`**: Non-German surfaces `Talahon` -- a recent German slang term for young men of migrant background, itself a "coded" way of referencing ethnicity -- fitting neatly with what this label is actually supposed to capture. German Coded instead surfaces `clientele`/`beautiful`, a softer register.
- **`theme=Staff`**: both groups now show `racist` *and* `unfriendly` in their top-10 (German n=360, Non-German n=1,454) -- shared vocabulary, not exclusive -- but Non-German additionally adds `rude`/`behavior`, a harsher register layered on top of what's shared.

Two corrections to the first pass, now that filler isn't crowding the ranking: **`burkini` appears in both groups' `theme=Clothing` top-10** (German rank 4, Non-German rank 1) -- not Non-German-exclusive as it first looked. And **`theme=Hygiene` now shows `hygiene`/`dirty`/`disgusting` directly in German's own top-10** (not just present-but-buried, as the raw substring check below already found) -- consistent with, not contradicting, the drill-down takeaway below. `connotation=Positive` likewise shares `clean`/`friendly` across both groups. Net read: the generic-filler cut doesn't invent new group differences so much as it removes false-looking ones and makes the two or three real ones (Implicit approach, Access theme, Coded's specific slang) much easier to see.

### Zooming into `Hygiene`: is `unhygienic` a broad Non-German pattern, or driven by one specific group?

`Non-German` above pools Arab, Türkiye, Europe, other/long-tail countries, and Other into one bucket -- `unhygienic` showing up only there could mean "every non-German group says this" or "one specific group says this, and the others just get carried along in the pooled bucket." We re-split `theme=Hygiene` into **Germany**, **Türkiye**, **Arab**, **Other** (every remaining known nationality -- Europe/US/etc. folded in here, since the question is specifically Germany/Türkiye/Arab vs. everyone else, not a fourth pole), and **Unknown** (missing name-origin prediction, excluded from the German/Non-German cut above but included here since it's its own group now, not a excluded-from-comparison bucket).

In [ ]:
def to_group5(ethnicity_final_value):
    if ethnicity_final_value in ("Germany", "Arab", "Türkiye", "Unknown"):
        return ethnicity_final_value
    return "Other"

df["ethnicity_group5"] = df["ethnicity_final"].map(to_group5)

hygiene = df[df["theme"].apply(lambda t: "Hygiene" in t)].copy()
print(f"Hygiene-themed racialised reviews: {len(hygiene)}")
print(hygiene["ethnicity_group5"].value_counts())

GROUP5_ORDER = ["Germany", "Türkiye", "Arab", "Other", "Unknown"]
fig, axes = plt.subplots(1, len(GROUP5_ORDER), figsize=(5 * len(GROUP5_ORDER), 6))
for ax, grp in zip(axes, GROUP5_ORDER):
    texts = hygiene.loc[hygiene["ethnicity_group5"] == grp, "trans_review"]
    wc = make_wordcloud(texts)
    ax.axis("off")
    if wc is None:
        ax.set_title(f"{grp} (n={len(texts)})\nno text", fontsize=10)
        continue
    ax.imshow(wc, interpolation="bilinear")
    ax.set_title(f"{grp} (n={len(texts)})", fontsize=11)
    print(f"Hygiene / {grp} (n={len(texts)}) top 10: {list(wc.words_.keys())[:10]}")
plt.suptitle(f"theme=Hygiene word clouds by ethnicity_group5 (N={len(hygiene)})", y=1.03, fontsize=14)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "wordcloud_hygiene_by_ethnicity_group5.png", dpi=150, bbox_inches="tight")
plt.show()

# does "unhygienic"/"hygiene"/"dirty" actually appear in each group's raw text, and how often?
DIRT_WORDS = ["unhygienic", "hygiene", "dirty", "clean", "disgusting"]
for grp in GROUP5_ORDER:
    texts = hygiene.loc[hygiene["ethnicity_group5"] == grp, "trans_review"].str.lower()
    n = len(texts)
    hits = {w: texts.str.contains(w, regex=False).sum() for w in DIRT_WORDS}
    print(f"{grp:10s} (n={n:3d}): " + ", ".join(f"{w}={c}({c/n:.0%})" for w, c in hits.items()))


**Takeaway -- with filler removed, `hygiene`/`dirty`/`disgusting` now show up directly in German's own top-10 too, reinforcing rather than reversing the earlier correction: the "only Non-German" read was a ranking artifact, and raw usage rates are broadly similar across groups.**

Germany's `theme=Hygiene` top-10 (n=160) is now `children, hygiene, wearing, room, women, dirty, disgusting, clothes, outdoor, fully` -- `hygiene`, `dirty`, and `disgusting` all present, matching the raw substring rates below (17-23% of German Hygiene reviews use each). `Other` (n=272, the largest group) and `Unknown` (n=71) both lead with `hygiene`/`unhygienic` too. The two thin groups add their own flavour but shouldn't be over-read given the N: **Türkiye** (n=16) now surfaces `racist` in its own top-10 alongside `unhygienic`/`dirty`, and **Arab** (n=10) surfaces `foreigner`/`clientele`/`bad` -- interesting as individual data points, but each is 1-3 actual reviews at this size, not a stable group pattern. **Bottom line unchanged from before**: this is not an Arab- or Türkiye-specific pattern; it's near-universal vocabulary across every ethnicity group once you look at usage rate rather than top-10 ranking alone, and the ranking itself was sensitive to exactly the kind of generic filler word just stripped out.

### Zooming into `approach=Coded`: which group actually drives `Talahon`?

`Talahon` showed up only in the `Non-German` word cloud for `approach=Coded` -- but `Non-German` pools Arab, Türkiye, Europe, and every other nationality into one bucket, so that alone doesn't say *who* is actually using the term. Same five-way split as the `Hygiene` drill-down above (`ethnicity_group5` -- Germany, Türkiye, Arab, Other, Unknown), reused unchanged, now applied to `approach=Coded`.

In [ ]:
coded = df[df["approach"].apply(lambda a: "Coded" in a)].copy()
print(f"Coded-approach racialised reviews: {len(coded)}")
print(coded["ethnicity_group5"].value_counts())

fig, axes = plt.subplots(1, len(GROUP5_ORDER), figsize=(5 * len(GROUP5_ORDER), 6))
for ax, grp in zip(axes, GROUP5_ORDER):
    texts = coded.loc[coded["ethnicity_group5"] == grp, "trans_review"]
    wc = make_wordcloud(texts)
    ax.axis("off")
    if wc is None:
        ax.set_title(f"{grp} (n={len(texts)})\nno text", fontsize=10)
        continue
    ax.imshow(wc, interpolation="bilinear")
    ax.set_title(f"{grp} (n={len(texts)})", fontsize=11)
    print(f"Coded / {grp} (n={len(texts)}) top 10: {list(wc.words_.keys())[:10]}")
plt.suptitle(f"approach=Coded word clouds by ethnicity_group5 (N={len(coded)})", y=1.03, fontsize=14)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "wordcloud_coded_by_ethnicity_group5.png", dpi=150, bbox_inches="tight")
plt.show()

# how many reviews in each group actually contain "Talahon" (case-insensitive), raw count not just top-10 presence
for grp in GROUP5_ORDER:
    texts = coded.loc[coded["ethnicity_group5"] == grp, "trans_review"].str.lower()
    n = len(texts)
    hits = texts.str.contains("talahon", regex=False).sum()
    print(f"{grp:10s} (n={n:3d}): 'talahon' in {hits} reviews ({hits/n:.0%})" if n else f"{grp:10s} (n=0)")


**Takeaway -- `Talahon` isn't German-exclusive either: German-authored Coded reviews use it too, just at roughly half the rate of every other group, and the two thinnest groups (Türkiye, Arab) don't have enough reviews to say anything reliable about them specifically.**

Raw rate of `talahon` appearing anywhere in the review text, by group: **Germany 5% (12/251)**, Türkiye 11% (2/18), Arab 6% (1/18), **Other 12% (42/364)**, Unknown 11% (10/88). So the real pattern is a roughly 2x higher usage rate outside German-authored reviews across the board (6-12% vs. 5%) -- not a term that only appears in one group -- and by volume it's overwhelmingly the large `Other` bucket (42 of the 55 total non-German-status mentions) doing the work, the same shape as the `Hygiene`/`unhygienic` finding above. Türkiye's 11% and Arab's 6% rest on **2 and 1 review respectively** -- a single review moving the rate by 5-6 points -- so neither is trustworthy as a per-nationality claim at this N; `theme`/`approach` word clouds this thin (n=18) generally, e.g. Türkiye's own top-10 here (`police`, `pizza`, `old`, `woman`) look like scattered individual reviews rather than a coherent pattern. **Net read**: `Talahon` is a real, roughly 2x-elevated marker of non-German-authored Coded-approach discourse in aggregate, concentrated in `Other` by volume -- but "which specific non-German group drives it" isn't answerable from this data; Arab and Türkiye simply don't have enough Coded-labelled reviews (18 each) to support that claim.

### Collectively: do specific dimension *combinations* stand out?

A single dimension's marginal distribution can hide a sharper pattern that only appears once you condition on a second dimension -- e.g. German-status might look unremarkable for `approach` overall, yet be sharply skewed for one specific `theme`. We cross every pair of the four dimensions (6 pairs total) against `german_status`, keep only label combinations with at least 20 reviews (thinner cells are too noisy for a stable percentage), and rank every surviving combination by how far its German-share deviates from the `baseline_german_pct` computed above -- this is an exploratory scan for "what's interesting," not a pre-registered set of hypotheses, so we report both directions (most non-German-skewed and most German-skewed) rather than a single test.


In [ ]:
from itertools import combinations

MIN_COMBO_N = 20
records = []
for d1, d2 in combinations(DIMENSIONS, 2):
    exploded = known[[d1, d2, "german_status"]].explode(d1).explode(d2)
    grp = exploded.groupby([d1, d2])["german_status"].agg(
        n="size", german_pct=lambda s: (s == "German").mean() * 100
    )
    grp = grp[grp["n"] >= MIN_COMBO_N].reset_index()
    grp = grp.rename(columns={d1: "label_1", d2: "label_2"})
    grp["dim_pair"] = f"{d1} x {d2}"
    records.append(grp)

combo_df = pd.concat(records, ignore_index=True)
combo_df["deviation"] = combo_df["german_pct"] - baseline_german_pct

print(f"combinations tested (n>={MIN_COMBO_N} each) across all 6 dimension pairs: {len(combo_df)}\n")
print(f"Most non-German-skewed combinations (baseline = {baseline_german_pct:.1f}% German):")
print(combo_df.sort_values("german_pct").head(10)[["dim_pair", "label_1", "label_2", "n", "german_pct", "deviation"]].to_string(index=False))
print(f"\nMost German-skewed combinations:")
print(combo_df.sort_values("german_pct", ascending=False).head(10)[["dim_pair", "label_1", "label_2", "n", "german_pct", "deviation"]].to_string(index=False))


In [ ]:
# approach x theme stood out above as the richest pairing (Coded approach skews German across
# most themes) -- shown as a full heatmap rather than just a top-10 list, so the pattern (or lack
# of one) is visible across every theme at once, not just the extremes.
exploded_at = known[["approach", "theme", "german_status"]].explode("approach").explode("theme")
at_pct = exploded_at.groupby(["approach", "theme"])["german_status"].agg(
    n="size", german_pct=lambda s: (s == "German").mean() * 100
)
at_pct = at_pct[at_pct["n"] >= MIN_COMBO_N].reset_index()
pivot = at_pct.pivot(index="approach", columns="theme", values="german_pct")
n_pivot = at_pct.pivot(index="approach", columns="theme", values="n")

fig, ax = plt.subplots(figsize=(11, 4))
sns.heatmap(
    pivot, annot=True, fmt=".0f", cmap="RdBu_r", center=baseline_german_pct, ax=ax,
    cbar_kws={"label": "German % of reviews with this approach x theme combination"},
)
ax.set_title(f"German share by approach x theme (baseline={baseline_german_pct:.1f}%, blank = n<{MIN_COMBO_N})")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "german_share_by_approach_x_theme_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()

print("cell sizes (n):")
print(n_pivot)


**Takeaway.** 95 dimension-pair combinations cleared the `n>=20` bar across the 6 possible pairs. The clearest, most repeatable pattern in the scan: reviews using the **`Coded` approach skew German across nearly every theme they co-occur with** -- Clothing (44.7%), Hygiene (41.2%), Etiquette (39.6-40.9%), Belonging (40.4%), Crowd (40.1%), and Safety (37.6%) all sit well above the 31.1% baseline, visible as a near-solid warm row in the `approach x theme` heatmap above. `Access` and `Staff` are the clearest counter-pattern: both stay non-German-skewed **regardless of approach** (Access: Explicit 13.7%, Implicit 11.8%; Staff: Implicit 15.4%) -- consistent with `Access`/`Staff` already being the two most non-German-skewed themes on their own (individual-dimension subsection above), and confirming that's not just an `Explicit`-approach artifact.

The single most extreme cells: **Clothing reviews with a `Positive` connotation are the most non-German-skewed combination found** (10.6% German, n=47, -20.4pp from baseline), and `Access` clusters at the non-German extreme almost regardless of what else co-occurs with it (paired with `incident` context, `Negative` connotation, or `Implicit` approach, all land at 12-18% German). At the other extreme, `Coded` + `Neutral` connotation (46.4% German, n=28) and `Implicit` + `Hygiene` (45.5%, n=55) are the two most German-skewed combinations found.

**Caveat.** Several of the most extreme single combinations sit at n=25-60 -- thin enough that individual percentages are better read as leads for the qualitative RQ3 work than as stable estimates. The `approach x theme` pattern is the one result here with enough independently-repeated, same-direction cells (6 of 9 visible themes landing above baseline for `Coded`) to treat as a genuine pattern rather than a handful of noisy extremes -- worth flagging directly for the `theme`/`approach` qualitative read planned for RQ3.

### Do non-German reviewers use different languages for different labels?

Within German-origin reviews, language isn't a useful further split -- the ethnicity x language table earlier in this section already showed `Germany` reviews are 98.9% German-language. Non-German reviewers, though, genuinely split across many languages (French, Arabic, Turkish, Russian, Polish, German-as-a-second-language, etc.), so it's worth asking a sharper question than the German-status cut above: **does the specific language a non-German-origin reviewer writes in predict which dimension labels show up, beyond what their name-origin alone predicts?** We reuse the `lang_grouped` column already fit earlier in this section (same `LANG_MIN_N=10` cut, on the full racialised subset) rather than refitting it on the smaller Non-German-only slice, restricted here to the 4,762 Non-German rows of `known`.


In [ ]:
non_german = known[known["german_status"] == "Non-German"].copy()
print(f"Non-German known-ethnicity racialised reviews: {len(non_german):,}")
print("\nlang_grouped distribution within this subset (languages this thin should be read cautiously below):")
print(non_german["lang_grouped"].value_counts())


In [ ]:
lang_results = {}
for dim in DIMENSIONS:
    exploded = non_german[[dim, "lang_grouped"]].explode(dim)
    ct = pd.crosstab(exploded[dim].to_numpy(), exploded["lang_grouped"].to_numpy())
    chi2, p, dof_, _ = stats.chi2_contingency(ct)
    n = ct.values.sum()
    cv = np.sqrt(chi2 / (n * (min(ct.shape) - 1)))
    col_pct = ct.div(ct.sum(axis=0), axis=1) * 100
    lang_results[dim] = {"ct": ct, "col_pct": col_pct, "chi2": chi2, "p": p, "cramers_v": cv}
    print(f"\n--- {dim}: chi2={chi2:.1f}, dof={dof_}, p={p:.2e}, Cramer's V={cv:.3f} ---")

print("Cramer's V by dimension (language, within Non-German), largest first:")
print(pd.Series({d: r["cramers_v"] for d, r in lang_results.items()}).sort_values(ascending=False).round(3))
print("\n(for comparison, Cramer's V by dimension for German-status itself, from the subsection above): "
      f"theme={dim_results['theme']['cramers_v']:.3f}, context={dim_results['context']['cramers_v']:.3f}, "
      f"approach={dim_results['approach']['cramers_v']:.3f}, connotation={dim_results['connotation']['cramers_v']:.3f}")


In [ ]:
# connotation has the largest language effect above -- plotted directly, restricted to languages
# with a reasonably interpretable N (>=40 within this Non-German subset); thinner ones are listed
# but not plotted, consistent with this notebook's usual thin-cell caveat
conn_pct = lang_results["connotation"]["col_pct"]
lang_n = non_german["lang_grouped"].value_counts()
plot_langs = [l for l in conn_pct.columns if lang_n.get(l, 0) >= 40]
print(f"languages shown (n>=40): {plot_langs}")
print(f"languages excluded from the plot for thin N (<40): {[l for l in conn_pct.columns if l not in plot_langs]}")

neg_share = conn_pct.loc["Negative", plot_langs].sort_values()
overall_non_german_negative = (non_german["connotation"].explode() == "Negative").mean() * 100

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.barh(neg_share.index, neg_share.values, color="#C44E52")
ax.axvline(overall_non_german_negative, color="black", linestyle="--", linewidth=1,
           label=f"Non-German overall ({overall_non_german_negative:.1f}%)")
ax.set_xlabel("% Negative connotation")
ax.set_title("Negative-connotation share by review language (Non-German reviewers, n>=40 only)")
ax.legend()
plt.tight_layout()
plt.savefig(FIGURES_DIR / "negative_connotation_share_by_language_nongerman.png", dpi=150, bbox_inches="tight")
plt.show()


**Takeaway.** Language, within Non-German reviewers, tells a genuinely different story than name-origin does -- and in one dimension, a stronger one. Cramér's V by dimension for **language** (within the 4,762 Non-German reviews): **connotation = 0.183**, approach = 0.095, context = 0.073, theme = 0.067. Compare that to Cramér's V for **German-status itself** (subsection above): theme = 0.180, context = 0.085, approach = 0.050, connotation = 0.032. The ranking essentially flips: `connotation` -- the dimension name-origin barely touches -- is the dimension language explains best, while `theme` -- name-origin's strongest dimension -- is language's weakest.

The pattern itself is a clean, near-monotonic gradient in the chart above (restricted to the 8 languages with n>=40, out of 4,762): **German-language reviews are the most negative at 89.8%**, then French (80.1%), Dutch (72.7%), Arabic (70.0%), Polish (64.5%), Russian (62.5%), Other language (62.2%), down to **Czech at 56.8%, the least negative**. In other words, among Non-German-origin reviewers, *writing in German specifically* -- not just being a non-German name-origin reviewer generally -- goes with the harshest tone; writing in one's own likely home language goes with a milder one. This is a plausible, substantive lead (language-of-writing as a proxy for something like audience/integration, distinct from and stronger than name-origin for tone specifically), not just noise -- though it's a correlational read, not a causal one.

**Caveat.** Several languages sit right at or just above the `LANG_MIN_N=10` floor (Turkish n=19, Ukrainian n=19, Spanish n=17, English n=16, Italian n=14, Hungarian n=11, Romanian n=10) and were deliberately excluded from the chart for that reason -- e.g. Hungarian (41.7% Negative / 41.7% Neutral) and Ukrainian (36.8% Negative / 36.8% Positive) look like striking outliers in the printed table above, but at n=11 and n=19 these are single-digit-count differences, not stable estimates.

### Breaking "Non-German" into its top ethnicities

The German-status binary above averages together every non-German name-origin into one bucket -- worth checking whether that hides real differences between, say, `Arab`/`Türkiye` and `United States`/`France`. We repeat the same per-dimension test using the full `ethnicity_final` scheme already built earlier in this section (`Germany` + the top-9 individual countries + `Arab` + `Other`, excluding `Unknown`), rather than collapsing everything non-German into one group.


In [ ]:
known_eth = df[df["ethnicity_final"] != "Unknown"].copy()
print(f"known-ethnicity racialised reviews (ethnicity_final scheme): {len(known_eth):,}")
print(known_eth["ethnicity_final"].value_counts())


In [ ]:
eth_results = {}
for dim in DIMENSIONS:
    exploded = known_eth[[dim, "ethnicity_final"]].explode(dim)
    ct = pd.crosstab(exploded[dim].to_numpy(), exploded["ethnicity_final"].to_numpy())
    chi2, p, dof_, _ = stats.chi2_contingency(ct)
    n = ct.values.sum()
    cv = np.sqrt(chi2 / (n * (min(ct.shape) - 1)))
    col_pct = ct.div(ct.sum(axis=0), axis=1) * 100
    eth_results[dim] = {"ct": ct, "col_pct": col_pct, "chi2": chi2, "p": p, "cramers_v": cv}
    print(f"\n--- {dim}: chi2={chi2:.1f}, dof={dof_}, p={p:.2e}, Cramer's V={cv:.3f} ---")

print("\nCramer's V by dimension (full ethnicity_final scheme), largest first:")
print(pd.Series({d: r["cramers_v"] for d, r in eth_results.items()}).sort_values(ascending=False).round(3))


In [ ]:
# for every (dimension, ethnicity, label) cell with n>=20, how far does that ethnicity's share of
# that label deviate from the grand mean across all known-ethnicity racialised reviews? surfaces
# which specific ethnicities are driving each dimension's overall association, rather than just
# reporting an aggregate Cramer's V.
grand_mean = {dim: known_eth[[dim]].explode(dim)[dim].value_counts(normalize=True) * 100 for dim in DIMENSIONS}

records = []
for dim in DIMENSIONS:
    col_pct = eth_results[dim]["col_pct"]
    n_per_col = eth_results[dim]["ct"].sum(axis=0)
    for eth in col_pct.columns:
        if n_per_col[eth] < 20:
            continue
        for label in col_pct.index:
            records.append({
                "dim": dim, "ethnicity": eth, "label": label,
                "pct": col_pct.loc[label, eth], "grand_mean_pct": grand_mean[dim].get(label, 0),
                "dev": col_pct.loc[label, eth] - grand_mean[dim].get(label, 0), "n_eth": int(n_per_col[eth]),
            })
dev_df = pd.DataFrame(records)
dev_df = dev_df.reindex(dev_df["dev"].abs().sort_values(ascending=False).index)

print("biggest single (dimension, ethnicity, label) deviations from the grand mean, n>=20 only:")
print(dev_df.head(20).to_string(index=False))

# theme x ethnicity heatmap of the same deviation, for a full picture beyond the top-20 list
theme_pivot = eth_results["theme"]["col_pct"].sub(grand_mean["theme"], axis=0)
fig, ax = plt.subplots(figsize=(11, 5))
sns.heatmap(theme_pivot, annot=True, fmt=".0f", cmap="RdBu_r", center=0, ax=ax,
            cbar_kws={"label": "percentage points vs. grand mean"})
ax.set_title("theme: each ethnicity's deviation from the grand-mean theme distribution")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "theme_deviation_from_grand_mean_by_ethnicity_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()


**Takeaway.** Splitting "Non-German" into its 11 constituent categories (6,908 known-ethnicity racialised reviews total) shows the earlier German-vs-non-German gap was real but was averaging over a much sharper underlying pattern. **`Arab` and `Türkiye` are consistently the two most extreme categories, in the same direction as each other, and opposite to `Germany`:** both show far less `Crowd` (Arab 9.8% vs. a 26.5% grand mean across all 12 categories, -16.6pp; Türkiye 12.7%, -13.8pp), far less `Etiquette` (Arab 5.4% vs. 16.9%, -11.6pp; Türkiye 7.8%, -9.1pp), far more `Staff` (Arab 28.1% vs. 14.9%, +13.2pp; Türkiye 27.4%, +12.5pp), more `Access` (Arab 15.1% vs. 4.9%, +10.2pp), and much more `incident`-specific framing (Arab 49.0% vs. 33.1%, +15.9pp; Türkiye 47.3%, +14.2pp) -- visible as the two solidly blue-on-Crowd/Etiquette, red-on-Staff/Access columns in the heatmap above. `Germany` itself deviates in the *opposite* direction on the same labels (Crowd +6.1pp, incident -6.1pp) but by a much smaller margin -- Arab and Türkiye are roughly 2-3x further from the grand mean than Germany is.

The categories in between are not a uniform "everyone else": `Poland` and `Russian Federation` actually lean mildly *toward* the Germany pattern rather than sitting neutral -- Poland shows more `Explicit` approach (84.7% vs. 75.8%, +8.9pp), more `generalised` framing (74.5% vs. 66.9%), more `Crowd` (32.8% vs. 26.5%), and less `Negative` connotation (80.3% vs. 86.9%); Russian Federation shows less `Negative` (79.5% vs. 86.9%) and more `Explicit` (82.7% vs. 75.8%) too. `United States`, `France`, `Italy`, `Indonesia`, `Philippines`, `Brazil`, and `Other` sit closer to the grand mean throughout, without a strong signature of their own in this scan.

Put together: the German-vs-non-German split from the subsection above is best read as **"Arab + Türkiye vs. Germany (+ mildly Poland/Russian Federation)," not a uniform German-vs-everyone-else effect** -- the other seven non-German categories were diluting, not contradicting, the Arab/Türkiye pattern when lumped into one "Non-German" bucket.

**Caveat on comparing Cramér's V across the two subsections.** `theme`'s Cramér's V here (0.098) looks smaller than the German-status version above (0.180), and `context`'s here (0.139) looks larger than its German-status version (0.085) -- but these aren't directly comparable: Cramér's V is normalised by `min(rows, cols) - 1`, which grows from 1 (an 11-theme x 2-group table) to 10 (an 11-theme x 12-category table) once we stop collapsing to a binary. A smaller V here reflects testing a finer-grained, 12-way distinction against the same theme categories, not that theme actually matters less once you look at specific ethnicities -- the raw deviations above (Arab -16.6pp on Crowd alone) are, if anything, sharper than the binary comparison suggested.

### Consolidating into four theoretically-motivated groups: Germany, Arab, Türkiye, Europe

The full 12-category `ethnicity_final` scan above is useful for finding which specific ethnicities drive a pattern, but it's more categories than needed for reuse downstream (e.g. crossing against geography next, or against constituencies later). We consolidate into a smaller, deliberately-chosen scheme rather than an arbitrary top-N cut: **`Germany`**, **`Arab`**, and **`Türkiye`** keep their own groups (both because they're large enough on their own and because Arab and Türkiye reviewers have distinct migration histories in Germany that are worth being able to tell apart later, rather than merging them into one "non-European" catch-all), plus a new **`Europe`** group (`Russian Federation`, `Italy`, `Poland`, `France` -- the top-10 countries above that are European; Germany and Türkiye keep their own groups rather than folding into this one). Everything else (`United States`, `Philippines`, `Indonesia`, `Brazil`, and the long-tail `Other`) falls into a residual **`Other`** group, and `Unknown` (missing prediction) stays separate and excluded from comparisons, as throughout this section.


In [ ]:
EUROPE_COUNTRIES = {"Russian Federation", "Italy", "Poland", "France"}

def to_ethnicity_group4(ethnicity_final_value):
    if ethnicity_final_value in ("Germany", "Arab", "Türkiye", "Unknown"):
        return ethnicity_final_value
    if ethnicity_final_value in EUROPE_COUNTRIES:
        return "Europe"
    return "Other"

df["ethnicity_group4"] = df["ethnicity_final"].map(to_ethnicity_group4)
racialised_df["ethnicity_group4"] = df["ethnicity_group4"]

print("ethnicity_group4 counts (of all 7,768 racialised reviews):")
print(df["ethnicity_group4"].value_counts())
print((df["ethnicity_group4"].value_counts(normalize=True) * 100).round(1))

known4 = df[df["ethnicity_group4"] != "Unknown"].copy()
group4_results = {}
for dim in DIMENSIONS:
    exploded = known4[[dim, "ethnicity_group4"]].explode(dim)
    ct = pd.crosstab(exploded[dim].to_numpy(), exploded["ethnicity_group4"].to_numpy())
    chi2, p, dof_, _ = stats.chi2_contingency(ct)
    n = ct.values.sum()
    cv = np.sqrt(chi2 / (n * (min(ct.shape) - 1)))
    group4_results[dim] = {"ct": ct, "col_pct": ct.div(ct.sum(axis=0), axis=1) * 100, "chi2": chi2, "p": p, "cramers_v": cv}
    print(f"\n--- {dim}: chi2={chi2:.1f}, dof={dof_}, p={p:.2e}, Cramer's V={cv:.3f} ---")
    print(group4_results[dim]["col_pct"].round(1))


**Takeaway.** The four-group consolidation (Germany 2,146 / Arab 462 / Türkiye 371 / Europe 655 / Other 3,274 / Unknown 860) reproduces the same ranking as the full 12-category scan, just easier to read: `theme` is still the strongest dimension (Cramér's V=0.146), then `context` (0.132), then `approach` (0.065) and `connotation` (0.053) trailing well behind. And it makes the earlier "Arab + Türkiye vs. Germany" story directly visible in one table: on `theme`, **Arab and Türkiye track each other closely** (`Staff` 28.1%/27.4%, `Access` 15.1%/10.2%, `Crowd` 9.8%/12.7%, `Etiquette` 5.4%/7.8%) and sit at the opposite end from **Germany** (`Staff` 9.5%, `Access` 2.1%, `Crowd` 32.6%, `Etiquette` 21.5%). `Europe` and `Other` both land in between, close to each other and closer to Germany's profile than to Arab/Türkiye's -- confirming Europe isn't a third pole, it's a middling group. Same pattern on `context`: Arab (49.0% incident) and Türkiye (47.3%) are far more incident-framed than Germany (27.1%), with Europe (31.8%) and Other (33.5%) in between.

### Theme-pair composition: do Germany/Europe and Arab/Turkiye pair different themes together?

Beyond which single themes each group raises most (previous subsection), do multi-themed reviews (a review can carry up to 3 `theme` labels) pair themes together differently depending on who's writing? We check the *rate* of multi-theming by group first (is one group just more likely to pile on themes at all), then which specific theme-pairs dominate each group, then whether the same pair gets a different tone (connotation/context/approach) depending on the writer's `ethnicity_group4`.

In [ ]:
from itertools import combinations

GROUPS4 = ["Germany", "Arab", "Türkiye", "Europe"]
known4_grp = known4[known4["ethnicity_group4"].isin(GROUPS4)].copy()
known4_grp["n_themes"] = known4_grp["theme"].apply(
    lambda c: len(c) if isinstance(c, (list, tuple, set, np.ndarray)) else 1
)

print("multi-theme rate (>=2 themes) by group:")
multi_rate = known4_grp.groupby("ethnicity_group4")["n_themes"].apply(lambda s: (s >= 2).mean())
print(multi_rate.round(3))

ct_multi = pd.crosstab(known4_grp["ethnicity_group4"], known4_grp["n_themes"] >= 2)
chi2_m, p_m, dof_m, _ = stats.chi2_contingency(ct_multi)
v_m = np.sqrt(chi2_m / (ct_multi.values.sum() * (min(ct_multi.shape) - 1)))
print(f"chi2={chi2_m:.2f}, dof={dof_m}, p={p_m:.3f}, Cramer's V={v_m:.3f} "
      f"-- not a real group difference in *how often* reviews are multi-themed")

# --- theme-pair composition, restricted to multi-theme reviews ---
# pairs stored as "A + B" strings (not tuples) -- a Series of tuples confuses
# pandas' crosstab internals (tuple-valued Series reads as MultiIndex-like)
multi = known4_grp[known4_grp["n_themes"] >= 2].copy()

def theme_pairs(themes):
    return [f"{a} + {b}" for a, b in combinations(sorted(set(themes)), 2)]

multi["pairs"] = multi["theme"].apply(theme_pairs)
pair_rows = multi[["pairs", "ethnicity_group4"]].explode("pairs").dropna(subset=["pairs"])

pair_counts_overall = pair_rows["pairs"].value_counts()
top_pairs = pair_counts_overall.head(10).index.tolist()

top_mask = pair_rows["pairs"].isin(top_pairs)
# .to_numpy() (not raw Series): explode() leaves a duplicated index, which crosstab's
# internal alignment can't handle, same fix used in the theme x connotation cell earlier
ct_pairs = pd.crosstab(pair_rows.loc[top_mask, "pairs"].to_numpy(), pair_rows.loc[top_mask, "ethnicity_group4"].to_numpy())
chi2_p, p_p, dof_p, _ = stats.chi2_contingency(ct_pairs)
v_p = np.sqrt(chi2_p / (ct_pairs.values.sum() * (min(ct_pairs.shape) - 1)))
print(f"\ntop-10 theme-pairs x group: chi2={chi2_p:.1f}, dof={dof_p}, p={p_p:.2e}, Cramer's V={v_p:.3f}")

row_pct_pairs = (ct_pairs.div(ct_pairs.sum(axis=1), axis=0) * 100).round(1)
print("\nrow % (share of each pair's occurrences, by group):")
print(row_pct_pairs)

print("\ntop-5 pairs by group, as % of that group's own multi-theme reviews:")
for grp in GROUPS4:
    grp_pairs = pair_rows.loc[pair_rows["ethnicity_group4"] == grp, "pairs"].value_counts()
    grp_n = (multi["ethnicity_group4"] == grp).sum()
    print(f"\n{grp} (n multi-theme={grp_n}):")
    print((grp_pairs.head(5) / grp_n * 100).round(1))


In [ ]:
FOCUS_PAIRS = ["Belonging + Staff", "Access + Staff"]

for pair in FOCUS_PAIRS:
    mask = multi["pairs"].apply(lambda ps: pair in ps)
    sub = multi[mask]
    print(f"\n=== {pair} (n={len(sub)}) ===")
    for dim in ["connotation", "context", "approach"]:
        exploded = sub[[dim, "ethnicity_group4"]].explode(dim)
        ct = pd.crosstab(exploded[dim].to_numpy(), exploded["ethnicity_group4"].to_numpy())
        print(f"\n{dim} x group:")
        print(ct)
        if (ct < 10).any().any():
            print("(>=1 cell < 10 -- too thin to trust a chi-square here, not computing one)")
            continue
        chi2_t, p_t, dof_t, _ = stats.chi2_contingency(ct)
        v_t = np.sqrt(chi2_t / (ct.values.sum() * (min(ct.shape) - 1)))
        print(f"chi2={chi2_t:.2f}, dof={dof_t}, p={p_t:.3f}, Cramer's V={v_t:.3f}")


**Takeaway.** Multi-theming rate itself doesn't differ by group (57-65% carry >=2 themes, chi2=6.52, p=0.089) -- the split isn't in *how often* reviews stack themes. It's in *which* themes get stacked together (top-10 pairs x group: chi2=609.1, dof=27, p=3.7e-111, **Cramér's V=0.311**, substantially stronger than the theme-marginal association found in the previous subsection, V=0.146): Germany/Europe reviews cluster a crowd-behaviour/hygiene complex -- `Crowd+Etiquette`, `Crowd+Safety`, `Etiquette+Safety`, `Clothing+Crowd`/`Clothing+Hygiene` are 72-81% German by occurrence -- while Arab/Türkiye reviews cluster a staff-interaction complex instead: `Belonging+Staff` is Arab's (27.0% of Arab's own multi-theme reviews) and Türkiye's (24.2%) single most common pair, doesn't crack Germany's top-5 at all, and sits at only 8.6% for Europe; `Access+Staff` runs the same way (Arab's #2 pair at 18.3%, Türkiye's #2 at 13.8%). `Language+Staff` runs the other way -- 46.1% of its occurrences are German, 34.8% European, vs. ~9-10% each for Arab/Türkiye.\n\nChecking whether the *same* pair gets a different tone depending on who raises it (`Belonging+Staff` n=221, `Access+Staff` n=122, against connotation/context/approach -- 6 dimension x pair combinations total): connotation stays 88-100% Negative and approach 70-94% Explicit across every group in both pairs, and only one of the six combinations had cells large enough to actually test (`Belonging+Staff` x context: chi2=1.73, p=0.630, not significant -- a mild, non-significant drift toward incident-framing for Arab/Türkiye/Europe vs. Germany). The other five (both pairs' connotation and approach, and `Access+Staff`'s context) had cells of 0-9 -- too thin to trust a chi-square, so no significance claim is made there. The finding worth standing behind is compositional (*which* pairs get raised differs sharply by group, V=0.311) -- not tonal (*how* a shared pair is framed once raised looks the same across groups, as far as this sample size can tell).

### Proxy check: reported vs. attributed racism, by ethnicity group

Before committing annotation effort to a fifth dimension (reviewer stance -- reported vs. attributed), a cheap proxy from the existing `context` x `theme` labels:

- `proxy_reported` = `context == 'incident'` and `theme` in `{Staff, Access, Belonging, Language}` -- a specific incident involving the reviewer's own treatment.
- `proxy_attributed` = `context == 'generalised'` and `theme` in `{Crowd, Etiquette}` -- a generalised claim about a group's behaviour.

`Unknown`-ethnicity reviews are kept in (not dropped) so the panel reflects the full 7,768. If German-origin reviews skew attributed and Arab/Türkiye-origin reviews skew reported -- consistent with the theme-pair Cramér's V panel above -- that's justification to build the full dimension.

In [ ]:
ALL_GROUPS = ["Germany", "Arab", "Türkiye", "Europe", "Other", "Unknown"]

proxy_reported = df["context"].apply(lambda c: "incident" in c if isinstance(c, (list, tuple, set, np.ndarray)) else c == "incident") & \
    df["theme"].apply(lambda t: any(x in t for x in ["Staff", "Access", "Belonging", "Language"]) if isinstance(t, (list, tuple, set, np.ndarray)) else t in ["Staff", "Access", "Belonging", "Language"])
proxy_attributed = df["context"].apply(lambda c: "generalised" in c if isinstance(c, (list, tuple, set, np.ndarray)) else c == "generalised") & \
    df["theme"].apply(lambda t: any(x in t for x in ["Crowd", "Etiquette"]) if isinstance(t, (list, tuple, set, np.ndarray)) else t in ["Crowd", "Etiquette"])

df["proxy_reported"] = proxy_reported
df["proxy_attributed"] = proxy_attributed

print(f"proxy_reported: {proxy_reported.sum()} ({proxy_reported.mean():.1%} of all {len(df)} racialised reviews)")
print(f"proxy_attributed: {proxy_attributed.sum()} ({proxy_attributed.mean():.1%} of all {len(df)} racialised reviews)")
print(f"both (multi-theme overlap): {(proxy_reported & proxy_attributed).sum()}")
print(f"neither: {(~proxy_reported & ~proxy_attributed).sum()}")

proxy_results = {}
for proxy_name in ["proxy_reported", "proxy_attributed"]:
    ct = pd.crosstab(df["ethnicity_group4"], df[proxy_name]).reindex(ALL_GROUPS)
    chi2, p, dof_, _ = stats.chi2_contingency(ct)
    n = ct.values.sum()
    cv = np.sqrt(chi2 / (n * (min(ct.shape) - 1)))
    row_pct = (ct.div(ct.sum(axis=1), axis=0) * 100).round(1)
    proxy_results[proxy_name] = {"ct": ct, "row_pct": row_pct, "chi2": chi2, "p": p, "cramers_v": cv}
    print(f"\n--- {proxy_name} x ethnicity_group4 (all {len(ALL_GROUPS)} groups incl. Unknown): chi2={chi2:.1f}, dof={dof_}, p={p:.2e}, Cramer\'s V={cv:.3f} ---")
    print(row_pct)

### Does geography (East/West/Berlin) modulate any of this?

Before treating `ethnicity_group4` as fixed and moving to constituency-level work, one more cross-check: does *where in Germany* a reviewer is (East, West, or Berlin -- the `region` column built in Section 1) interact with any of the ethnicity-vs-dimension patterns above? We test this within each of the four known groups separately (rather than one combined test), since the hypothesis was specifically about whether e.g. German reviewers *in the East* differ from German reviewers *in the West*, not about East/West differing overall.


In [ ]:
GROUPS4 = ["Germany", "Arab", "Türkiye", "Europe", "Other"]

print("region counts within each ethnicity_group4:")
print(pd.crosstab(known4["ethnicity_group4"], known4["region"]))

region_results = {}
for grp in GROUPS4:
    sub = known4[known4["ethnicity_group4"] == grp]
    print(f"\n{'='*70}\n{grp} (n={len(sub)})\n{'='*70}")
    region_results[grp] = {}
    for dim in DIMENSIONS:
        exploded = sub[[dim, "region"]].explode(dim)
        ct = pd.crosstab(exploded[dim].to_numpy(), exploded["region"].to_numpy())
        chi2, p, dof_, _ = stats.chi2_contingency(ct)
        n = ct.values.sum()
        cv = np.sqrt(chi2 / (n * (min(ct.shape) - 1)))
        region_results[grp][dim] = {"chi2": chi2, "p": p, "cramers_v": cv}
        print(f"{dim}: chi2={chi2:.1f}, dof={dof_}, p={p:.3f}, Cramer's V={cv:.3f}")

print("\n" + "=" * 70)
print("summary -- p-value and Cramer's V for every (group, dimension) region test:")
summary = pd.DataFrame({
    (grp, dim): {"p": region_results[grp][dim]["p"], "V": region_results[grp][dim]["cramers_v"]}
    for grp in GROUPS4 for dim in DIMENSIONS
}).T
print(summary.round(3))
print(f"\ntests reaching p<0.05: {(summary['p'] < 0.05).sum()} / {len(summary)}")


**Takeaway.** No -- geography doesn't meaningfully modulate any of this, even at the finer four-group grain. Across all 20 tests (5 groups x 4 dimensions), only **3 reach p<0.05** (`Europe`/theme: p=0.028, V=0.122; `Other`/theme: p<0.001, V=0.072; `Other`/approach: p=0.026, V=0.040) -- and every one of them has a small Cramér's V, the same "statistically detectable at this N but substantively minor" pattern seen elsewhere in this notebook. Critically, **`Germany`, `Arab`, and `Türkiye` -- the three groups the original hypothesis was actually about -- show no significant region effect on any dimension** (all p>=0.155). The specific hypothesis (Germans in the East being more implicit/incident-prone) is not supported at either grain: `context` for German reviewers comes back p=0.404 (binary) and p=0.404 again here (unchanged, since it's the same German subset), and `approach` comes back p=0.817. **Region is not a variable worth carrying into the constituency-level work below** -- it doesn't explain who racialises differently or how.

### Region × dimension, unconditioned on ethnicity

The subsection above tested region's effect *within* each ethnicity group specifically. The simpler, aggregate version -- does `theme`/`connotation` shift by region at all, ignoring who's writing -- is worth having directly too, moved here from what was originally drafted as a Part 2 section since `region` (derived from `state`) needs no political/socioeconomic data. Every one of the 7,768 racialised reviews has a well-populated region with thousands of reviews each, so this is as reliable as the corpus-level dimension analysis in WHEN.

In [ ]:
def chi2_cramers_v(ct):
    chi2, p, dof, expected = stats.chi2_contingency(ct)
    n = ct.values.sum()
    v = np.sqrt(chi2 / (n * (min(ct.shape) - 1)))
    min_expected = expected.min()
    return chi2, p, dof, v, min_expected

# --- region x connotation (single-label, no explode needed) ---
ct_region_conn = pd.crosstab(df["region"], df["connotation"].apply(lambda x: x[0]))
chi2, p, dof, v, min_exp = chi2_cramers_v(ct_region_conn)
print("region x connotation:")
print(ct_region_conn)
print(f"chi2={chi2:.1f}, dof={dof}, p={p:.2e}, Cramer's V={v:.3f}, min expected count={min_exp:.1f}\n")

# --- region x theme (multi-label, explode) ---
region_theme = df[["review_id", "region", "theme"]].explode("theme")
ct_region_theme = pd.crosstab(region_theme["region"].to_numpy(), region_theme["theme"].to_numpy())
chi2, p, dof, v, min_exp = chi2_cramers_v(ct_region_theme)
print("region x theme:")
print(ct_region_theme)
print(f"chi2={chi2:.1f}, dof={dof}, p={p:.2e}, Cramer's V={v:.3f}, min expected count={min_exp:.1f}")


**Takeaway.** Same pattern the ethnicity-conditioned version above found: connotation doesn't shift with region at all (chi2=6.7, dof=4, p=0.152, Cramér's V=0.021 -- negligible), while theme does, modestly (chi2=84.7, dof=20, p=6.2e-10, Cramér's V=0.056 -- statistically real at this N, small in practice). Berlin skews slightly toward `Staff`/`Security` relative to West/East, and West carries proportionally more `Crowd`/`Etiquette` -- differences visible in the raw counts above but not large enough to call region a strong driver of *what* gets said, only *whether* it gets said (the rate-level finding elsewhere in this notebook). Some `theme` cells run below the usual expected-count-5 comfort threshold (min expected count=4.5) -- a caveat noted rather than acted on, consistent with how thin cells are flagged elsewhere in this notebook.

### Draft: paper write-up tables (WHO / cross-dimension significance)



Two tables: (1) ethnicity name-origin composition, social=yes vs racialised=yes, (2) a master chi-square/Cramer's V reference across every WHO cross-tab test run in this notebook (ethnicity x racialised, language x racialised, ethnicity x language, dimension x German-status/ethnicity_final/ethnicity_group4/language-within-Non-German, theme x connotation, theme-pair x group4, geography x group4). Source file: `who_section_findings_tables.tex` (session scratchpad, not reachable from your side -- this cell is the copy to actually use). Move into the paper repo when ready.

```latex
% Requires: \usepackage{booktabs} \usepackage{array}
% All numbers pulled directly from descriptive_analysis.ipynb WHO section
% (takeaway cells + underlying code-cell printed output, cells 03060c4b/
% ff15dc3f/7586cc41/1c5ac726/19e28b8e/bfc2f3b8/dac639cb/ed670d1d/83a62ea8/
% 8bf1a65c/8130b4b0/9e722a72 -- the theme-pair cell added earlier).

% ------------------------------------------------------------------
% Table 1 -- ethnicity name-origin composition: social=yes vs racialised=yes
% ------------------------------------------------------------------
\begin{table}[htbp]
\centering
\footnotesize
\setlength{\tabcolsep}{4pt}
\caption{Reviewer name-origin ethnicity composition, social=yes vs.\ racialised=yes ($n$=504,072 social=yes; overall association chi2=1573.1, $p\ll0.001$, Cram\'er's V=0.056).}
\label{tab:who-ethnicity-composition}
\begin{tabular}{@{} l r r r @{}}
\toprule
Category & Social (\%) & Racialised (\%) & Fold change \\
\midrule
Germany & 39.62 & 27.63 & 0.70$\times$ \\
Arab & 1.92 & 5.95 & 3.1$\times$ \\
T\"urkiye & 1.81 & 4.78 & 2.6$\times$ \\
Other (real, non-top-10) & 23.27 & 25.54 & 1.1$\times$ \\
Unknown (no prediction) & 7.74 & 11.07 & 1.4$\times$ \\
\bottomrule
\end{tabular}

\vspace{2pt}
\raggedright\footnotesize
Remaining $\sim$26\% spread across the other top-10 countries (US, Russia, France, Indonesia, Poland, Philippines, Italy, Brazil), each within 1--2pp of its social-grain share at every grain --- the racialised-vs-not shift is concentrated in Germany/Arab/T\"urkiye/Unknown, not spread evenly.
\end{table}

% ------------------------------------------------------------------
% Table 2 -- master significance / effect-size reference
% ------------------------------------------------------------------
\begin{table}[htbp]
\centering
\footnotesize
\setlength{\tabcolsep}{3pt}
\caption{WHO section and cross-dimension association tests (chi-square, Cram\'er's V).}
\label{tab:who-significance-master}
\begin{tabular}{@{} l r r l r @{}}
\toprule
Test & $n$ & $\chi^2$ (dof) & $p$ & V \\
\midrule
\multicolumn{5}{@{}l}{\emph{Composition}} \\
Ethnicity (12-cat) $\times$ racialised & 504{,}072 & 1573.1 & $\ll 0.001$ & 0.056 \\
Language (11-cat) $\times$ racialised & 504{,}072 & 645.4 (10) & 3.3e-132 & 0.036 \\
Ethnicity (13) $\times$ language (15) & 7{,}768 & 5349.8 (168) & $\approx 0$ & 0.240 \\
\midrule
\multicolumn{5}{@{}l}{\emph{Dimension $\times$ German-status (binary, $n$=6,908)}} \\
theme & 6{,}908 & 393.1 (10) & 2.8e-78 & 0.180 \\
context & 6{,}908 & --- & $<0.05$ & 0.085 \\
approach & 6{,}908 & 18.2 (2) & 1.1e-04 & 0.050 \\
connotation & 6{,}908 & 7.1 (2) & 0.029 & 0.032 \\
\midrule
\multicolumn{5}{@{}l}{\emph{Dimension $\times$ ethnicity\_final (12-way)}} \\
theme & 6{,}908 & 1163.7 (110) & 1.9e-175 & 0.098 \\
context & 6{,}908 & 165.1 (11) & 1.2e-29 & 0.139 \\
approach & 6{,}908 & 70.8 (22) & 4.9e-07 & 0.070 \\
connotation & 6{,}908 & 57.7 (22) & 4.9e-05 & 0.064 \\
\midrule
\multicolumn{5}{@{}l}{\emph{Dimension $\times$ ethnicity\_group4 (5-way)}} \\
theme & 6{,}908 & 1036.6 (40) & 2.6e-191 & 0.146 \\
context & 6{,}908 & --- & --- & 0.132 \\
approach & 6{,}908 & 60.3 (8) & 4.0e-10 & 0.065 \\
connotation & 6{,}908 & 39.3 (8) & 4.4e-06 & 0.053 \\
\midrule
\multicolumn{5}{@{}l}{\emph{Dimension $\times$ language, within Non-German ($n$=4,762)}} \\
connotation & 4{,}762 & 324.5 (28) & 3.2e-52 & 0.183 \\
approach & 4{,}762 & 89.5 (28) & 2.4e-08 & 0.095 \\
context & 4{,}762 & 31.7 (14) & 0.004 & 0.073 \\
theme & 4{,}762 & 381.2 (140) & 3.2e-24 & 0.067 \\
\midrule
\multicolumn{5}{@{}l}{\emph{Theme content \& theme-pair composition}} \\
theme $\times$ connotation (full subset) & 7{,}768 & 413.9 (20) & 2.6e-75 & 0.122 \\
theme-pair (top-10) $\times$ group4 & 3{,}634 & 609.1 (27) & 3.7e-111 & 0.311 \\
multi-theme rate $\times$ group4 & 3{,}634 & 6.52 (3) & 0.089 & 0.042 \\
\midrule
\multicolumn{5}{@{}l}{\emph{Geography (East/West/Berlin) $\times$ dimension, per group4 (3/20 significant)}} \\
Europe $\times$ theme & 655 & 33.7 (20) & 0.028 & 0.122 \\
Other $\times$ theme & 3{,}274 & 60.7 (20) & $<0.001$ & 0.072 \\
Other $\times$ approach & 3{,}274 & 11.0 (4) & 0.026 & 0.040 \\
\bottomrule
\end{tabular}

\vspace{2pt}
\raggedright\footnotesize
Cells marked --- were not reported as an exact statistic in the source cell (V and significance direction are still as stated). Every other geography$\times$dimension test (17/20, including \emph{all} of Germany/Arab/T\"urkiye on every dimension) is not significant, $p \geq 0.097$ throughout, Germany/Arab/T\"urkiye specifically $p \geq 0.155$. All effect sizes here are small-to-moderate by convention (Cram\'er's V $<0.35$); \texttt{theme-pair $\times$ group4} (V=0.311) is the single largest association found anywhere in this section.
\end{table}

```


**Taking stock of the WHO section.**

**Ethnicity facet.** A refined `ethnicity_final` categorization (top-10 non-Arab countries, an auditable `Arab` aggregate, explicit `Other` vs. `Unknown`, fit once on `social=='yes'` for stability) replaces a conflated first-pass cut, and is reused unchanged across every grain in this section. The racialised=yes vs. no comparison shows a small but real shift (Cramér's V=0.056): Germany down (39.8%->27.6%), `Arab` and `Türkiye` both roughly tripling. A three-way check (entire corpus / social / racialised) confirms this shift is concentrated entirely in the racialised grain, not an artifact of the `social=='yes'` filter -- entire-corpus and social percentages are nearly identical for every category. A data-quality fix along the way (65,908 corpus-wide nulls in `org_review_lang` were mechanically mislabeled English, not missing data) recovered 17 racialised reviews `pd.crosstab` had been silently dropping from every language-facet table below.

**Language facets -- two of them, and they don't behave the same way.** The ethnicity x language cross-tab (Cramér's V=0.240) is concentrated in exactly the categories where a genuine identity signal is plausible: `France` (49.7% French, ~16x its own corpus baseline), `Poland`/`Russian Federation` (16-33x baseline), `Arab`/`Türkiye` (~12-13x) -- while categories without an obvious "home language" in `org_review_lang` (Brazil, Indonesia, Philippines, US, Other, Unknown) show no elevation, a limit of what this column can register rather than necessarily an absence of signal. A parallel three-way language table (mirroring the ethnicity one) shows language does **not** stay stable between entire-corpus and social=yes the way ethnicity does -- German rises 88.6%->95.2%, `Other language` (mostly English) drops 6.4%->0.8%, since many non-social reviews are short generic non-German snippets that get filtered out once a review has to be about the venue's social experience. The social->racialised shift is significant (chi2=645, p=3.3e-132, Cramér's V=0.036) and reproduces the same France/Arab/Czech/Polish elevation.

**Content differences, four ways.** German vs. non-German racialised reviews differ mainly on `theme` (Cramér's V=0.180) and `context` (0.085): German-authored reviews skew `Crowd`/`Etiquette`, framed `generalised`; non-German-authored skew `Staff`/`Access`, framed as a specific `incident` -- both equally `Negative` in tone (~87%), so the ethnicity signal tracks *what*/*how concretely*, not *how negative*. Breaking "Non-German" into its actual constituent ethnicities (`Arab`/`Türkiye`/`Europe`/`Other`, consolidated from the full 12-category scheme) shows this is really an **`Arab` + `Türkiye` vs. `Germany`** polarity, not a uniform non-German effect -- `Poland`/`Russian Federation` lean mildly toward the Germany pattern instead, and `US`/`France`/`Italy`/`Indonesia`/`Philippines`/`Brazil`/`Other` show no strong signature either way. Within Non-German reviewers specifically, the *language* written in predicts `connotation` (tone) far better than name-origin does (Cramér's V=0.183 vs. 0.032) -- a clean gradient from German-language (89.8% Negative, harshest) down to Czech (56.8%, mildest among languages with adequate N). Crossing dimension pairs directly also surfaced a lead for RQ3: the `Coded` approach skews German across nearly every theme it co-occurs with, while `Access` stays non-German-skewed almost regardless of what else co-occurs with it.

**Geography checked and ruled out.** East/West/Berlin doesn't meaningfully modulate any of this -- only 3 of 20 (group x dimension) tests reach significance, all with small effect sizes, and none involve `Germany`, `Arab`, or `Türkiye`, the three groups the hypothesis was actually about.

**Still open / deferred**: the `"Other"` category (~140 distinct countries) remains too heterogeneous to interpret as one group; several small-language and thin-ethnicity cells throughout should be read as suggestive, not precise; and categories without a matching "home language" in `org_review_lang` can't be distinguished from genuinely having no language signal at all. The France/Poland/Russian-Federation/Arab/Türkiye language elevation and the `Arab`+`Türkiye`-vs-`Germany` content polarity are the two strongest leads worth carrying into the qualitative dimension-level work for RQ3.

## Content baseline: theme, approach, connotation, context

`approach`, `theme`, `connotation`, and `context` are **multi-label** (each review can carry more than one theme, for instance), stored as arrays and populated only for the 7,768 racialised=yes reviews. This establishes the unconditioned content baseline for WHO's dimension analysis above: every review in this subset is in scope regardless of geography, so frequencies and cross-dimension associations computed at this full N are as reliable as the corpus gets, and give the reference point that the German/non-German and ethnicity-group comparisons earlier in WHO are measured against. It also feeds directly into **RQ3** (the content of racialised discourse); geographic breakdowns of these dimensions are deferred to Part 2's state-level cut, where sample size becomes the binding constraint.

In [ ]:
# racialised_df was already built earlier in this notebook (Part 1 setup); reused here for the dimension analysis
DIMENSIONS = ["approach", "theme", "connotation", "context"]

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
dimension_counts = {}
for ax, dim in zip(axes.flat, DIMENSIONS):
    exploded = racialised_df[dim].explode()
    counts = exploded.value_counts()
    dimension_counts[dim] = counts
    pct_of_reviews = counts / len(racialised_df)
    pct_of_reviews.plot(kind="barh", ax=ax, color="#55A868")
    ax.set_title(f"{dim} (% of the 7,768 racialised reviews carrying this label)")
    ax.set_xlabel("share of reviews")
    ax.invert_yaxis()
plt.tight_layout()
plt.savefig(FIGURES_DIR / "dimension_label_prevalence_overview.png", dpi=150, bbox_inches="tight")
plt.show()

for dim in DIMENSIONS:
    n_labels_per_review = racialised_df[dim].apply(len)
    print(f"{dim}: avg {n_labels_per_review.mean():.2f} labels/review, "
          f"{(n_labels_per_review > 1).mean():.1%} of reviews carry >1 label")


In [ ]:
# theme x connotation association: explode both, one row per (review, theme, connotation) combination
pair_df = racialised_df[["review_id", "theme", "connotation"]].explode("theme").explode("connotation")
# pass .values (not Series) to crosstab: the double explode leaves a duplicated index, which
# crosstab's internal alignment can't handle if given Series that "look" index-aligned but aren't unique
ct_theme_conn = pd.crosstab(pair_df["theme"].to_numpy(), pair_df["connotation"].to_numpy())
ct_theme_conn.index.name, ct_theme_conn.columns.name = "theme", "connotation"
print(ct_theme_conn)

chi2, p, dof, _ = stats.chi2_contingency(ct_theme_conn)
n = ct_theme_conn.values.sum()
cramers_v = np.sqrt(chi2 / (n * (min(ct_theme_conn.shape) - 1)))
print(f"\ntheme x connotation: chi2 = {chi2:.1f}, dof = {dof}, p = {p:.2e}, Cramer's V = {cramers_v:.3f}")

fig, ax = plt.subplots(figsize=(7, 6))
row_pct = ct_theme_conn.div(ct_theme_conn.sum(axis=1), axis=0)
sns.heatmap(row_pct, annot=ct_theme_conn.values, fmt="d", cmap="Greens", ax=ax, cbar_kws={"label": "row %"})
ax.set_title("Theme x connotation (annotated with raw counts, colour = row-normalised %)")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "theme_connotation_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()


**Takeaway.** `theme` is the dimension that actually behaves as multi-label in practice (avg 1.77 labels/review, 61.6% of reviews carry more than one), while `approach` and `connotation` are functionally close to single-label (3.6% and 1.4% multi-label respectively) — worth knowing before choosing how to encode each dimension for any downstream modeling. `connotation` is overwhelmingly **Negative** across every theme, but not uniformly so: `Security` and `Safety` are almost purely negative (97.7% and 98.3%), while `Crowd`, `Belonging`, and `Staff` have a noticeably larger (still minority) neutral/positive share. The theme×connotation association is real and moderate (Cramér's V = 0.122, larger than the ethnicity association above) — i.e. which theme a review invokes does meaningfully shift how likely it is to carry a non-negative connotation, which is a legitimate quantitative finding at full sample size regardless of geography. `Crowd` is the single largest theme by a wide margin (3,724 of the theme-label instances), ahead of `Etiquette`, `Staff`, and `Belonging`.

# Bringing WHO and WHERE together: what's actually driving the constituency-level rate?

WHERE established *that* the racialised rate varies sharply by constituency (1.54% overall, up to 8.6% in `Offenburg`) and *that* raw border-proximity doesn't cleanly explain it. WHO established *who* is racialising (`Germany`/`Arab`/`Türkiye`/`Europe`/`Other`) and *what* they racialise about. This section asks the question those two only gesture at separately: does knowing *who* is present in a constituency help explain *why* its rate is high or low -- and is any of this just an artifact of how many reviews/pools a constituency happens to have? This directly matters for whether `racialised_rate` has real signal to work with as a target variable, or would just be modelling review-count noise.


## Is the constituency-level rate just a volume artifact?

The concentration subsection in WHERE already ruled out a *venue-level* artifact (a high rate isn't usually one bad pool skewing an otherwise-quiet constituency -- rate correlates with the *share* of pools affected, not with one dominant pool). But a *constituency-level* volume artifact is a distinct question: do bigger constituencies (more pools, more reviews) simply show different rates because they're bigger, regardless of anything about who's there? We check this directly across all 247 `reliable_n` constituencies (already restricted to >=100 social / >=10 racialised reviews, so this isn't a small-N noise question).


In [ ]:
rel = const_agg[const_agg["reliable_n"]].copy()
print(f"reliable_n constituencies: {len(rel)} / 299")

rho_sr, p_sr = stats.spearmanr(rel["social_reviews"], rel["racialised_rate"])
rho_pc, p_pc = stats.spearmanr(rel["pool_count"], rel["racialised_rate"])
print(f"racialised_rate vs. social_reviews: rho={rho_sr:.3f}, p={p_sr:.2e}")
print(f"racialised_rate vs. pool_count:     rho={rho_pc:.3f}, p={p_pc:.2e}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for ax, col, rho, p in [(axes[0], "social_reviews", rho_sr, p_sr), (axes[1], "pool_count", rho_pc, p_pc)]:
    colors = np.where(rel["reliable_n"], "#4C72B0", "#C44E52")
    ax.scatter(rel[col], rel["racialised_rate"], c=colors, s=16, alpha=0.7)
    ax.set_xscale("log")
    ax.set_xlabel(f"{col} (log scale)")
    ax.set_ylabel("racialised_rate")
    ax.set_title(f"{col} vs. racialised_rate\n(rho={rho:.3f}, p={p:.1e})")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "racialised_rate_vs_social_reviews_and_pool_count.png", dpi=150, bbox_inches="tight")
plt.show()


**Takeaway.** This is real, and it's not small. Across all 247 `reliable_n` constituencies (already restricted to >=100 social / >=10 racialised reviews, so this isn't a thin-sample artifact), `racialised_rate` correlates negatively with both **`social_reviews`** (rho=-0.337, p=5.5e-08) and, more strongly, **`pool_count`** (rho=-0.556, p=1.8e-21) -- constituencies with more pools and more reviews systematically show *lower* racialised rates, visible directly in the funnel-shaped scatter above.

This connects to, and is reassured by, the concentration finding in WHERE: that check showed a high rate isn't usually one bad *venue* skewing an otherwise-quiet constituency (rate tracks the *share* of pools affected, not one dominant pool's contribution). Put together: **this is a genuine constituency-scale pattern, not a venue-scale artifact and not sampling noise** -- something about *larger, more pool-dense* constituencies (whether that's genuinely different demographics, more diffuse/anonymous venues, or something else entirely) goes with a lower rate. Whatever the mechanism, `pool_count` (and to a lesser extent `social_reviews`) is too strong a relationship to ignore as a covariate -- see the urban/rural investigation below, which explains why.

## Who is present in the highest- vs. lowest-rate constituencies?

We compare the 15 highest-rate and 15 lowest-rate `reliable_n` constituencies directly: their volume (confirming the pattern above concretely), the `ethnicity_group4` composition of their racialised reviews, and their dimension profile. Then, rather than relying on just these 30 hand-picked constituencies, we check the same relationship properly across all 247 `reliable_n` constituencies: does a constituency's `Arab`/`Türkiye`/`Germany`/`Europe`/`Other` *share* of racialised reviews correlate with its `racialised_rate` directly?


In [ ]:
high15 = rel.sort_values("racialised_rate", ascending=False).head(15)["constituency_code"].tolist()
low15 = rel.sort_values("racialised_rate", ascending=True).head(15)["constituency_code"].tolist()

print("Top 15 (highest rate) -- volume:")
print(rel[rel["constituency_code"].isin(high15)][["social_reviews", "pool_count"]].agg(["mean", "min", "max"]).round(1))
print("\nBottom 15 (lowest rate) -- volume:")
print(rel[rel["constituency_code"].isin(low15)][["social_reviews", "pool_count"]].agg(["mean", "min", "max"]).round(1))

df["tier"] = np.where(df["constituency_code"].isin(high15), "High-rate (top 15)",
              np.where(df["constituency_code"].isin(low15), "Low-rate (bottom 15)", "Other"))
sub = df[df["tier"] != "Other"]
print(f"\nracialised reviews backing each tier: {sub['tier'].value_counts().to_dict()}")

print("\nethnicity_group4 composition by tier (col %):")
print((pd.crosstab(sub["ethnicity_group4"], sub["tier"], normalize="columns") * 100).round(1))

print("\ntheme by tier (col %) -- the dimension with the sharpest tier difference:")
exploded_theme = sub[["theme", "tier"]].explode("theme")
ct_theme = pd.crosstab(exploded_theme["theme"].to_numpy(), exploded_theme["tier"].to_numpy())
print((ct_theme.div(ct_theme.sum(axis=0), axis=1) * 100).round(1))


In [ ]:
# more rigorous version: correlate each ethnicity_group4's *share* of a constituency's racialised
# reviews directly against racialised_rate, across all 247 reliable_n constituencies -- not just
# the hand-picked top/bottom 15
eth_share = pd.crosstab(
    df.loc[df["ethnicity_group4"] != "Unknown", "constituency_code"],
    df.loc[df["ethnicity_group4"] != "Unknown", "ethnicity_group4"],
    normalize="index",
)
eth_share = eth_share.reindex(rel["constituency_code"]).fillna(0)
merged = rel.set_index("constituency_code").join(eth_share)

print("correlation: a constituency's ethnicity_group4 share (of its racialised reviews) vs. its racialised_rate:")
for grp in ["Germany", "Arab", "Türkiye", "Europe", "Other"]:
    rho, p = stats.spearmanr(merged[grp], merged["racialised_rate"])
    rho_pc, p_pc = stats.spearmanr(merged[grp], merged["pool_count"])
    print(f"{grp:10s}: vs. racialised_rate  rho={rho:+.3f} p={p:.3f}   |   vs. pool_count  rho={rho_pc:+.3f} p={p_pc:.3f}")


**Takeaway.** The volume pattern holds concretely for these specific constituencies: the low-rate tier isn't smaller or thinner, **it's bigger** -- mean 3,195 social reviews / 32.4 pools vs. the high-rate tier's 1,427 / 10.9 (roughly 2-3x more of both). This directly answers "is it just volume": partly yes (see above), but not *only* volume -- composition differs too, and does so systematically, not just for these 30 hand-picked constituencies.

The rigorous version (all 247 `reliable_n` constituencies, not just the top/bottom 15) shows a constituency's `racialised_rate` correlates with its **`ethnicity_group4` composition directly**: **`Arab` share (rho=+0.211, p=0.001)** and **`Türkiye` share (rho=+0.180, p=0.004)** both predict a *higher* rate; **`Germany` share (rho=-0.161, p=0.011)** predicts a *lower* one; `Europe` trends the same direction as Arab/Türkiye but doesn't clear significance (rho=+0.105, p=0.100); **`Other` (the long-tail/non-top-ethnicity group) shows no relationship at all (rho=-0.036, p=0.573)**. This specificity matters -- it isn't "any non-German presence," it's specifically Arab and Türkiye presence that tracks a higher rate.

**One genuinely counter-intuitive result, worth flagging rather than smoothing over.** `theme` shows the sharpest tier difference, but in a direction that doesn't simply mirror the individual-level ethnicity/theme finding earlier in this section: the **low-rate tier is far more `Staff`-themed (18.5% vs. 8.6%)**, while the high-rate tier skews toward `Crowd` (31.3% vs. 25.0%) and `Clothing` (11.4% vs. 7.4%) -- even though the high-rate tier also has *more* Arab/Türkiye share, and Arab/Türkiye reviews were individually shown (earlier in WHO) to skew *toward* `Staff`. This is a real example of the ecological-fallacy caution this notebook has flagged before: a constituency-level aggregate pattern doesn't have to match the individual-level pattern that (partly) composes it. It's a genuine, non-obvious lead worth a closer qualitative look, not something this descriptive scan can fully explain.

**On "have some places figured out how to handle it."** This data can't distinguish that hypothesis from a duller one (different venue mix, different demographic composition, or the volume-dilution effect above) -- all three predict the same observable pattern. Flagging it as an open question rather than a finding.

### Following up on the Staff-theme puzzle directly

The tier comparison above flagged something that looked contradictory: the low-rate tier is *more* `Staff`-themed even though it has *less* Arab/Türkiye presence (the two groups that individually skew `Staff`-heavy). Two explanations are possible: either the ethnicity mix shift is doing this indirectly (a compositional/ecological effect), or every ethnicity group independently writes more about `Staff` in low-rate constituencies (a genuine place-level effect that composition can't explain). We test this directly with a standard decomposition: **hold each tier's actual ethnicity mix fixed, and ask what the low tier's `Staff`-share would be if it kept the high tier's within-group `Staff` rates (and vice versa)** -- if the gap mostly disappears under this swap, it's compositional; if it survives, it's a real within-group/place effect.


In [ ]:
exploded_theme = sub[["theme", "tier", "ethnicity_group4"]].explode("theme")

# within-group Staff-mention share, by tier
denom = exploded_theme.groupby(["tier", "ethnicity_group4"]).size()
staff_n = exploded_theme[exploded_theme["theme"] == "Staff"].groupby(["tier", "ethnicity_group4"]).size()
staff_share_by_group = (staff_n / denom).fillna(0)
print("Staff theme-mention share, by ethnicity_group4 x tier:")
print(staff_share_by_group.unstack("tier").round(3))

overall = exploded_theme.groupby("tier")["theme"].apply(lambda s: (s == "Staff").mean())
print(f"\noverall Staff share: high tier={overall['High-rate (top 15)']:.3f}, low tier={overall['Low-rate (bottom 15)']:.3f}")

# decomposition: high tier's ethnicity mix (by theme-mention count) x low tier's within-group rates
mix_high = denom["High-rate (top 15)"] / denom["High-rate (top 15)"].sum()
counterfactual = sum(
    mix_high.get(g, 0) * staff_share_by_group.get(("Low-rate (bottom 15)", g), 0)
    for g in mix_high.index
)
print(f"\ncounterfactual: high tier's ethnicity mix + low tier's within-group Staff rates = {counterfactual:.3f}")
print(f"(compare: actual high tier = {overall['High-rate (top 15)']:.3f}, actual low tier = {overall['Low-rate (bottom 15)']:.3f})")
print("If this counterfactual sits close to the LOW tier's actual share, composition explains "
      "little of the gap -- it's a within-group effect, not an ethnicity-mix effect.")


In [ ]:
from scipy.stats import rankdata
import statsmodels.api as sm

# does this hold as a continuous relationship across all 247 reliable_n constituencies,
# not just the hand-picked top/bottom 15?
exploded_all = df[["constituency_code", "theme"]].explode("theme")
staff_by_const = exploded_all.groupby("constituency_code")["theme"].apply(lambda s: (s == "Staff").mean())
rel = rel.set_index("constituency_code") if rel.index.name != "constituency_code" else rel
rel["staff_share"] = staff_by_const.reindex(rel.index)

rho_rate, p_rate = stats.spearmanr(rel["staff_share"], rel["racialised_rate"])
rho_pool, p_pool = stats.spearmanr(rel["staff_share"], rel["pool_count"])
print(f"staff_share vs. racialised_rate (n=247): rho={rho_rate:.3f}, p={p_rate:.2e}")
print(f"staff_share vs. pool_count (n=247):      rho={rho_pool:.3f}, p={p_pool:.2e}")

rx, ry, rc = rankdata(rel["staff_share"]), rankdata(rel["racialised_rate"]), rankdata(rel["pool_count"])
resid_x = sm.OLS(rx, sm.add_constant(rc)).fit().resid
resid_y = sm.OLS(ry, sm.add_constant(rc)).fit().resid
rho_partial, p_partial = stats.pearsonr(resid_x, resid_y)
print(f"partial (controlling for pool_count):    rho={rho_partial:.3f}, p={p_partial:.3f}")

# is Staff just structurally more "incident"-coded than other themes, regardless of who/where?
exploded_ctx = df[["theme", "context"]].explode("theme").explode("context")
ct_ctx = pd.crosstab(exploded_ctx["theme"].to_numpy(), exploded_ctx["context"].to_numpy())
ctx_pct = (ct_ctx.div(ct_ctx.sum(axis=1), axis=0) * 100).sort_values("incident", ascending=False)
print("\nshare of each theme's mentions framed as 'incident' (all 7,768 racialised reviews):")
print(ctx_pct.round(1))

print(f"\nstate composition -- high-rate 15: {rel.loc[high15, 'state'].value_counts().to_dict()}")
print(f"state composition -- low-rate 15: {rel.loc[low15, 'state'].value_counts().to_dict()}")


**Resolved: it's a within-group effect, not a compositional one -- composition, if anything, works against it.** Every single ethnicity group independently writes about `Staff` roughly 2x more in the low-rate tier than the high-rate tier: `Germany` 5.2%->14.4%, `Arab` 22.0%->44.4%, `Türkiye` 18.5%->25.0%, `Europe` 9.5%->24.0%, `Other` 8.4%->19.9%, `Unknown` 6.8%->12.1%. The decomposition confirms this directly: giving the low tier the *high* tier's ethnicity mix but keeping the low tier's own within-group `Staff` rates still predicts **19.5%** -- almost exactly the actual observed low-tier share (18.5%), and nowhere near the high tier's 8.6%. The ethnicity-mix shift explains essentially none of the gap.

This holds up as a genuine, continuous relationship too, not just a top/bottom-15 artifact: across all 247 `reliable_n` constituencies, a constituency's `Staff`-share of its racialised reviews correlates with a *lower* `racialised_rate` directly (rho=-0.200, p=1.6e-03), and **survives controlling for `pool_count`** (partial rho=-0.155, p=0.015) even though `pool_count` itself has a small positive relationship with `Staff`-share (rho=+0.131, p=0.040) -- so this isn't simply riding on the volume effect either.

**Part of the mechanism is structural, not about who's writing at all**: `Staff` is intrinsically the second-most "incident"-coded theme in the entire corpus (53.1% of `Staff` mentions are framed as a specific `incident`, vs. only 20.9% for `Crowd`) -- a `Staff` complaint is almost definitionally about a specific interaction, while a `Crowd` complaint is almost definitionally ambient/general. That single fact mechanically links the low-rate tier's `Staff`-tilt to its earlier-noted incident-tilt (32.7% vs. 28.3%) -- these aren't two independent findings, they're the same one showing up in two dimensions.

**What this doesn't resolve**: *why* low-rate (and larger, more pool-dense) constituencies skew toward `Staff`-type racialised content regardless of who's writing it. The state breakdown is suggestive context, not an answer -- the high-rate 15 are dominated by Ruhr-area NRW and Berlin (urban), the low-rate 15 by Bayern/Mecklenburg-Vorpommern/Schleswig-Holstein (rural/coastal) -- consistent with a genuine urban/rural or venue-type difference, but this notebook can't distinguish that from the "rare racialised commentary selects for more clearly incident-worthy content" hypothesis floated earlier. Both remain open, qualitative leads for RQ3, not something this descriptive scan settles -- but the one thing now settled is that **it is not an artifact of which ethnicities are present.**

## Where is each ethnicity group most dominant?

The subsections above show *that* a constituency's `ethnicity_group4` composition correlates with its rate -- this asks the more concrete WHO x WHERE question directly: *which specific constituencies* have the highest share of `Arab`, `Türkiye`, `Europe`, or `Germany` in their racialised reviews? We restrict to constituencies with at least 15 known-ethnicity racialised reviews (a floor for a stable share, well below `reliable_n`'s own bar since we're ranking within-constituency composition, not re-testing the rate itself).


In [ ]:
MIN_RACIALISED_FOR_SHARE = 15
known_all = df[df["ethnicity_group4"] != "Unknown"]
eth_share_dom = pd.crosstab(known_all["constituency_code"], known_all["ethnicity_group4"], normalize="index")
n_known_dom = known_all.groupby("constituency_code").size()
eth_share_dom = eth_share_dom[n_known_dom.reindex(eth_share_dom.index) >= MIN_RACIALISED_FOR_SHARE]

meta_cols = ["constituency", "state", "racialised_rate", "racialised_reviews", "reliable_n"]
eth_share_dom = eth_share_dom.join(const_agg.set_index("constituency_code")[meta_cols])
print(f"constituencies with >={MIN_RACIALISED_FOR_SHARE} known-ethnicity racialised reviews: {len(eth_share_dom)} / 299")

for grp in ["Arab", "Türkiye", "Europe", "Germany"]:
    print(f"\ntop 8 constituencies by {grp} share of racialised reviews:")
    top = eth_share_dom.sort_values(grp, ascending=False).head(8)
    print(top[[grp, "constituency", "state", "racialised_reviews", "racialised_rate", "reliable_n"]].round(3).to_string(index=False))

print("\nstate composition of the top-10-by-share constituencies, per group (a rough geographic-clustering check):")
for grp in ["Arab", "Türkiye", "Europe", "Germany"]:
    top10_states = eth_share_dom.sort_values(grp, ascending=False).head(10)["state"].value_counts()
    print(f"{grp:10s}: {top10_states.to_dict()}")


**Takeaway.** Two of the four groups show real geographic clustering; two don't. **`Europe` dominance clusters in Baden-Württemberg** (4 of the top 10 constituencies, including `Lörrach – Müllheim` itself -- the known high-rate hotspot near the France/Switzerland border, at 21.9% Europe share and a large, reliable N=228) -- consistent with the France-border proximity signal already found in the WHO language facet. **`Germany` dominance clusters in eastern states** (`Sachsen`/`Thüringen`/`Sachsen-Anhalt` account for 5 of the top 10, e.g. `Zwickau` at 71.4% Germany share), consistent with the East's lower foreign-population share found throughout this notebook. **`Arab` and `Türkiye` dominance, by contrast, show no comparable single-state cluster** -- `Arab`'s top 10 spans 5 states (NRW appearing most often, 5 of 10) and `Türkiye`'s spans 7 states (NRW again most frequent, 3 of 10) -- concentrated in specific constituencies rather than specific regions.

Read together with the earlier density finding: `Europe`'s clustering in southwestern Baden-Württemberg and `Germany`'s clustering in the (less dense, more rural) East both plausibly ride on the same geography/density story documented earlier in this section. `Arab`/`Türkiye`'s more scattered, constituency-specific pattern (rather than state-level clustering) is consistent with immigration settlement patterns following specific cities/industrial areas (era Ruhr-area industrial recruitment) rather than a broad regional gradient -- a genuine difference in *kind* of geographic pattern, not just degree.

## Does this connect to the border-distance finding from WHERE?

WHERE found no significant relationship between `distance_to_neighbor_country_km` and
`racialised_rate`, because different border segments pull in opposite directions (the
France/Switzerland stretch high, the Denmark stretch average-to-low). Now that we have
`ethnicity_group4` and the land-border-only distance measure, we can check the more specific
question directly: is *who* is racialising different near the border vs. in the interior?

In [ ]:
merged["is_land_border_constituency"] = const_agg.set_index("constituency_code").loc[merged.index, "is_land_border_constituency"]
merged["distance_to_neighbor_country_km"] = const_agg.set_index("constituency_code").loc[merged.index, "distance_to_neighbor_country_km"]

print("ethnicity_group4 share vs. distance_to_neighbor_country_km (n=247 reliable_n):")
for grp in ["Germany", "Arab", "Türkiye", "Europe", "Other"]:
    rho, p = stats.spearmanr(merged[grp], merged["distance_to_neighbor_country_km"])
    print(f"{grp:10s}: rho={rho:+.3f} p={p:.3f}")

print(f"\nmean ethnicity_group4 share: true land-border (n={merged['is_land_border_constituency'].sum()}) "
      f"vs. interior (n={(~merged['is_land_border_constituency']).sum()}):")
print((merged.groupby("is_land_border_constituency")[["Germany", "Arab", "Türkiye", "Europe", "Other"]].mean() * 100).round(1))

print(f"\ntrue land-border constituencies in the top-15/bottom-15 rate tiers (corrected measure): "
      f"top15={merged.loc[high15, 'is_land_border_constituency'].sum()}, "
      f"bottom15={merged.loc[low15, 'is_land_border_constituency'].sum()}")


**Takeaway.** No -- once the coastline-inclusion bug is fixed, **border-proximity essentially doesn't matter for who's racialising either.** Ethnicity share vs. `distance_to_neighbor_country_km` is at best borderline: `Germany` share rises slightly farther from a border (rho=+0.126, p=0.047), `Arab` share falls slightly (rho=-0.115, p=0.072, just short of significance) -- both tiny effects. Comparing true land-border constituencies (n=39) directly against interior ones (n=208), the composition is **nearly identical**: Germany 30.1% vs. 32.0%, Arab 7.6% vs. 6.9%, Türkiye 4.0% vs. 5.4%, Europe 8.8% vs. 8.4%, Other 49.5% vs. 47.3% -- no meaningful gap anywhere. And among the rate tiers, true land-border constituencies are a small minority either way (2/15 high-rate, 4/15 low-rate) -- not the lopsided pattern the *uncorrected* coastline-inclusive measure seemed to suggest.

Combined with the corrected null correlation between distance and rate directly (previous subsection above), the conclusion is consistent throughout: **whatever makes `Offenburg`/`Lörrach – Müllheim`/`Saarbrücken` distinctive is specific to those particular places, not a general "being near a border" effect** -- and it isn't explained by who's racialising there either.

### Draft: paper write-up tables (Bringing WHO+WHERE together)

Two tables: (1) constituency-level correlations with racialised_rate (pool_count, social_reviews, ethnicity_group4 shares, border distance), (2) descriptive summary (volume tiers, theme-mix tier reversal, geographic clustering by group, land-border composition). Source file: `bringing_together_findings_tables.tex` (session scratchpad, not reachable from your side -- this cell is the copy to actually use). Move into the paper repo when ready.

```latex
% Requires: \usepackage{booktabs} \usepackage{array}
% All numbers pulled directly from descriptive_analysis.ipynb's "Bringing WHO
% and WHERE together" section takeaway cells (90d2ee84/07c72082/60eec58c/3dc05a02).

% ------------------------------------------------------------------
% Table 1 -- constituency-level correlations
% ------------------------------------------------------------------
\begin{table}[htbp]
\centering
\footnotesize
\setlength{\tabcolsep}{4pt}
\caption{Constituency-level correlations with \texttt{racialised\_rate} (Spearman $\rho$, 247 \texttt{reliable\_n} constituencies unless noted).}
\label{tab:bringing-together-correlations}
\begin{tabular}{@{} l r r c @{}}
\toprule
Predictor & $\rho$ & $p$ & Sig. \\
\midrule
pool\_count & $-0.556$ & 1.8e-21 & *** \\
social\_reviews & $-0.337$ & 5.5e-08 & *** \\
Arab share (ethnicity\_group4) & \phantom{-}0.211 & 0.001 & ** \\
T\"urkiye share & \phantom{-}0.180 & 0.004 & ** \\
Germany share & $-0.161$ & 0.011 & * \\
Europe share & \phantom{-}0.105 & 0.100 & ns \\
Other share & $-0.036$ & 0.573 & ns \\
\midrule
\multicolumn{4}{@{}l}{\emph{Ethnicity share vs.\ distance to nearest land border (299 constituencies)}} \\
Germany share & \phantom{-}0.126 & 0.047 & * \\
Arab share & $-0.115$ & 0.072 & ns \\
\bottomrule
\end{tabular}

\vspace{2pt}
\raggedright\footnotesize
$^{*}p<0.05$, $^{**}p<0.01$, $^{***}p<0.001$, ns = not significant. \texttt{pool\_count}/\texttt{social\_reviews} correlate negatively with the rate (larger, more pool-dense constituencies show a lower rate); it is specifically Arab/T\"urkiye presence, not non-German presence generally, that predicts a higher rate. Both border-distance correlations are small and one is borderline non-significant --- consistent with WHERE's own null border-distance finding.
\end{table}

% ------------------------------------------------------------------
% Table 2 -- descriptive summary (volume tiers, geography, border composition)
% ------------------------------------------------------------------
\begin{table}[htbp]
\centering
\footnotesize
\setlength{\tabcolsep}{4pt}
\caption{Bringing WHO and WHERE together: descriptive summary.}
\label{tab:bringing-together-summary}
\begin{tabular}{@{} p{0.4\linewidth} p{0.54\linewidth} @{}}
\toprule
Finding & Value \\
\midrule
Volume, low-- vs.\ high-rate tier (15 each) & mean 3{,}195 social reviews / 32.4 pools (low) vs.\ 1{,}427 / 10.9 (high) --- low-rate tier is 2--3$\times$ bigger on both \\
Theme mix, low-- vs.\ high-rate tier & Staff 18.5\% (low) vs.\ 8.6\% (high); Crowd 25.0\% vs.\ 31.3\%; Clothing 7.4\% vs.\ 11.4\% --- reverses the individual-level Arab/T\"urkiye$\to$Staff pattern found in WHO (ecological-fallacy caveat) \\
Europe dominance, geography & clusters in Baden-W\"urttemberg (4/10 top constituencies), incl.\ L\"orrach--M\"ullheim itself (21.9\% Europe share, $n$=228) \\
Germany dominance, geography & clusters in the East (Sachsen/Th\"uringen/Sachsen-Anhalt, 5/10 top constituencies), e.g.\ Zwickau at 71.4\% \\
Arab/T\"urkiye dominance, geography & no single-state cluster --- Arab spans 5 states (NRW most frequent, 5/10), T\"urkiye spans 7 (NRW 3/10); concentrated in specific cities, not a regional gradient \\
Land-border ($n$=39) vs.\ interior ($n$=208) composition & Germany 30.1\% vs.\ 32.0\%; Arab 7.6\% vs.\ 6.9\%; T\"urkiye 4.0\% vs.\ 5.4\%; Europe 8.8\% vs.\ 8.4\%; Other 49.5\% vs.\ 47.3\% --- no meaningful gap anywhere \\
Land-border representation in rate tiers & only 2/15 high-rate and 4/15 low-rate constituencies are true land-border seats --- a small minority either way \\
\bottomrule
\end{tabular}
\end{table}

```


## Investigating what's still open: is this urban vs. rural?

Two candidate explanations were left open above for why low-rate (larger, more pool-dense) constituencies skew `Staff`-themed: a genuine urban/rural or venue-type difference, or a "rare racialised commentary selects for more notable content" selection effect. We can actually test the first one directly -- `political` (loaded in Section 0, already used for the electoral columns) also carries `population_density_per_km2` and `area_km2` from the federal baseline, which haven't been pulled into this notebook before now. This also lets us check something the synthesis above didn't: whether the `ethnicity_group4` composition finding is itself partly an urban/rural proxy, since immigrant populations are well known to concentrate in German cities.


In [ ]:
from scipy.stats import rankdata
import statsmodels.api as sm

def partial_spearman(x, y, control):
    """Rank-residualize x and y against control, then Spearman-correlate the residuals."""
    rx, ry, rc = rankdata(x), rankdata(y), rankdata(control)
    Z = sm.add_constant(rc)
    x_resid = sm.OLS(rx, Z).fit().resid
    y_resid = sm.OLS(ry, Z).fit().resid
    return stats.pearsonr(x_resid, y_resid)

political_idx = political.set_index("constituency_code")
rel["population_density_per_km2"] = rel.index.map(political_idx["population_density_per_km2"])
rel["area_km2"] = rel.index.map(political_idx["area_km2"])
rel["foreign_population"] = rel.index.map(political_idx["foreign_population"])
rel["total_population"] = rel.index.map(political_idx["total_population"])
rel["foreign_population_pct"] = rel["foreign_population"] / rel["total_population"] * 100

rho_dp, p_dp = stats.spearmanr(rel["population_density_per_km2"], rel["pool_count"])
rho_dr, p_dr = stats.spearmanr(rel["population_density_per_km2"], rel["racialised_rate"])
rho_ds, p_ds = stats.spearmanr(rel["population_density_per_km2"], rel["staff_share"])
print(f"population_density_per_km2 vs. pool_count:     rho={rho_dp:.3f}, p={p_dp:.2e}")
print(f"population_density_per_km2 vs. racialised_rate: rho={rho_dr:.3f}, p={p_dr:.2e}")
print(f"population_density_per_km2 vs. staff_share:      rho={rho_ds:.3f}, p={p_ds:.3f}")

print(f"\nhigh-rate 15 -- density: mean={rel.loc[high15, 'population_density_per_km2'].mean():.0f}, "
      f"range {rel.loc[high15, 'population_density_per_km2'].min():.0f}-{rel.loc[high15, 'population_density_per_km2'].max():.0f} per km2; "
      f"area: mean={rel.loc[high15, 'area_km2'].mean():.0f} km2")
print(f"low-rate 15 -- density: mean={rel.loc[low15, 'population_density_per_km2'].mean():.0f}, "
      f"range {rel.loc[low15, 'population_density_per_km2'].min():.0f}-{rel.loc[low15, 'population_density_per_km2'].max():.0f} per km2; "
      f"area: mean={rel.loc[low15, 'area_km2'].mean():.0f} km2")

# is density just standing in for pool_count, or does each carry independent information?
rho_d_partial, p_d_partial = partial_spearman(rel["population_density_per_km2"], rel["racialised_rate"], rel["pool_count"])
rho_pc_partial, p_pc_partial = partial_spearman(rel["pool_count"], rel["racialised_rate"], rel["population_density_per_km2"])
print(f"\ndensity vs. rate, controlling for pool_count:  rho={rho_d_partial:.3f}, p={p_d_partial:.3f}")
print(f"pool_count vs. rate, controlling for density:  rho={rho_pc_partial:.3f}, p={p_pc_partial:.3f}")
print("(both survive controlling for the other -- density is not just relabelling pool_count)")


In [ ]:
# venue-type composition of the pools themselves (not reviews), by tier -- a structural look at
# what kind of facility dominates each tier's swimming-pool stock
places_tier = corpus[["unique_identifier", "constituency_code", "primary_place_type"]].drop_duplicates(subset="unique_identifier")
places_tier["tier"] = np.where(places_tier["constituency_code"].isin(high15), "High-rate (top 15)",
                       np.where(places_tier["constituency_code"].isin(low15), "Low-rate (bottom 15)", "Other"))
venue_ct = pd.crosstab(places_tier.loc[places_tier["tier"] != "Other", "primary_place_type"],
                        places_tier.loc[places_tier["tier"] != "Other", "tier"], normalize="columns") * 100
print("primary_place_type composition of pools, by tier (col %, top 8 by high-tier share):")
print(venue_ct.sort_values("High-rate (top 15)", ascending=False).head(8).round(1))

# reviewer_rating (the 1-5 star rating attached to the review) by theme -- a severity signal
exploded_rating = df[["theme", "reviewer_rating"]].explode("theme")
print("\nmean reviewer_rating by theme, all 7,768 racialised reviews (lower = more negative overall experience):")
print(exploded_rating.groupby("theme")["reviewer_rating"].mean().sort_values().round(2))


In [ ]:
eth_share_all = pd.crosstab(df["constituency_code"], df["ethnicity_group4"], normalize="index")
eth_share_aligned = eth_share_all.reindex(rel.index)
for grp in ["Germany", "Arab", "Türkiye", "Europe", "Other"]:
    rel[grp] = eth_share_aligned[grp].values

rho_fp, p_fp = stats.spearmanr(rel["population_density_per_km2"], rel["foreign_population_pct"])
print(f"population_density_per_km2 vs. foreign_population_pct: rho={rho_fp:.3f}, p={p_fp:.2e}")

print("\ndensity vs. each ethnicity_group4 share of a constituency's racialised reviews:")
for grp in ["Germany", "Arab", "Türkiye", "Europe", "Other"]:
    rho, p = stats.spearmanr(rel["population_density_per_km2"], rel[grp])
    print(f"{grp:10s}: rho={rho:+.3f} p={p:.3f}")

print("\ndoes ethnicity_group4 share still predict racialised_rate once density is controlled?")
for grp in ["Arab", "Türkiye", "Germany"]:
    rho_p, p_p = partial_spearman(rel[grp], rel["racialised_rate"], rel["population_density_per_km2"])
    print(f"{grp:10s} share vs. rate | controlling for density: rho={rho_p:+.3f} p={p_p:.3f}")


**Yes -- this is decisively an urban/rural story, much bigger than the `pool_count` finding alone suggested.** `population_density_per_km2` correlates with `racialised_rate` at **rho=0.597** (p=3.1e-25) -- one of the strongest relationships found anywhere in this notebook -- and with `pool_count` at **rho=-0.794**. The tier means make this concrete: the high-rate 15 average **3,425 people/km2** across a **260 km2** footprint; the low-rate 15 average **186 people/km2** across **2,355 km2** -- a ~20x density gap and ~9x area gap. This also explains *why* `pool_count` behaves the way it does, mechanically: Germany's Bundestag constituencies are drawn to roughly equal population, so a sparse rural constituency must cover far more land (and therefore accumulate more small local pools) to reach the same population quota that a dense city constituency reaches in a fraction of the area with fewer, larger venues. **The `pool_count` finding earlier in this section was, in substantial part, an urban/rural proxy, not an independent "volume" effect** -- though both survive controlling for each other (density partial rho=0.307, p<0.001; pool_count partial rho=-0.169, p=0.008), so neither fully subsumes the other.

**This also reframes the ethnicity-composition finding from earlier.** Density correlates with `foreign_population_pct` at rho=0.773 (the well-documented fact that immigrant populations concentrate in German cities), and with `Arab`/`Türkiye` share of racialised reviews (rho=+0.196/+0.195) and `Germany` share (rho=-0.202) in the expected directions. Once density is controlled: **`Germany` share's relationship with `racialised_rate` disappears completely** (rho=-0.014, p=0.828, vs. the raw -0.161) -- it was entirely a density proxy. **`Türkiye` share's relationship weakens to non-significance** (rho=+0.082, p=0.198, vs. raw +0.180). **Only `Arab` share survives as an independent effect**, weakened but still real (rho=+0.132, p=0.039, vs. raw +0.211). So of the three ethnicity findings reported earlier in this section, one was fully a density artifact, one was mostly one, and one -- `Arab` presence -- adds something beyond urban/rural alone.

**`Staff`-share itself is not simply explained by density** (rho=-0.113, p=0.076, not significant) -- so the specific Staff-vs-Crowd theme shift is a distinct phenomenon that happens to co-occur with the urban/rural split in these tiers, not a direct product of it. Two supporting, more circumstantial observations: the pool stock itself differs by tier (high-rate constituencies' pools skew toward staffed municipal facilities -- `Public swimming pool` 14.0% vs. 4.9%, `Sports complex` 6.1% vs. 2.5% -- while low-rate constituencies have proportionally more `Swimming lake`/natural sites), and `Staff` and `Access` are, by a wide margin, the two most negatively-rated themes in the entire racialised corpus (mean reviewer rating 1.64 and 1.37 respectively, vs. 2.63 for `Crowd` and 2.68 for `Other`) -- consistent with `Staff`-labelled racialised content marking a more severely negative individual experience, though this doesn't yet explain why that concentrates in lower-rate places specifically.

**In short**: `population_density_per_km2` explains much of what `pool_count` was proxying for, and the earlier `ethnicity_group4`-as-independent-predictor story needs a real caveat: only `Arab` share clearly survives density control, `Germany` and `Türkiye` shares mostly or fully don't. The `Staff`-theme mechanism remains the one genuinely open thread -- density doesn't explain it, so it's a real candidate for the qualitative RQ3 follow-up, not a solved case.

# Part 1 findings: prevalence of racialised discourse across WHEN, WHERE, and WHO

**WHEN.** The racialised rate rises significantly from 2016-2025 (p=1.9e-48), and 2.7x faster from 2021-2025 specifically (p=1.9e-71; 0.92%->2.11%, z-test p=1.7e-49) -- a nationwide rise, not a data artifact: 288/299 constituencies (96.3%) have racialised reviews both before and after 2021. Seasonal effects are significant but small (outdoor 1.92% vs. indoor 1.25%; summer 2.03% vs. other 1.33%; both p<1e-54), and not stable over time: absent in 2016-2017, present in 2018-2019, gone again in 2020-2021 (plausibly COVID-related), then widening every year from 2022 onward. `Clothing` is the one theme with both the steepest post-2021 rise and the strongest season effect of any label tested. Full-sample (N=7,768) dimension prevalence and cross-dimension associations (theme x connotation, Cramér's V=0.122) hold at the corpus level, independent of geography.

**WHERE.** Constituency-grain prevalence estimates cover 247/299 constituencies (83%, `>=10` racialised-review reliability bar), state-grain covers all 16. A pool-level concentration check shows high-rate constituencies are ones where *more* of their pools carry a racialised review (rho=0.65 with rate), not ones dominated by a single outlier venue (rho=0.09, ns) -- ruling out a venue-level artifact for the pattern *across* constituencies. Within individual constituencies, though, the dominant pool itself is often a genuine outlier: 99/247 (40%) have a rate significantly elevated above the rest of their own constituency, and 90/247 (36%) of those carry a minority of the local review volume -- a real venue-specific effect in a substantial minority of cases, not just a big-venue volume artifact. Land-neighbour border distance shows no significant relationship with the rate at any grain (all 299: rho=-0.055, p=0.341; reliable_n only: rho=-0.000, p=0.996) -- different border segments (France/Switzerland high, Denmark average-to-low) pull in opposite directions and cancel out in aggregate.

**WHO.** Refined ethnicity categorization (Germany/Arab/Türkiye/Europe/Other/Unknown, consolidated from a 12-category scheme) with a three-way population comparison (entire corpus / social / racialised) for both ethnicity and language -- language shifts noticeably between entire-corpus and social=yes grains, unlike ethnicity, and both shifts are statistically significant (language: chi2=645, p=3.3e-132; ethnicity: chi2=1573, p<<.001), with small effect sizes typical of this large-N corpus. German-authored racialised reviews skew toward `Crowd`/`Etiquette` themes framed `generalised`; non-German-authored ones skew toward `Staff`/`Access` framed as a specific `incident` -- really an Arab + Türkiye vs. Germany polarity (Cramér's V up to 0.180 for theme), not a uniform non-German effect: `Poland`/`Russian Federation` lean mildly toward the Germany pattern, half the other categories show no strong signature either way. Within non-German reviewers, the *language* written in predicts tone (`connotation`) far better than name-origin does -- a clean gradient from German-language (harshest) to Czech (mildest). Geography checked twice and ruled out: East/West/Berlin doesn't modulate any ethnicity group's dimension profile in a way that survives effect-size scrutiny (3/20 tests significant, all Cramér's V<=0.12), and `Arab`/`Türkiye`-dominant constituencies show no state-level clustering (unlike `Europe`, clustered in Baden-Württemberg, and `Germany`, clustered in the East).

**The WHO x WHERE capstone finding.** Constituency-level `racialised_rate` is driven substantially by **population density/urban-rural status** (rho=+0.597 with rate -- the strongest relationship in this notebook), which also mechanically explains the `pool_count` finding (rho=-0.556): Germany's equal-population constituencies force sparse rural areas to be geographically huge, accumulating more small pools. Once density is controlled, the earlier ethnicity-composition story mostly evaporates -- `Germany` share's link to rate disappears completely, `Türkiye` share weakens to non-significance -- and only `Arab` share survives as an independent predictor (rho=+0.132, p=0.039). The `Staff`-theme concentration in low-rate constituencies is real, holds within every ethnicity group (not compositional), and is not explained by density either -- the one genuinely open mechanism, flagged for qualitative RQ3 follow-up.

# Open threads: what needs socioeconomic and political data to resolve

Everything below is a loose end from Part 1 that Part 2 (`02_socioeconomic_political_relations.ipynb`) is needed to tie off -- gathered here in one place rather than scattered as forward-references throughout.

**Ecological-fallacy / venue-level caveats.** Two checks in WHERE narrow this concern without eliminating it. Pool-concentration shows high-rate constituencies are ones where *more* of their pools carry a racialised review, not areas dominated by one outlier venue (rho=0.65 vs. rho=0.09, ns) -- so a socioeconomic/political correlation at constituency level is not, in general, an artifact of one-venue outliers. But within individual constituencies, 90/247 (36%) have a dominant pool that is itself a genuine, small-share, statistically distinct outlier -- a real limitation on reading any single constituency's rate as a pure area-level signal. These 90 are flagged (`dominant_pool_is_small_high_rate` on `const_agg`) to list explicitly wherever constituency-level rate is used, and as a candidate sensitivity check: exclude or down-weight them and see whether the socioeconomic/political correlations hold.

**Covariates for a predictive/regression model of `racialised_rate`.** `population_density_per_km2` is the strongest single relationship found in this notebook (rho=+0.597) and should lead the covariate list, ahead of `pool_count` (rho=-0.556, largely a density proxy given Germany's equal-population constituency boundaries -- both survive controlling for each other, so neither subsumes the other). `Arab` share of racialised reviews is a secondary candidate: the only `ethnicity_group4` share that survives density control (rho=+0.132, p=0.039), while `Germany` share's link disappears entirely once density is controlled and `Türkiye` share weakens to non-significance. East/West/Berlin geography and land-border proximity were checked directly and are safely excludable from a first-pass specification -- both are null.

**Genuinely open, not yet explained.** The `Staff`-theme concentration in low-rate (rural, low-density) constituencies is real, holds within every ethnicity group, and is explained by neither density nor ethnicity composition -- flagged for qualitative RQ3 follow-up, not resolved here.

**Data-quality caveats that stand regardless of Part 2.**
- The 52 constituencies below the reliability bar (`reliable_n`) should be shown, not hidden, but flagged wherever they appear in a ranking or map.
- The temporal trend rests on 99.94% timestamp coverage after the relative-time recovery step (up from 69.5%); recovery isn't perfectly neutral across states, so treat the trend as directionally reliable, not exact to the percentage point.
- The ethnicity/language `"Other"` category (~140 distinct countries) is too heterogeneous to interpret as one group; several small-language/thin-N cells throughout WHO are suggestive, not precise.
- `Arab` share and `pool_count` are themselves correlated (rho=-0.199) -- the partial-correlation checks here show both relationships exist but don't fully disentangle them; needs the regression-based control Part 2 already applies to `second_vote_afd_pct`/`foreign_population_pct`, extended to `ethnicity_group4` shares and density together.

**Deferred to Part 2's state-level cut.** Dimension-level (`theme`/`approach`/`connotation`/`context`) geographic breakdowns -- 7,768 racialised reviews over 299 constituencies is too thin per-cell for most dimension categories; state level has enough N to be worth trying.

**Spatial pattern to test formally.** The choropleth shows a visually convincing cluster (Ruhr NRW + southwestern Baden-Württemberg); Part 2's Moran's I test should confirm or disconfirm this is genuine spatial autocorrelation, not an eyeballed pattern.

**A note on paper structure.** Several results here are fundamentally *ruling-out-a-confound* findings (pool-level concentration, the land-border null, the East/West null) rather than new substantive claims about racialised discourse itself -- candidates for an appendix or robustness-checks section rather than main-text placement; the final call is Ethel's.

This is where Part 1 hands off: `population_density_per_km2`, `pool_count`, and `Arab` share as the covariate starting point, the 90 dominant-pool-outlier constituencies as a sensitivity check, and the `Staff`-theme mechanism as an open qualitative thread -- everything else needed to test these formally requires the socioeconomic and political data Part 2 brings in.

# Saving outputs for Part 2

This notebook is organized into WHEN (temporal trend), WHERE (geographic distribution), and WHO
(reviewer ethnicity/language, including a content baseline bridging toward RQ3), following the
spatial join and coverage setup above. **Part 2**
(`02_socioeconomic_political_relations.ipynb`) covers the relationship to socioeconomic/
political data, and picks up from here by loading the two small outputs saved below rather
than repeating the spatial join or the corpus load.

- `const_agg.parquet` — the 299-row constituency-level aggregate table built in the WHERE section (pool/review counts, racialised rate + CI, reliability flag revised to a `>=10` racialised-review bar, plus the `is_land_border_constituency` / `distance_to_neighbor_country_km` geometric flags, land-neighbour distance only, coastline excluded).
- `racialised_reviews_geo.parquet` — the 7,768 racialised reviews with their constituency/state/region assignment, dimension labels, and `ethnicity_group4` (needed for Part 2's dimension × political-lean cross-tabs, and for testing the Arab-share lead from the WHO section against socioeconomic/political predictors).

Neither file duplicates the canonical corpus — they're small, derived, single-purpose interchange files between the two notebooks, not a new source of truth (that remains `master_corpus_final.parquet`).

In [ ]:
OUTPUTS_DIR = ANALYSIS_DIR / "notebook_outputs"
OUTPUTS_DIR.mkdir(exist_ok=True)

const_agg.to_parquet(OUTPUTS_DIR / "const_agg.parquet", index=False)

racialised_geo_cols = ["review_id", "constituency_code", "constituency", "state", "region",
                       "approach", "theme", "connotation", "context", "ethnicity_group4"]
racialised_df[racialised_geo_cols].to_parquet(OUTPUTS_DIR / "racialised_reviews_geo.parquet", index=False)

print(f"saved const_agg.parquet ({len(const_agg)} rows) and racialised_reviews_geo.parquet "
      f"({len(racialised_df)} rows) to {OUTPUTS_DIR}")